# Correction Experiment Archive

Complete experimental history for the pitch/duration correction project.
This notebook is **read-only** — all code and outputs are preserved for reference.
The active inference interface is in `m2bert_finetune_comparison.ipynb`.

**Archived on:** 2026-11-02
**Source notebooks:**
- `correction_experiments.ipynb` — from-scratch Transformer experiments (v1–v8)
- `m2bert_finetune_comparison.ipynb` (pre-cleanup) — M2BERT fine-tuning, duration, joint model

---
# Part A: From-Scratch Transformer Experiments (v1–v8)

**Source:** `correction_experiments.ipynb` (56 cells)

Covers: vocab building, windowing, CorrectionModel (from-scratch Transformer),
v1–v4 masking experiments, v5 voice conditioning, v6 larger model, v7 dynamic
masking, v8 structural features, error analysis, and dataset preparation.

# OLiMPiC Pitch Correction — Experiment Notebook

Unified entry point for training, evaluation, and baseline comparison.
Run cells in order from a fresh kernel.

**Cell tags:** `⚙️` config · `⚡` run-once setup · `📊` display (no writes) · `⚠️` writes or trains

### Historical runs

| Run | Train | Criterion | Best dev top-1 |
|-----|-------|-----------|----------------|
| correction_v1 | 3K | loss | 30/499 = 6.01% |
| correction_v2 | 3K | loss | 44/499 = 8.82% |
| correction_v3 | 10K | loss | 55/499 = 11.02% |
| correction_v3_top1 | 10K | top-1 | 47/499 = 9.42% |
| correction_v4_mask | 10K | loss | 46/499 = 9.22% |
| correction_v4_mask_top1 | 10K | top-1 | 63/499 = 12.63% |

Baselines: MFC_10k 26/499 = 5.21% · voice_mfc 46/499 = 9.22% · prev_pitch 20/499 · next_pitch 17/499

### v5 / v6 experiments (seed 42, MPS)

| Run | Arch | Params | Best ep | Dev top-1 | Dev top-5 | Wall (s) |
|-----|------|--------|---------|-----------|-----------|----------|
| v5_v4_control | 128d·3L·512ff | 657,972 | 4/9 | 57/499=11.42% | 34.87% | 46 |
| **v5_voice_conditioned** | 128d·3L·512ff + voice | 659,924 | 10/15 | **83/499=16.63%** | **52.30%** | 74 |
| v6_voice_large | 256d·4L·1024ff + voice | 3,287,252 | 14/19 | 56/499=11.22% | 41.08% | 211 |
| v7_dynamic_mask | 128d·3L·512ff + voice | 659,924 | 1/6 | 55/499=11.02% | 39.28% | 38 |

**v5 is the current best.** v7 (dynamic masking) = 11.02% — worse than v5 static; early stop at ep 1. v6 underfits (train top-1 = 8% < dev 11%) — 3.3M params under-served by 10K examples at lr=1e-3.


In [ ]:
# ⚙️  CONFIG — edit this cell to switch experiments
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path
import json

# Auto-detect ml/ directory (works whether Jupyter is launched from project root or ml/)
_cwd = Path.cwd()
for _c in [_cwd / "ml", _cwd, _cwd.parent / "ml"]:
    if (_c / "data" / "processed").exists():
        ML_ROOT = _c.resolve()
        break
else:
    raise RuntimeError(
        f"Cannot locate ml/ directory. Launch Jupyter from the project root.\n"
        f"cwd={_cwd}"
    )
print(f"ML_ROOT: {ML_ROOT}")

DATASET_ROOT = ML_ROOT / "dataset/olimpic-1.0-synthetic"
DATA_DIR     = ML_ROOT / "data"
RUNS_DIR     = DATA_DIR / "runs"

DEV_JSONL  = DATA_DIR / "processed/correction_dev_v1.jsonl"
TRAIN_10K  = DATA_DIR / "processed/correction_train_10k.jsonl"
TRAIN_3K   = DATA_DIR / "processed/correction_train_v1.jsonl"
CONTEXT_BASELINE_JSON = RUNS_DIR / "context_baselines/context_baseline_results.json"

# All existing run directories
RUNS = {
    "v1":              RUNS_DIR / "correction_v1",
    "v2":              RUNS_DIR / "correction_v2",
    "v3":              RUNS_DIR / "correction_v3",
    "v3_top1":         RUNS_DIR / "correction_v3_top1",
    "v4_mask":         RUNS_DIR / "correction_v4_mask",
    "v4_mask_top1":    RUNS_DIR / "correction_v4_mask_top1",
    "v5_v4_control":   RUNS_DIR / "v5_v4_control",
    "v5_voice":        RUNS_DIR / "v5_voice_conditioned",
    "v6_voice_large":  RUNS_DIR / "v6_voice_large",
    "v7_dynamic_mask": RUNS_DIR / "v7_dynamic_mask",
}

# ── New experiment config ─────────────────────────────────────────────────────
# Edit this dict, then run the training cell (Section 3).
# To switch between sizes:
#   small (v5):  d_model=128, num_layers=3, dim_ff=512
#   large (v6):  d_model=256, num_layers=4, dim_ff=1024
EXP = {
    "name":               "correction_v7",  # → RUNS_DIR / name  (must not already exist)
    "train_jsonl":        TRAIN_10K,
    "use_mask":           True,
    "voice_conditioning": True,
    "voice_dim":          32,
    "d_model":            256,    # 128 = small  |  256 = large
    "num_layers":         4,      # 3   = small  |  4   = large
    "dim_ff":             1024,   # 512 = small  |  1024 = large
    "checkpoint_by":      "top1",
    "seed":               42,
    "batch_size":         32,
    "lr":                 1e-3,
    "weight_decay":       0.01,
    "max_epochs":         30,
    "patience":           5,
    "time_limit":         1800,
    "device":             None,   # None = auto  |  "cpu" = deterministic
}
EXP["output_dir"] = RUNS_DIR / EXP["name"]
print(f"New experiment → {EXP['output_dir']}")


In [ ]:
import re, csv, random, sys, time
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

MAX_LEN            = 257
PAD_TOKEN          = "<PAD>"
UNK_TOKEN          = "<UNK>"
MASK_TOKEN         = "[MASK]"
PITCH_RE           = re.compile(r'^[A-G][0-9]$')
N_HEADS            = 4          # fixed across all experiments
DROPOUT_EXPERIMENT = 0.1


## Core Library

Define all functions once; reused by setup, results, and training cells.

In [ ]:
# Vocabulary builders

def build_vocab_from_train(dataset_root, train_ids):
    """Scan train LMX files; returns {token: id} with PAD=0, UNK=1."""
    raw = set()
    for sid in train_ids:
        p = dataset_root / f"{sid}.lmx"
        if p.exists():
            raw.update(p.read_text().split())
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for t in sorted(raw):
        vocab[t] = len(vocab)
    return vocab


def build_pitch_classes(train_ids, dataset_root):
    """Return {pitch_token: class_id} from train split (sorted for determinism)."""
    pitches = set()
    for sid in train_ids:
        p = dataset_root / f"{sid}.lmx"
        if p.exists():
            pitches.update(t for t in p.read_text().split() if PITCH_RE.match(t))
    return {p: i for i, p in enumerate(sorted(pitches))}


def add_mask_to_vocab(vocab):
    """Append [MASK] as the last vocab entry (ID = len(vocab))."""
    assert MASK_TOKEN not in vocab
    v = dict(vocab)
    v[MASK_TOKEN] = len(vocab)
    return v


In [ ]:
# Windowing and LMX voice-map extraction

def window_tokens(tokens, error_pos, replacement_token, max_len=MAX_LEN):
    """
    Crop to at most max_len tokens centred on error_pos.
    Returns (window, window_error_pos, window_lo).
    Invariant: window[window_error_pos] == replacement_token.
    """
    n = len(tokens)
    if n <= max_len:
        lo, window = 0, tokens[:]
    else:
        lo     = max(0, min(error_pos - max_len // 2, n - max_len))
        window = tokens[lo : lo + max_len]
    w_pos = error_pos - lo
    if window[w_pos] != replacement_token:
        raise ValueError(
            f"Window invariant broken: error_pos={error_pos}, lo={lo}, "
            f"got={window[w_pos]!r}, expected={replacement_token!r}"
        )
    return window, w_pos, lo


def build_voice_map(tokens):
    """
    For each pitch position i in tokens, return its LMX voice label (str or None).

    LMX convention: standalone notes have voice:N immediately after the pitch token.
    Chord notes (preceded by 'chord') inherit the current running voice.
    Orphaned voice:N tokens at the window start update current_voice.
    """
    voice_map, current_voice = {}, None
    for i, tok in enumerate(tokens):
        if PITCH_RE.match(tok):
            nxt = tokens[i + 1] if i + 1 < len(tokens) else ""
            if nxt.startswith("voice:"):
                current_voice = nxt
            voice_map[i] = current_voice
        elif tok.startswith("voice:") and (i == 0 or not PITCH_RE.match(tokens[i - 1])):
            current_voice = tok
    return voice_map


In [ ]:
# CorrectionExample and JSONL loading

class CorrectionExample:
    """One processed example, ready for batching."""
    __slots__ = ("sample_id", "token_ids", "padding_mask", "window_error_pos",
                 "target_class", "window_lo", "target_token", "replacement_token",
                 "voice_label", "voice_id")

    def __init__(self, sample_id, token_ids, padding_mask, window_error_pos,
                 target_class, window_lo, target_token, replacement_token,
                 voice_label=None, voice_id=0):
        self.sample_id, self.token_ids, self.padding_mask = sample_id, token_ids, padding_mask
        self.window_error_pos, self.target_class, self.window_lo = window_error_pos, target_class, window_lo
        self.target_token, self.replacement_token = target_token, replacement_token
        self.voice_label, self.voice_id = voice_label, voice_id


def build_voice_vocab(examples):
    """
    Build {voice_label_str: id} from train examples.
    ID 0 is reserved for UNKNOWN/None; labels start at 1.
    Pass only train examples; dev labels absent from vocab map to 0.
    """
    labels = sorted(set(e.voice_label for e in examples if e.voice_label is not None))
    return {label: i + 1 for i, label in enumerate(labels)}


def load_all_examples(jsonl_path, vocab, pitch_classes, expected_id_set,
                      split_label, mask=False, voice_vocab=None):
    """
    Load every line of a JSONL as CorrectionExample objects.

    mask=True     : replace replacement_token with MASK_TOKEN before tokenisation.
    voice_vocab   : if provided, map voice_label → voice_id (0 = UNKNOWN/None).
                    Voice is parsed BEFORE masking so the pitch token is present.
    Returns (examples, unk_count, total_window_token_count).
    """
    unk_id, pad_id = vocab[UNK_TOKEN], vocab[PAD_TOKEN]
    examples, unk_count, total_count = [], 0, 0
    with open(jsonl_path) as f:
        for lineno, line in enumerate(f, 1):
            ex  = json.loads(line)
            sid = ex["sample_id"]
            if sid not in expected_id_set:
                raise ValueError(f"line {lineno}: {sid} not in {split_label} split")
            tgt = ex["target_token"]
            if tgt not in pitch_classes:
                raise ValueError(f"line {lineno}: target {tgt!r} not in pitch_classes")
            rep    = ex["replacement_token"]
            window, w_pos, w_lo = window_tokens(ex["input_tokens"], ex["error_position"], rep)
            # Parse voice BEFORE masking (replacement token is still a pitch token here)
            voice_label = build_voice_map(window).get(w_pos)
            voice_id    = 0 if voice_vocab is None else voice_vocab.get(voice_label, 0)
            if mask:
                window = list(window)
                assert window[w_pos] == rep
                window[w_pos] = MASK_TOKEN
            ids     = [vocab.get(t, unk_id) for t in window]
            pad_len = MAX_LEN - len(ids)
            unk_count  += ids.count(unk_id)
            total_count += len(ids)
            examples.append(CorrectionExample(
                sample_id         = sid,
                token_ids         = ids + [pad_id] * pad_len,
                padding_mask      = [False] * len(ids) + [True] * pad_len,
                window_error_pos  = w_pos,
                target_class      = pitch_classes[tgt],
                window_lo         = w_lo,
                target_token      = tgt,
                replacement_token = rep,
                voice_label       = voice_label,
                voice_id          = voice_id,
            ))
    return examples, unk_count, total_count


def print_dataset_stats(label, examples, unk_tokens, total_tokens, pitch_classes):
    score_ids     = {e.sample_id.split("/")[1] for e in examples}
    target_counts = Counter(e.target_token for e in examples)
    wls           = [sum(1 for m in e.padding_mask if not m) for e in examples]
    unk_rate      = unk_tokens / total_tokens if total_tokens else 0.0
    print(f"\n{'─'*60}\n {label}\n{'─'*60}")
    print(f"  examples : {len(examples)}")
    print(f"  scores   : {len(score_ids)}")
    print(f"  UNK rate : {unk_rate:.4%}  ({unk_tokens}/{total_tokens})")
    print(f"  window   : min={min(wls)}  max={max(wls)}  mean={sum(wls)/len(wls):.1f}")
    print(f"  target top-5: " + "  ".join(f"{p}={n}" for p, n in target_counts.most_common(5)))
    print(f"  classes covered: {len(target_counts)} / {len(pitch_classes)}")


In [ ]:
# CorrectionModel — bidirectional Transformer encoder for pitch correction

class CorrectionModel(nn.Module):
    """
    Minimal bidirectional Transformer encoder.
    Input : corrupted token sequence + known error position.
    Output: logits over pitch classes (extraction at error_pos hidden state).

    voice_conditioning=True: a learned voice embedding is concatenated with
    the error-position hidden state before the classification head.
    """
    def __init__(self, vocab_size, num_pitch_classes, d_model=128, nhead=4,
                 num_layers=3, dim_feedforward=512, dropout=0.1,
                 max_len=MAX_LEN, pad_id=0,
                 voice_conditioning=False, n_voices=0, voice_dim=32):
        super().__init__()
        self.voice_conditioning = voice_conditioning
        self.token_emb  = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb    = nn.Embedding(max_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=False)
        # enable_nested_tensor=False: avoids MPS-unsupported op (no correctness impact)
        self.encoder    = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                                enable_nested_tensor=False)
        if voice_conditioning:
            self.voice_emb  = nn.Embedding(n_voices + 1, voice_dim)  # 0 = UNKNOWN
            classifier_in   = d_model + voice_dim
        else:
            classifier_in   = d_model
        self.classifier = nn.Linear(classifier_in, num_pitch_classes)

    def forward(self, token_ids, error_pos, padding_mask, voice_ids=None):
        """token_ids: (B,L)  error_pos: (B,)  padding_mask: (B,L) True=pad"""
        B, L   = token_ids.shape
        pos    = torch.arange(L, device=token_ids.device).unsqueeze(0).expand(B, -1)
        x      = self.token_emb(token_ids) + self.pos_emb(pos)
        x      = self.encoder(x, src_key_padding_mask=padding_mask)
        hidden = x[torch.arange(B, device=x.device), error_pos]
        if self.voice_conditioning:
            v_emb  = self.voice_emb(voice_ids)        # (B, voice_dim)
            hidden = torch.cat([hidden, v_emb], dim=-1)
        return self.classifier(hidden)


In [ ]:
# Training and evaluation helpers

def collate(examples, device):
    """Collate a list of CorrectionExample into a batch dict of tensors."""
    return {
        "token_ids":      torch.tensor([e.token_ids        for e in examples], dtype=torch.long, device=device),
        "padding_mask":   torch.tensor([e.padding_mask     for e in examples], dtype=torch.bool, device=device),
        "error_pos":      torch.tensor([e.window_error_pos for e in examples], dtype=torch.long, device=device),
        "target_classes": torch.tensor([e.target_class     for e in examples], dtype=torch.long, device=device),
        "voice_ids":      torch.tensor([e.voice_id         for e in examples], dtype=torch.long, device=device),
    }


def run_train_epoch(model, examples, batch_size, optimizer, loss_fn, device, rng):
    """One full training pass. Returns (avg_loss, top1, top5) — example-level aggregation."""
    model.train()
    shuffled = list(examples); rng.shuffle(shuffled)
    loss_sum = top1 = top5 = n = 0
    for i in range(0, len(shuffled), batch_size):
        b    = collate(shuffled[i : i + batch_size], device)
        bs   = b["token_ids"].size(0)
        optimizer.zero_grad()
        lg   = model(b["token_ids"], b["error_pos"], b["padding_mask"], b["voice_ids"])
        loss = loss_fn(lg, b["target_classes"])
        loss.backward(); optimizer.step()
        tgt      = b["target_classes"]
        loss_sum += loss.item() * bs
        top1     += lg.topk(1, dim=-1)[1].squeeze(1).eq(tgt).sum().item()
        top5     += lg.topk(5, dim=-1)[1].eq(tgt.unsqueeze(1)).any(1).sum().item()
        n        += bs
    return loss_sum / n, top1 / n, top5 / n


@torch.no_grad()
def evaluate_dataset(model, examples, batch_size, loss_fn, device):
    """Eval loop. Returns (avg_loss, top1, top5, pred_class_list) — example-level."""
    model.eval()
    loss_sum = top1 = top5 = n = 0; all_preds = []
    for i in range(0, len(examples), batch_size):
        b    = collate(examples[i : i + batch_size], device)
        bs   = b["token_ids"].size(0)
        lg   = model(b["token_ids"], b["error_pos"], b["padding_mask"], b["voice_ids"])
        loss = loss_fn(lg, b["target_classes"])
        tgt  = b["target_classes"]
        loss_sum += loss.item() * bs
        top1     += lg.topk(1, dim=-1)[1].squeeze(1).eq(tgt).sum().item()
        top5     += lg.topk(5, dim=-1)[1].eq(tgt.unsqueeze(1)).any(1).sum().item()
        n        += bs
        all_preds.extend(lg.argmax(dim=-1).tolist())
    return loss_sum / n, top1 / n, top5 / n, all_preds


@torch.no_grad()
def get_logits_for_dataset(model, examples, batch_size, device):
    """Return full logit tensor (N, num_pitch_classes) on CPU."""
    model.eval()
    parts = []
    for i in range(0, len(examples), batch_size):
        b = collate(examples[i : i + batch_size], device)
        parts.append(model(b["token_ids"], b["error_pos"], b["padding_mask"], b["voice_ids"]).cpu())
    return torch.cat(parts, dim=0)


def build_predictions(examples, pred_classes, class_to_pitch, logits_all):
    """Build list of per-example prediction dicts for saving to JSON."""
    return [{
        "sample_id":         e.sample_id,
        "error_position":    e.window_error_pos + e.window_lo,
        "window_error_pos":  e.window_error_pos,
        "window_lo":         e.window_lo,
        "replacement_token": e.replacement_token,
        "target_token":      e.target_token,
        "predicted_token":   class_to_pitch[pc],
        "correct":           pc == e.target_class,
        "top5_candidates":   [class_to_pitch[i] for i in lg.topk(5).indices.tolist()],
        "voice_label":       e.voice_label,
    } for e, pc, lg in zip(examples, pred_classes, logits_all)]


In [ ]:
# Baseline functions (MFC, context rules, per-voice MFC)

def compute_mfc_baseline(train_examples, dev_examples):
    """Most-frequent-class from train frequency, evaluated on dev."""
    counts = Counter(e.target_token for e in train_examples)
    top1   = counts.most_common(1)[0][0]
    top5   = {p for p, _ in counts.most_common(5)}
    h1 = sum(1 for e in dev_examples if e.target_token == top1)
    h5 = sum(1 for e in dev_examples if e.target_token in top5)
    n  = len(dev_examples)
    print(f"  MFC top-1: {h1}/{n} = {h1/n:.2%}  ({top1!r})")
    print(f"  MFC top-5: {h5}/{n} = {h5/n:.2%}  {sorted(top5)}")
    return h1 / n, h5 / n, top1


def context_baseline_eval(dev_jsonl, mfc_token):
    """
    prev_pitch and next_pitch context rules directly from raw JSONL.
    Falls back to mfc_token when no same-voice neighbour found within the window.
    Returns list of per-example result dicts.
    """
    results = []
    with open(dev_jsonl) as f:
        for line in f:
            ex   = json.loads(line)
            inp, pos = ex["input_tokens"], ex["error_position"]
            rep, tgt = ex["replacement_token"], ex["target_token"]
            window, w_pos, _ = window_tokens(inp, pos, rep)
            vmap = build_voice_map(window)
            v    = vmap.get(w_pos)
            prev = next_tok = None
            for i in range(w_pos - 1, -1, -1):
                if i in vmap and vmap[i] == v: prev = window[i]; break
            for i in range(w_pos + 1, len(window)):
                if i in vmap and vmap[i] == v: next_tok = window[i]; break
            results.append({
                "sample_id":  ex["sample_id"], "target": tgt,
                "prev":       prev or mfc_token,  "next": next_tok or mfc_token,
                "prev_found": prev is not None,   "next_found": next_tok is not None,
            })
    return results


def voice_mfc_eval(train_jsonl, dev_jsonl):
    """
    Per-voice most-frequent-class from train, evaluated on dev.
    Falls back to global MFC for unseen voices.
    Returns (results_list, voice_mfc_map, global_mfc_token).
    """
    global_counts = Counter()
    voice_counts  = defaultdict(Counter)
    with open(train_jsonl) as f:
        for line in f:
            ex = json.loads(line)
            w, wp, _ = window_tokens(ex["input_tokens"], ex["error_position"],
                                     ex["replacement_token"])
            v = build_voice_map(w).get(wp)
            global_counts[ex["target_token"]] += 1
            voice_counts[v][ex["target_token"]] += 1
    global_mfc    = global_counts.most_common(1)[0][0]
    voice_mfc_map = {v: c.most_common(1)[0][0] for v, c in voice_counts.items()}
    hits = fallbacks = 0
    results = []
    with open(dev_jsonl) as f:
        for line in f:
            ex = json.loads(line)
            w, wp, _ = window_tokens(ex["input_tokens"], ex["error_position"],
                                     ex["replacement_token"])
            v    = build_voice_map(w).get(wp)
            pred = voice_mfc_map.get(v, global_mfc)
            if v not in voice_mfc_map: fallbacks += 1
            correct = pred == ex["target_token"]
            if correct: hits += 1
            results.append({"sample_id": ex["sample_id"], "correct": correct, "voice": v})
    n = len(results)
    print(f"  voice_mfc: {hits}/{n} = {hits/n:.2%}  (fallback on {fallbacks})")
    return results, voice_mfc_map, global_mfc


## 1. Setup

Run this section first (fresh kernel).
Vocab building scans all train-split LMX files (~10–20 s on first run).


In [ ]:
# ⚡ Load official split IDs and build vocabulary
def _read_split(name):
    return [l.strip() for l in
            (DATASET_ROOT / f"samples.{name}.txt").read_text().splitlines()
            if l.strip()]

train_ids    = _read_split("train")
dev_ids      = _read_split("dev")
train_id_set = set(train_ids)
dev_id_set   = set(dev_ids)
print(f"Split IDs: train={len(train_ids)}  dev={len(dev_ids)}")

print("Building vocab from train split LMX files...")
vocab          = build_vocab_from_train(DATASET_ROOT, train_ids)
pitch_classes  = build_pitch_classes(train_ids, DATASET_ROOT)
class_to_pitch = {v: k for k, v in pitch_classes.items()}
print(f"Vocab: {len(vocab)}  Pitch classes: {len(pitch_classes)}")

DEVICE = (torch.device(EXP["device"]) if EXP["device"] else
          torch.device("cuda" if torch.cuda.is_available() else
                       "mps"  if torch.backends.mps.is_available() else "cpu"))
print(f"Device: {DEVICE}")


In [ ]:
# ⚡ Load train and dev examples (unmasked; training cells handle masking internally)
print("Loading 10K train examples...")
train_examples, train_unk, train_total = load_all_examples(
    TRAIN_10K, vocab, pitch_classes, train_id_set, "train", mask=False)

print("Loading dev examples...")
dev_examples, dev_unk, dev_total = load_all_examples(
    DEV_JSONL, vocab, pitch_classes, dev_id_set, "dev", mask=False)

# Score-level cross-check (no score should appear in both splits)
ts = {e.sample_id.split("/")[1] for e in train_examples}
ds = {e.sample_id.split("/")[1] for e in dev_examples}
assert not (ts & ds), f"Score overlap found: {ts & ds}"
print(f"Score overlap: CLEAN  (train={len(ts)} scores, dev={len(ds)} scores)")


In [ ]:
# 📊 Dataset statistics and MFC baseline
print_dataset_stats("TRAIN (10K)", train_examples, train_unk, train_total, pitch_classes)
print_dataset_stats("DEV   (499)", dev_examples,   dev_unk,   dev_total,   pitch_classes)

print("\nMost-frequent-class baseline (train → dev):")
mfc_top1_rate, mfc_top5_rate, mfc_token = compute_mfc_baseline(train_examples, dev_examples)


## 2. Existing Run Results 📊

Load and compare all saved run artifacts. No model inference — reads JSON files only.
Requires Section 1 (setup) to have run.


In [ ]:
# 📊 Comparison table: actual checkpoint performance (from dev_predictions.json)
# Checkpoint epoch is read directly from best_checkpoint.pt (ground truth).
# Dev top-1 is counted from saved dev_predictions.json (not metrics peak).
all_metrics = {}
for run_name, run_dir in RUNS.items():
    mf = run_dir / "metrics.json"
    if mf.exists():
        all_metrics[run_name] = json.loads(mf.read_text())

import torch as _torch

header = f"{'Run':<22} {'Eps':>4} {'Ckpt':>5} {'Dev top-1':>17} {'Dev top-5':>10}"
sep    = "─" * len(header)
print(header); print(sep)
for rn, metrics in all_metrics.items():
    run_dir = RUNS[rn]
    n_ep    = max(m["epoch"] for m in metrics)

    # Actual checkpoint epoch from the saved checkpoint file
    ckpt_f  = run_dir / "best_checkpoint.pt"
    if ckpt_f.exists():
        ck      = _torch.load(ckpt_f, map_location="cpu", weights_only=False)
        ckpt_ep = ck["epoch"]
    else:
        ckpt_ep = "?"

    # Actual top-1: count from saved predictions (reflects checkpoint, not peak epoch)
    pred_f = run_dir / "dev_predictions.json"
    if pred_f.exists():
        preds   = json.loads(pred_f.read_text())
        n_corr  = sum(1 for p in preds if p["correct"])
        n_tot   = len(preds)
        t1_disp = f"{n_corr}/{n_tot} = {n_corr/n_tot:.2%}"
    else:
        best_row = max(metrics, key=lambda m: m["dev_top1"])
        t1_disp  = f"{best_row['dev_top1']:.2%}  (metrics peak)"

    # Top-5 from the checkpoint epoch row in metrics
    ep_row = next((m for m in metrics if m["epoch"] == ckpt_ep), metrics[-1])
    print(f"{rn:<22} {n_ep:>4} {str(ckpt_ep):>5} {t1_disp:>17} {ep_row['dev_top5']:>10.2%}")

# Baselines from saved context_baseline_results.json
if CONTEXT_BASELINE_JSON.exists():
    cb    = json.loads(CONTEXT_BASELINE_JSON.read_text())
    N     = len(cb)
    mfc_h  = sum(1 for r in cb if r["mfc_correct"])
    prev_h = sum(1 for r in cb if r["prev_correct"])
    next_h = sum(1 for r in cb if r["next_correct"])
    vm_h   = 46   # from voice_mfc_baseline run
    print(sep)
    print(f"{'MFC_10k':<22} {'—':>4} {'—':>5} {mfc_h}/{N} = {mfc_h/N:.2%}")
    print(f"{'voice_mfc':<22} {'—':>4} {'—':>5} {vm_h}/{N} = {vm_h/N:.2%}")
    print(f"{'prev_pitch':<22} {'—':>4} {'—':>5} {prev_h}/{N} = {prev_h/N:.2%}")
    print(f"{'next_pitch':<22} {'—':>4} {'—':>5} {next_h}/{N} = {next_h/N:.2%}")


In [ ]:
# 📊 Training curves for all saved runs (solid=dev, dashed=train)
if not all_metrics:
    print("No metrics loaded. Run Cell 14 first.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.tab10.colors

    for (rn, metrics), color in zip(all_metrics.items(), colors):
        epochs = [m["epoch"] for m in metrics]
        ax1.plot(epochs, [m["dev_top1"]   for m in metrics], color=color, lw=2,   label=rn)
        ax1.plot(epochs, [m["train_top1"] for m in metrics], color=color, lw=1, ls="--", alpha=0.5)
        ax2.plot(epochs, [m["dev_loss"]   for m in metrics], color=color, lw=2,   label=rn)
        ax2.plot(epochs, [m["train_loss"] for m in metrics], color=color, lw=1, ls="--", alpha=0.5)

    ax1.set(title="Dev top-1 accuracy (solid=dev  dashed=train)",
            xlabel="Epoch", ylabel="Top-1 accuracy")
    ax2.set(title="Loss (solid=dev  dashed=train)",
            xlabel="Epoch", ylabel="Cross-entropy")
    for ax in (ax1, ax2):
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


In [ ]:
# 📊 Prediction examples from a saved run
# Change PREVIEW_RUN to any key in RUNS
PREVIEW_RUN = "v4_mask_top1"

preds_file = RUNS[PREVIEW_RUN] / "dev_predictions.json"
if not preds_file.exists():
    print(f"No predictions file for {PREVIEW_RUN}")
else:
    preds     = json.loads(preds_file.read_text())
    n_correct = sum(1 for p in preds if p["correct"])
    n_total   = len(preds)
    print(f"{PREVIEW_RUN}  →  dev top-1: {n_correct}/{n_total} = {n_correct/n_total:.2%}\n")

    print("── Correct predictions (first 5) ──")
    for p in [x for x in preds if x["correct"]][:5]:
        print(f"  ✓ {p['sample_id']}  pos={p['error_position']}")
        print(f"    rep={p['replacement_token']!r}  "
              f"target={p['target_token']!r}  predicted={p['predicted_token']!r}")
        if "top5_candidates" in p:
            print(f"    top-5: {p['top5_candidates']}")

    print("\n── Wrong predictions (first 5) ──")
    for p in [x for x in preds if not x["correct"]][:5]:
        print(f"  ✗ {p['sample_id']}  pos={p['error_position']}")
        print(f"    rep={p['replacement_token']!r}  "
              f"target={p['target_token']!r}  predicted={p['predicted_token']!r}")
        if "top5_candidates" in p:
            print(f"    top-5: {p['top5_candidates']}")


In [ ]:
# 📊 Context baselines (from saved JSON) and voice-MFC (recomputed)

print("── Context baseline results (from saved JSON) ──")
if CONTEXT_BASELINE_JSON.exists():
    cb  = json.loads(CONTEXT_BASELINE_JSON.read_text())
    N   = len(cb)
    def _h(field): return sum(1 for r in cb if r[field])
    fb_p = sum(1 for r in cb if r["prev_used_fallback"])
    fb_n = sum(1 for r in cb if r["next_used_fallback"])
    print(f"  MFC_10k    : {_h('mfc_correct'):3d}/{N} = {_h('mfc_correct')/N:.2%}")
    print(f"  prev_pitch : {_h('prev_correct'):3d}/{N} = {_h('prev_correct')/N:.2%}  (fallback {fb_p})")
    print(f"  next_pitch : {_h('next_correct'):3d}/{N} = {_h('next_correct')/N:.2%}  (fallback {fb_n})")
    print(f"  v3 (ref)   : {_h('v3_correct'):3d}/{N} = {_h('v3_correct')/N:.2%}")
    print(f"  Coverage: prev {N-fb_p}/{N}  next {N-fb_n}/{N}")
else:
    print("  context_baseline_results.json not found.")

print("\n── Voice-MFC baseline (recomputed from JSONL) ──")
vm_results, voice_mfc_map, vm_global_mfc = voice_mfc_eval(TRAIN_10K, DEV_JSONL)
vm_hits = sum(1 for r in vm_results if r["correct"])
N       = len(vm_results)
print(f"  voice_mfc  : {vm_hits:3d}/{N} = {vm_hits/N:.2%}")
print(f"  global MFC : {vm_global_mfc!r}")
print(f"\n  Per-voice MFC:")
for voice in sorted(voice_mfc_map, key=lambda x: (x is None, x or "")):
    print(f"    {voice or '(None)':10s} → {voice_mfc_map[voice]}")


In [ ]:
# 📊 Cross-run Venn: which dev examples each run gets correct
print("Cross-run correct-set overlap\n")
run_correct = {}
all_sample_ids = set()
for rn, rd in RUNS.items():
    pf = rd / "dev_predictions.json"
    if pf.exists():
        preds = json.loads(pf.read_text())
        run_correct[rn] = frozenset(p["sample_id"] for p in preds if p["correct"])
        all_sample_ids.update(p["sample_id"] for p in preds)

rns = list(run_correct.keys())
# Pairwise overlap table (diagonal = total correct for that run)
print(f"{'':>22}" + "".join(f"{r:>9}" for r in rns))
for rn_a in rns:
    row = f"{rn_a:<22}"
    for rn_b in rns:
        val = (len(run_correct[rn_a]) if rn_a == rn_b
               else len(run_correct[rn_a] & run_correct[rn_b]))
        row += f"{val:>9}"
    print(row)

N = len(all_sample_ids)
print(f"\nTotal dev: {N}")
if run_correct:
    all_correct  = frozenset.intersection(*run_correct.values())
    none_correct = all_sample_ids - frozenset.union(*run_correct.values())
    print(f"Correct in ALL runs:  {len(all_correct)}/{N}")
    print(f"Wrong  in ALL runs:   {len(none_correct)}/{N} = {len(none_correct)/N:.1%}")


## 3. Training ⚠️ TRAINS

These cells run a full training experiment. Before running:

1. Edit `EXP` in Cell 1 (name, hyperparameters, use_mask, checkpoint_by, device).
2. Ensure `EXP["output_dir"]` does not already exist.
3. Run Section 1 (setup cells) if not already done.

**MPS non-determinism:** Apple Silicon MPS produces slightly different results across
runs even with `seed=42`. Set `"device": "cpu"` in `EXP` for fully deterministic training.


In [ ]:
def run_experiment(cfg, train_ids, dev_ids, device):
    """
    ⚠️ TRAINS — Full train/eval loop.

    cfg keys:
      name, train_jsonl, use_mask, voice_conditioning, voice_dim,
      d_model, num_layers, dim_ff, checkpoint_by, seed,
      batch_size, lr, weight_decay, max_epochs, patience, time_limit, output_dir

    checkpoint_by="top1": saves on strict top-1 improvement; ties broken by lower loss.
    checkpoint_by="loss": saves on strict loss improvement.
    Patience resets only on strict improvement of the monitored metric.

    Writes to cfg['output_dir']; aborts if directory already exists.
    Returns dict with metrics, n_correct, best_epoch.
    """
    output_dir = cfg["output_dir"]
    if output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(
            f"Output dir not empty: {output_dir}\n"
            f"Change name or delete the directory first."
        )
    output_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"])
    rng        = random.Random(cfg["seed"])
    start_wall = time.time()
    train_id_set = set(train_ids)
    dev_id_set   = set(dev_ids)

    # Token vocab
    use_mask  = cfg["use_mask"]
    v_base    = build_vocab_from_train(DATASET_ROOT, train_ids)
    vocab_exp = add_mask_to_vocab(v_base) if use_mask else v_base
    pc_exp    = build_pitch_classes(train_ids, DATASET_ROOT)
    c2p       = {v: k for k, v in pc_exp.items()}
    (output_dir / "vocab.json").write_text(json.dumps(vocab_exp, indent=2))
    (output_dir / "pitch_classes.json").write_text(json.dumps(pc_exp, indent=2))

    # Voice conditioning
    use_voice = cfg.get("voice_conditioning", False)
    voice_dim = cfg.get("voice_dim", 32)

    # Load train first (voice_vocab=None → voice_id=0 until vocab built)
    tr, _, _ = load_all_examples(cfg["train_jsonl"], vocab_exp, pc_exp,
                                  train_id_set, "train", mask=use_mask, voice_vocab=None)
    if use_voice:
        voice_vocab = build_voice_vocab(tr)
        n_voices    = len(voice_vocab)
        for e in tr:
            e.voice_id = voice_vocab.get(e.voice_label, 0)
        (output_dir / "voice_vocab.json").write_text(json.dumps(voice_vocab, indent=2))
        print(f"Voice vocab ({n_voices}): {voice_vocab}")
    else:
        voice_vocab = None
        n_voices    = 0

    dv, _, _ = load_all_examples(DEV_JSONL, vocab_exp, pc_exp,
                                  dev_id_set, "dev", mask=use_mask, voice_vocab=voice_vocab)
    print(f"Vocab: {len(vocab_exp)}  Pitch classes: {len(pc_exp)}  "
          f"use_mask={use_mask}  voice_conditioning={use_voice}")
    print(f"Train: {len(tr)}  Dev: {len(dv)}")

    # Model config
    mcfg = {
        "vocab_size":         len(vocab_exp),
        "num_pitch_classes":  len(pc_exp),
        "d_model":            cfg["d_model"],
        "nhead":              N_HEADS,
        "num_layers":         cfg["num_layers"],
        "dim_feedforward":    cfg["dim_ff"],
        "dropout":            DROPOUT_EXPERIMENT,
        "max_len":            MAX_LEN,
        "pad_id":             vocab_exp[PAD_TOKEN],
        "voice_conditioning": use_voice,
        "n_voices":           n_voices,
        "voice_dim":          voice_dim,
    }
    if use_mask:
        mcfg["mask_token_id"] = vocab_exp[MASK_TOKEN]
        mcfg["mask_error"]    = True
    mcfg["checkpoint_by"] = cfg["checkpoint_by"]
    _nm    = {"mask_token_id", "mask_error", "checkpoint_by"}
    model  = CorrectionModel(**{k: v for k, v in mcfg.items() if k not in _nm}).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    (output_dir / "model_config.json").write_text(json.dumps(mcfg, indent=2))
    print(f"Params: {n_params:,}  checkpoint_by={cfg['checkpoint_by']}")

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                   weight_decay=cfg["weight_decay"])

    # Epoch 0 baseline
    tl0, tt1_0, tt5_0, _ = evaluate_dataset(model, tr, cfg["batch_size"], loss_fn, device)
    dl0, dt1_0, dt5_0, _ = evaluate_dataset(model, dv, cfg["batch_size"], loss_fn, device)
    print(f"epoch 0: train {tl0:.4f}/{tt1_0:.2%}  dev {dl0:.4f}/{dt1_0:.2%}")

    metrics = [{"epoch":0, "train_loss":round(tl0,5), "train_top1":round(tt1_0,4),
                "train_top5":round(tt5_0,4), "dev_loss":round(dl0,5),
                "dev_top1":round(dt1_0,4),   "dev_top5":round(dt5_0,4),
                "elapsed_s":round(time.time()-start_wall,1)}]

    best_loss, best_top1, best_epoch, patience_count = dl0, dt1_0, 0, 0
    ckpt_by = cfg["checkpoint_by"]

    for epoch in range(1, cfg["max_epochs"] + 1):
        if time.time() - start_wall >= cfg["time_limit"]:
            print(f"Time limit reached before epoch {epoch}."); break
        es = time.time()
        tr_l, tr_t1, tr_t5 = run_train_epoch(
            model, tr, cfg["batch_size"], optimizer, loss_fn, device, rng)
        d_l, d_t1, d_t5, _ = evaluate_dataset(
            model, dv, cfg["batch_size"], loss_fn, device)
        elapsed = time.time() - start_wall
        metrics.append({"epoch":epoch, "train_loss":round(tr_l,5),
                        "train_top1":round(tr_t1,4), "train_top5":round(tr_t5,4),
                        "dev_loss":round(d_l,5), "dev_top1":round(d_t1,4),
                        "dev_top5":round(d_t5,4), "elapsed_s":round(elapsed,1)})
        print(f"  epoch {epoch:2d}  train {tr_l:.4f}/{tr_t1:.2%}  "
              f"dev {d_l:.4f}/{d_t1:.2%}  [{time.time()-es:.0f}s]")

        # Checkpoint criterion: top1 with tie-break on loss, or pure loss
        if ckpt_by == "top1":
            save_ckpt = (d_t1 > best_top1) or (d_t1 == best_top1 and d_l < best_loss)
            ptn_reset = d_t1 > best_top1      # patience on strict top1 gain only
        else:
            save_ckpt = ptn_reset = d_l < best_loss

        if save_ckpt:
            best_loss, best_top1, best_epoch = d_l, d_t1, epoch
            torch.save({"model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "epoch": epoch, "dev_loss": d_l, "dev_top1": d_t1},
                       output_dir / "best_checkpoint.pt")
            print(f"    ↑ best (by {ckpt_by})")

        if ptn_reset:
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg["patience"]:
                print("  Early stopping."); break

    # Save metrics
    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    fields = ["epoch","train_loss","train_top1","train_top5",
              "dev_loss","dev_top1","dev_top5","elapsed_s"]
    with open(output_dir / "metrics.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(metrics)

    # Reload best checkpoint and verify
    print(f"\nReloading best checkpoint (epoch {best_epoch})...")
    ckpt = torch.load(output_dir / "best_checkpoint.pt",
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    d_l_v, d_t1_v, d_t5_v, best_preds = evaluate_dataset(
        model, dv, cfg["batch_size"], loss_fn, device)
    best_row = next(m for m in metrics if m["epoch"] == best_epoch)
    assert abs(d_t1_v - best_row["dev_top1"]) < 1e-3, \
        f"Reload mismatch: {d_t1_v} vs {best_row['dev_top1']}"
    n_correct = sum(1 for e, p in zip(dv, best_preds) if e.target_class == p)
    print(f"Reload PASSED  dev top-1={n_correct}/{len(dv)} = {n_correct/len(dv):.2%}")

    # Save predictions
    logits_dev  = get_logits_for_dataset(model, dv, cfg["batch_size"], device)
    predictions = build_predictions(dv, best_preds, c2p, logits_dev)
    (output_dir / "dev_predictions.json").write_text(json.dumps(predictions, indent=2))
    saved_t1 = sum(1 for p in predictions if p["correct"])
    assert saved_t1 == n_correct, f"Saved JSON count mismatch: {saved_t1} vs {n_correct}"

    total = time.time() - start_wall
    print(f"\n{'='*60}")
    print(f" DONE: {n_correct}/{len(dv)} = {n_correct/len(dv):.2%} dev top-1")
    print(f" Best epoch: {best_epoch}  Total: {total:.0f}s")
    print(f" Artifacts: {output_dir}")
    print(f"{'='*60}")
    return {"metrics": metrics, "n_correct": n_correct, "best_epoch": best_epoch,
            "dev_top1": d_t1_v, "dev_top5": d_t5_v, "n_params": n_params,
            "elapsed_s": total}


In [ ]:
# ⚠️ TRAINS — Run this cell to launch a new experiment.
# 1. Edit EXP in Cell 1 (name, hyperparams, use_mask, checkpoint_by, device).
# 2. EXP["output_dir"] must not already exist.
# 3. Section 1 (setup) must have run (train_ids, dev_ids, DEVICE must be defined).
#
# After training, add the run to RUNS with:
#   RUNS[EXP["name"]] = EXP["output_dir"]

result = run_experiment(EXP, train_ids, dev_ids, DEVICE)
print(f"\nTo add to RUNS: RUNS[{EXP['name']!r}] = RUNS_DIR / {EXP['name']!r}")


## 3b. v5 Experiment — Explicit Voice Conditioning

Control (v4 replication) vs voice-conditioned model, trained with identical settings.
Both checkpoint by highest dev top-1; ties broken by lower dev loss.

In [ ]:
# ⚙️ v5 experiment configs — identical settings, only voice_conditioning differs
_SHARED = dict(
    train_jsonl      = TRAIN_10K,
    use_mask         = True,
    voice_dim        = 32,
    d_model          = 128,
    num_layers       = 3,
    dim_ff           = 512,
    checkpoint_by    = "top1",   # saved on strict dev top-1 gain; ties → lower loss
    seed             = 42,
    batch_size       = 32,
    lr               = 1e-3,
    weight_decay     = 0.01,
    max_epochs       = 30,
    patience         = 5,
    time_limit       = 870,      # 14.5 min per run ≤ 30 min combined
    device           = None,     # None = auto-detect MPS/CUDA/CPU
)

EXP_V4_CTRL = {**_SHARED, "name": "v5_v4_control",         "voice_conditioning": False}
EXP_V5_VOICE = {**_SHARED, "name": "v5_voice_conditioned", "voice_conditioning": True}

for cfg in (EXP_V4_CTRL, EXP_V5_VOICE):
    cfg["output_dir"] = RUNS_DIR / cfg["name"]
    print(f"{cfg['name']}: voice_conditioning={cfg['voice_conditioning']}  → {cfg['output_dir']}")


In [ ]:
# ⚠️ TRAINS — v4 control (no voice conditioning)
# Run Section 1 (cells 11–12) before this cell.
_dev  = (torch.device(EXP_V4_CTRL["device"]) if EXP_V4_CTRL["device"] else
         torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu"))
result_ctrl = run_experiment(EXP_V4_CTRL, train_ids, dev_ids, _dev)


In [ ]:
# ⚠️ TRAINS — v5 voice-conditioned
_dev  = (torch.device(EXP_V5_VOICE["device"]) if EXP_V5_VOICE["device"] else
         torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu"))
result_voice = run_experiment(EXP_V5_VOICE, train_ids, dev_ids, _dev)


In [ ]:
# 📊 v4 control vs v5 voice-conditioned comparison
# Reads from saved dev_predictions.json; requires both training cells to have run.

_ctrl_dir  = RUNS_DIR / "v5_v4_control"
_voice_dir = RUNS_DIR / "v5_voice_conditioned"

def _load_preds(run_dir):
    pf = run_dir / "dev_predictions.json"
    mc = run_dir / "model_config.json"
    ck = run_dir / "best_checkpoint.pt"
    if not pf.exists():
        raise FileNotFoundError(f"No predictions at {pf}")
    preds  = json.loads(pf.read_text())
    n_corr = sum(1 for p in preds if p["correct"])
    n_tot  = len(preds)
    n_prms = json.loads(mc.read_text()) if mc.exists() else {}
    ckpt_ep = (torch.load(ck, map_location="cpu", weights_only=False)["epoch"]
               if ck.exists() else "?")
    metrics = json.loads((run_dir / "metrics.json").read_text())
    elapsed = metrics[-1]["elapsed_s"]
    return preds, n_corr, n_tot, ckpt_ep, elapsed

import torch as _t2

p_ctrl,  nc_c, nt, ep_c, el_c = _load_preds(_ctrl_dir)
p_voice, nc_v, _,  ep_v, el_v = _load_preds(_voice_dir)

mc_c  = json.loads((_ctrl_dir  / "model_config.json").read_text())
mc_v  = json.loads((_voice_dir / "model_config.json").read_text())

# Reconstruct models to count params (config already has them; just use stored n_params)
# Use metrics file n_params if available, else compute from config
def _n_params(mc):
    """Estimate total params from model config dict."""
    vc   = mc.get("voice_conditioning", False)
    vd   = mc.get("voice_dim", 32)
    nv   = mc.get("n_voices", 0)
    dm   = mc["d_model"]
    vs   = mc["vocab_size"]
    pc   = mc["num_pitch_classes"]
    ml   = mc["max_len"]
    nl   = mc["num_layers"]
    df   = mc["dim_feedforward"]
    nh   = mc["nhead"]
    emb  = vs * dm + ml * dm
    lay  = nl * (3*dm*dm + 3*dm + dm*dm + dm + dm*df + df + df*dm + dm + 2*2*dm)
    cin  = dm + vd if vc else dm
    clf  = cin * pc + pc
    vemb = (nv + 1) * vd if vc else 0
    return emb + lay + clf + vemb

np_c = _n_params(mc_c)
np_v = _n_params(mc_v)

# Dev top-5
def _top5(preds):
    return sum(1 for p in preds if any(c == p["target_token"]
               for c in p.get("top5_candidates", []))) / len(preds)

t5_c = _top5(p_ctrl)
t5_v = _top5(p_voice)

# ── Summary table ────────────────────────────────────────────────────────────
hdr = f"{'':26} {'v4_control':>16} {'v5_voice':>16} {'Δ':>8}"
sep = "─" * len(hdr)
print(hdr); print(sep)
print(f"{'Dev top-1':<26} {nc_c}/{nt}={nc_c/nt:.2%}  {nc_v}/{nt}={nc_v/nt:.2%}  "
      f"  {(nc_v-nc_c)/nt:+.2%}")
print(f"{'Dev top-5':<26} {t5_c:>16.2%} {t5_v:>16.2%} {t5_v-t5_c:>+8.2%}")
print(f"{'Params (est.)':<26} {np_c:>16,} {np_v:>16,} {np_v-np_c:>+8,}")
print(f"{'Best epoch':<26} {ep_c:>16} {ep_v:>16}")
print(f"{'Train wall time (s)':<26} {el_c:>16.0f} {el_v:>16.0f}")

# ── Per-voice breakdown ───────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"Per-voice accuracy")
print(f"{'─'*70}")

voice_data = {}
for src, preds in [("ctrl", p_ctrl), ("voice", p_voice)]:
    for p in preds:
        vl = p.get("voice_label") or "(None)"
        if vl not in voice_data:
            voice_data[vl] = {"n": 0, "ctrl": 0, "voice": 0}
        voice_data[vl]["n"]    += 1  # counted from ctrl side (same examples)
        voice_data[vl][src]    += int(p["correct"])

# Remove double-counting of n (both preds have same examples)
for vl in voice_data:
    voice_data[vl]["n"] //= 2   # each label was counted from both preds loops

# Re-count from ctrl only
voice_data2 = {}
for p in p_ctrl:
    vl = p.get("voice_label") or "(None)"
    if vl not in voice_data2:
        voice_data2[vl] = {"n": 0, "ctrl": 0}
    voice_data2[vl]["n"] += 1
    voice_data2[vl]["ctrl"] += int(p["correct"])
for p in p_voice:
    vl = p.get("voice_label") or "(None)"
    voice_data2.setdefault(vl, {"n": 0, "ctrl": 0})["__voice"] = voice_data2.setdefault(vl, {}).get("__voice", 0) + int(p["correct"])

# Clean approach
vd = {}
for p in p_ctrl:
    vl = p.get("voice_label") or "(None)"
    vd.setdefault(vl, {"n":0,"c":0,"v":0})["n"] += 1
    vd[vl]["c"] += int(p["correct"])
for p in p_voice:
    vl = p.get("voice_label") or "(None)"
    vd.setdefault(vl, {"n":0,"c":0,"v":0})["v"] += int(p["correct"])

vhdr = f"{'Voice':<12} {'N':>5} {'ctrl top-1':>12} {'voice top-1':>13} {'Δ':>8}"
print(vhdr)
print("─" * len(vhdr))
for vl, d in sorted(vd.items(), key=lambda x: -x[1]["n"]):
    n = d["n"]
    if n == 0: continue
    ac = d["c"] / n; av = d["v"] / n
    print(f"{vl:<12} {n:>5} {d['c']:>4}/{n}={ac:.1%} {d['v']:>4}/{n}={av:.1%} {av-ac:>+8.1%}")

# ── Venn: common/unique correct ───────────────────────────────────────────────
sid_ctrl  = {p["sample_id"] for p in p_ctrl  if p["correct"]}
sid_voice = {p["sample_id"] for p in p_voice if p["correct"]}
sid_all   = {p["sample_id"] for p in p_ctrl}
both     = sid_ctrl & sid_voice
only_c   = sid_ctrl  - sid_voice
only_v   = sid_voice - sid_ctrl
neither  = sid_all - sid_ctrl - sid_voice

print(f"\n{'─'*50}")
print(f"Prediction overlap (N={len(sid_all)})")
print(f"{'─'*50}")
print(f"  Both correct    : {len(both):3d}/{len(sid_all)} = {len(both)/len(sid_all):.1%}")
print(f"  ctrl-only       : {len(only_c):3d}/{len(sid_all)} = {len(only_c)/len(sid_all):.1%}")
print(f"  voice-only      : {len(only_v):3d}/{len(sid_all)} = {len(only_v)/len(sid_all):.1%}")
print(f"  Neither correct : {len(neither):3d}/{len(sid_all)} = {len(neither)/len(sid_all):.1%}")

# ── Octave error comparison ───────────────────────────────────────────────────
def _octave_errors(preds):
    wrong = [p for p in preds if not p["correct"]]
    if not wrong: return 0, len(preds) - len(wrong), len(preds)
    oct_ok = sum(1 for p in wrong
                 if p["target_token"][1] == p["predicted_token"][1])
    return oct_ok, len(preds) - len(wrong), len(preds)

oc_c, nc_co, nt2 = _octave_errors(p_ctrl)
oc_v, nc_vo, _   = _octave_errors(p_voice)
n_wrong_c = nt2 - nc_co
n_wrong_v = nt2 - nc_vo

print(f"\n{'─'*50}")
print(f"Octave errors among wrong predictions")
print(f"{'─'*50}")
if n_wrong_c:
    print(f"  ctrl  wrong={n_wrong_c}  correct octave={oc_c}/{n_wrong_c}={oc_c/n_wrong_c:.1%}")
if n_wrong_v:
    print(f"  voice wrong={n_wrong_v}  correct octave={oc_v}/{n_wrong_v}={oc_v/n_wrong_v:.1%}")

print(f"\nArtifacts:")
print(f"  ctrl  → {_ctrl_dir}")
print(f"  voice → {_voice_dir}")


## 3c. v6 Experiment — Larger Voice-Conditioned Model

Same as v5 (MASK + voice conditioning), only architecture size changes:
`d_model 128→256`, `num_layers 3→4`, `dim_feedforward 512→1024`.

In [ ]:
# ⚙️ v6 experiment config
# Identical to v5_voice_conditioned except architecture size.
EXP_V6 = {
    "name":               "v6_voice_large",
    "train_jsonl":        TRAIN_10K,
    "use_mask":           True,
    "voice_conditioning": True,
    "voice_dim":          32,      # unchanged from v5
    "d_model":            256,     # ↑ 128 → 256
    "num_layers":         4,       # ↑ 3   → 4
    "dim_ff":             1024,    # ↑ 512 → 1024
    "checkpoint_by":      "top1",
    "seed":               42,
    "batch_size":         32,
    "lr":                 1e-3,
    "weight_decay":       0.01,
    "max_epochs":         30,
    "patience":           5,
    "time_limit":         1800,
    "device":             None,
}
EXP_V6["output_dir"] = RUNS_DIR / EXP_V6["name"]
print(f"v6: d_model={EXP_V6['d_model']} num_layers={EXP_V6['num_layers']} "
      f"dim_ff={EXP_V6['dim_ff']}  → {EXP_V6['output_dir']}")


In [ ]:
# ⚠️ TRAINS — v6 larger voice-conditioned model
# Requires Section 1 (cells 11–12) to have run first.
_dev6 = (torch.device(EXP_V6["device"]) if EXP_V6["device"] else
         torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu"))
result_v6 = run_experiment(EXP_V6, train_ids, dev_ids, _dev6)


In [ ]:
# 📊 v5 vs v6 comparison
# Loads saved dev_predictions.json from both runs; verifies counts from artifacts.

import torch as _t3

_v5_dir = RUNS_DIR / "v5_voice_conditioned"
_v6_dir = RUNS_DIR / "v6_voice_large"

def _load_run(run_dir):
    preds   = json.loads((run_dir / "dev_predictions.json").read_text())
    mc      = json.loads((run_dir / "model_config.json").read_text())
    metrics = json.loads((run_dir / "metrics.json").read_text())
    ck      = _t3.load(run_dir / "best_checkpoint.pt",
                       map_location="cpu", weights_only=False)
    n_corr  = sum(1 for p in preds if p["correct"])
    t5      = sum(1 for p in preds
                  if p["target_token"] in p.get("top5_candidates", []))
    return {
        "preds":    preds,
        "n_corr":   n_corr,
        "n_tot":    len(preds),
        "t5":       t5,
        "ckpt_ep":  ck["epoch"],
        "elapsed":  metrics[-1]["elapsed_s"],
        "n_ep":     len(metrics) - 1,
        "mc":       mc,
    }

r5 = _load_run(_v5_dir)
r6 = _load_run(_v6_dir)
N  = r5["n_tot"]

# Verify v5 reference (83/499 top-1, 261/499 top-5)
assert r5["n_corr"] == 83,  f"v5 top-1 mismatch: {r5['n_corr']}"
assert r5["t5"]     == 261, f"v5 top-5 mismatch: {r5['t5']}"

# ── Summary table ─────────────────────────────────────────────────────────────
def _sep(w=66): return "─" * w
hdr = f"{'':28} {'v5_voice (small)':>18} {'v6_voice_large':>18} {'Δ':>7}"
print(hdr); print(_sep())
nc5, nc6 = r5["n_corr"], r6["n_corr"]
t55, t56 = r5["t5"],     r6["t5"]
print(f"{'Dev top-1':<28} {nc5}/{N}={nc5/N:.2%}  {nc6}/{N}={nc6/N:.2%}  {(nc6-nc5)/N:>+.2%}")
print(f"{'Dev top-5':<28} {t55}/{N}={t55/N:.2%}  {t56}/{N}={t56/N:.2%}  {(t56-t55)/N:>+.2%}")
print(f"{'Best epoch / total':<28} {r5['ckpt_ep']}/{r5['n_ep']:>18} "
      f"{r6['ckpt_ep']}/{r6['n_ep']:>18}")
print(f"{'Wall time (s)':<28} {r5['elapsed']:>18.0f} {r6['elapsed']:>18.0f}")

for key, label in [("d_model","d_model"), ("num_layers","num_layers"),
                    ("dim_feedforward","dim_ff")]:
    print(f"{label:<28} {r5['mc'].get(key,'?'):>18} {r6['mc'].get(key,'?'):>18}")

# Param count: sum all tensors in best_checkpoint.pt model_state_dict
def _param_count(run_dir):
    sd = _t3.load(run_dir / "best_checkpoint.pt",
                  map_location="cpu", weights_only=False)["model_state_dict"]
    return sum(v.numel() for v in sd.values())

np5 = _param_count(_v5_dir)
np6 = _param_count(_v6_dir)
print(f"{'Params':<28} {np5:>18,} {np6:>18,}  {np6-np5:>+7,}")

# ── Per-voice breakdown ────────────────────────────────────────────────────────
print(f"\n{_sep()}")
print("Per-voice accuracy  (all 499 examples, UNKNOWN included)")
print(_sep())

vd = {}
for p in r5["preds"]:
    vl = p.get("voice_label") or "(None/UNKNOWN)"
    vd.setdefault(vl, {"n":0,"v5":0,"v6":0})["n"]  += 1
    vd[vl]["v5"] += int(p["correct"])
for p in r6["preds"]:
    vl = p.get("voice_label") or "(None/UNKNOWN)"
    vd.setdefault(vl, {"n":0,"v5":0,"v6":0})["v6"] += int(p["correct"])

vhdr = f"{'Voice':<18} {'N':>5}  {'v5 top-1':>10} {'v6 top-1':>11}  {'Δ':>7}"
print(vhdr); print("─"*len(vhdr))
for vl, d in sorted(vd.items(), key=lambda x: -x[1]["n"]):
    n = d["n"]
    a5 = d["v5"]/n; a6 = d["v6"]/n
    print(f"{vl:<18} {n:>5}  {d['v5']:>3}/{n}={a5:.1%}  "
          f"{d['v6']:>3}/{n}={a6:.1%}  {a6-a5:>+7.1%}")

# ── Venn overlap ───────────────────────────────────────────────────────────────
sid5  = {p["sample_id"] for p in r5["preds"] if p["correct"]}
sid6  = {p["sample_id"] for p in r6["preds"] if p["correct"]}
sida  = {p["sample_id"] for p in r5["preds"]}
both   = sid5 & sid6
only5  = sid5 - sid6
only6  = sid6 - sid5
neither = sida - sid5 - sid6

print(f"\n{_sep(50)}")
print(f"Prediction overlap  (N={N})")
print(_sep(50))
print(f"  Both correct    : {len(both):3d}/{N} = {len(both)/N:.1%}")
print(f"  v5-only correct : {len(only5):3d}/{N} = {len(only5)/N:.1%}")
print(f"  v6-only correct : {len(only6):3d}/{N} = {len(only6)/N:.1%}")
print(f"  Neither correct : {len(neither):3d}/{N} = {len(neither)/N:.1%}")

# ── Octave accuracy (all examples) ────────────────────────────────────────────
def _oct_acc(preds):
    """Octave-correct rate over all wrong predictions."""
    wrong = [p for p in preds if not p["correct"]]
    if not wrong: return 0.0, 0
    ok = sum(1 for p in wrong if p["target_token"][1] == p["predicted_token"][1])
    return ok / len(wrong), len(wrong)

oa5, nw5 = _oct_acc(r5["preds"])
oa6, nw6 = _oct_acc(r6["preds"])
print(f"\n{_sep(50)}")
print(f"Octave accuracy among wrong predictions")
print(_sep(50))
print(f"  v5 wrong={nw5}  correct octave: {oa5:.1%}")
print(f"  v6 wrong={nw6}  correct octave: {oa6:.1%}  Δ={oa6-oa5:+.1%}")

print(f"\nArtifacts:")
print(f"  v5 → {_v5_dir}")
print(f"  v6 → {_v6_dir}")


### v6 underfitting analysis

**Diagnostic signal — train accuracy lower than dev accuracy at best checkpoint:**

| Run | train top-1 | dev top-1 | train loss | dev loss |
|-----|-------------|-----------|------------|----------|
| v5_voice (ep 10) | 17.73% | **16.63%** | 2.56 | 2.79 |
| v6_voice_large (ep 14) | 8.03% | **11.22%** | 3.03 | 3.03 |

v5 shows the expected pattern: train > dev, loss diverging. v6 inverts it — the model predicts the training examples *less accurately* than the held-out ones. Train loss and dev loss are nearly identical (3.03 vs 3.03) and barely decreased from epoch 0 (~3.95), indicating the optimizer failed to find a useful minimum.

**Root cause: parameter-to-data ratio.**
v6 has 3,287,252 parameters trained on 9,998 examples (329 params/example). v5 has 659,924 on the same data (66 params/example). At lr=1e-3 with no warmup, the gradient signal from each batch is too sparse to steer a 3.3M-param model reliably. The model converges to a nearly random-looking minimum rather than overfitting.

**Per-voice regression is uniform**, not voice-specific: voice:1 fell from 19.2% to 11.4% (−7.7%), voice:5 from 13.3% to 11.1% (−2.2%), voice:6 from 15.8% to 5.3% (−10.5%). No voice benefited from the larger architecture. The octave-correct rate among wrong predictions also fell (49.5% → 44.2%), confirming the model learned weaker representations overall.

**Experiment conclusions:**

1. **Voice conditioning is the dominant signal** (+5.21% absolute over unvoiced control at same architecture). This result is robust.
2. **Architecture size does not help at 10K examples.** Going from 660K to 3.3M params regresses by −5.41%. The current bottleneck is data quantity, not model capacity.
3. **v5 (128d · 3L · 512ff + voice, 660K params) remains the best model**: 83/499 = 16.63% dev top-1, 261/499 = 52.30% dev top-5.

**Recommended next experiments (in order of expected yield):**

- **v7a — more training data**: generate a second 10K split (different seed) and train v5-architecture on 20K. Data quantity is the identified bottleneck; architecture is already well-matched.
- **v7b — lr schedule on v6**: train v6 with linear warmup (3 epochs) + cosine decay to test whether the large model can learn with a more careful schedule. Requires no new data.
- **v7c — voice embedding ablation**: replace the continuous learned voice embedding with a one-hot feature passed directly to the classifier head (no backprop through it). Isolates whether the voice signal is used for representation or just routing.


## 3d. v7 Experiment — Dynamic Masked-Pitch Training

**Hypothesis:** Randomising the masked position each epoch (dynamic corruption, MDTK-style)
teaches broader pitch-context representations than always masking the same fixed position.

**Design**:
- Architecture: identical to v5 (128d·3L·512ff, voice conditioning, seed=42)
- Train set: same 9,998 examples; correct tokens from LMX source files
- Each epoch: one eligible pitch position per example, selected by `Random(epoch×9999991 + jsonl_idx)`
- Eligible = any position where the token is in `pitch_classes` (52 classes)
- [MASK] applied at selected position; target = original pitch at that position
- Dev: same 499 fixed examples (masked at original error positions)
- Budget: 4,695 optimizer steps = v5 total (15 epochs × 313 steps)
- Checkpoint criterion: highest dev top-1; ties broken by lower dev loss; patience=5

**Reference (MDTK):** Cífka et al., "Groove2Groove" / dynamic corruption in discrete-token
masked prediction; each epoch sees a fresh corruption sample.


In [ ]:
# ⚡ v7 pre-processing: cache LMX source files and eligible pitch positions (run once)
# Requires Section 1 (cells 11–12) to be run first so train_ids, vocab, pitch_classes exist.
# Re-run is safe: just re-builds the cache in memory.

_v5_dir = RUNS_DIR / "v5_voice_conditioned"

# Load v5 artifacts (vocab identical to what Section 1 built; reload for explicit traceability)
_v7_vocab       = json.loads((_v5_dir / "vocab.json").read_text())
_v7_pc          = json.loads((_v5_dir / "pitch_classes.json").read_text())
_v7_voice_vocab = json.loads((_v5_dir / "voice_vocab.json").read_text())
assert _v7_vocab == vocab,          "v7 vocab differs from Section 1 vocab — re-run Section 1"
assert _v7_pc    == pitch_classes,  "v7 pitch_classes differ — re-run Section 1"
print(f"v7 artifacts match Section 1  (vocab={len(_v7_vocab)}, PC={len(_v7_pc)}, voices={len(_v7_voice_vocab)})")

# Verify no dev/train score ID overlap (redundant with Section 1 but explicit)
_tr_score_ids  = {sid.split('/')[1] for sid in train_ids}
_dev_score_ids = {sid.split('/')[1] for sid in dev_ids}
_overlap = _tr_score_ids & _dev_score_ids
assert not _overlap, f"Dev/train score ID overlap: {_overlap}"
print(f"Score ID overlap: CLEAN  (train={len(_tr_score_ids)}, dev={len(_dev_score_ids)})")

# Cache LMX source files: sample_id → correct token list
# Eligible positions: sample_id → [(src_pos, pitch_tok), ...]
print("Caching LMX source files...")
_v7_lmx_tokens   = {}
_v7_eligible_pos = {}
_v7_missing      = []

for _e in train_examples:
    _sid = _e.sample_id
    if _sid in _v7_lmx_tokens:
        continue
    _p = DATASET_ROOT / f"{_sid}.lmx"
    if not _p.exists():
        _v7_missing.append(_sid); continue
    _toks = _p.read_text().split()
    _v7_lmx_tokens[_sid]   = _toks
    _v7_eligible_pos[_sid] = [(i, t) for i, t in enumerate(_toks) if t in _v7_pc]

if _v7_missing:
    print(f"WARNING: {len(_v7_missing)} LMX files not found")
else:
    print(f"All {len(_v7_lmx_tokens)} LMX source files loaded")

_total_elig = sum(len(v) for v in _v7_eligible_pos.values())
_min_elig   = min(len(v) for v in _v7_eligible_pos.values())
_max_elig   = max(len(v) for v in _v7_eligible_pos.values())
print(f"Eligible positions: total={_total_elig:,}  "
      f"per-sample min={_min_elig}  max={_max_elig}  mean={_total_elig/len(_v7_eligible_pos):.1f}")

# Smoke test: 5 random samples
import random as _rnd
_srng = _rnd.Random(0)
for _sid in _srng.sample(list(_v7_lmx_tokens.keys()), 5):
    _toks  = _v7_lmx_tokens[_sid]
    _elig  = _v7_eligible_pos[_sid]
    _pos, _ptok = _elig[len(_elig) // 2]
    assert _toks[_pos] == _ptok
    _win, _wpos, _ = window_tokens(_toks, _pos, _ptok)
    _vlabel = build_voice_map(_win).get(_wpos)
    _win_m  = list(_win); _win_m[_wpos] = MASK_TOKEN
    assert _win_m[_wpos] == MASK_TOKEN
    assert _v7_pc.get(_ptok) is not None
    print(f"  {_sid}: pos={_pos}  pitch={_ptok}  voice={_vlabel}  win={len(_win)}")
print("Smoke test PASSED — LMX cache ready for v7 training")


In [ ]:
# ⚙️ v7 dynamic masking config
# Budget: v5 total optimizer steps = 15 training epochs × ceil(9998/32)=313 steps = 4,695

_V7_STEP_BUDGET = 15 * 313  # = 4695 (matches v5 total; verified from metrics.csv)

EXP_V7_DYNAMIC = {
    "name":              "v7_dynamic_mask",
    "d_model":           128,     # same as v5
    "num_layers":        3,       # same as v5
    "dim_ff":            512,     # same as v5
    "voice_conditioning": True,
    "voice_dim":         32,
    "checkpoint_by":     "top1",
    "seed":              42,
    "batch_size":        32,
    "lr":                1e-3,
    "weight_decay":      0.01,
    "max_epochs":        15,      # ceil(4695 / 313) = 15
    "patience":          5,
    "time_limit":        600,     # 10 min wall-time limit
    "step_budget":       _V7_STEP_BUDGET,
    "output_dir":        RUNS_DIR / "v7_dynamic_mask",
}
print(f"v7 config: step_budget={_V7_STEP_BUDGET}  max_epochs={EXP_V7_DYNAMIC['max_epochs']}  "
      f"arch={EXP_V7_DYNAMIC['d_model']}d/{EXP_V7_DYNAMIC['num_layers']}L/{EXP_V7_DYNAMIC['dim_ff']}ff")
print(f"Output: {EXP_V7_DYNAMIC['output_dir']}")


In [ ]:
# Helper: dynamic example builder
# Each call produces a fresh set of examples with different masked positions.

def _make_dynamic_epoch(epoch_num, base_examples, lmx_tokens, eligible_pos,
                         vocab, pitch_classes, voice_vocab):
    """
    Build training examples for epoch_num using deterministic dynamic masking.

    Position selection: random.Random(epoch_num * 9999991 + jsonl_idx).randrange(len(positions))
    Ensures each (epoch, sample) pair always maps to the same position → reproducible.
    'replacement_token' is set to the masked pitch (= target) to satisfy CorrectionExample.

    Returns (examples_list, seen_positions_set).
    """
    unk_id, pad_id = vocab[UNK_TOKEN], vocab[PAD_TOKEN]
    examples, seen_pos = [], set()
    for idx, base_ex in enumerate(base_examples):
        sid       = base_ex.sample_id
        positions = eligible_pos.get(sid, [])
        if not positions:
            continue
        pos_idx          = random.Random(epoch_num * 9999991 + idx).randrange(len(positions))
        src_pos, pitch_tok = positions[pos_idx]
        src_toks         = lmx_tokens[sid]
        window, w_pos, w_lo = window_tokens(src_toks, src_pos, pitch_tok)
        voice_label      = build_voice_map(window).get(w_pos)
        voice_id         = voice_vocab.get(voice_label, 0)
        window           = list(window)
        window[w_pos]    = MASK_TOKEN
        ids     = [vocab.get(t, unk_id) for t in window]
        pad_len = MAX_LEN - len(ids)
        examples.append(CorrectionExample(
            sample_id         = sid,
            token_ids         = ids + [pad_id] * pad_len,
            padding_mask      = [False] * len(ids) + [True] * pad_len,
            window_error_pos  = w_pos,
            target_class      = pitch_classes[pitch_tok],
            window_lo         = w_lo,
            target_token      = pitch_tok,
            replacement_token = pitch_tok,   # dynamic: masking the correct token
            voice_label       = voice_label,
            voice_id          = voice_id,
        ))
        seen_pos.add((sid, src_pos))
    return examples, seen_pos


def run_experiment_dynamic(cfg, base_train, dev_examples,
                            lmx_tokens, eligible_pos, vocab, pitch_classes,
                            voice_vocab, device):
    """
    ⚠️ TRAINS — Dynamic masked-pitch training loop.

    Each epoch a fresh set of examples is generated via _make_dynamic_epoch.
    Train metrics are online (from training batches); dev is fully evaluated once per epoch.
    Budget enforced by both max_epochs and step_budget.
    """
    output_dir = cfg["output_dir"]
    if output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(f"Output dir not empty: {output_dir}")
    output_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"])
    rng        = random.Random(cfg["seed"])
    start_wall = time.time()
    n_voices   = len(voice_vocab)
    c2p        = {v: k for k, v in pitch_classes.items()}

    (output_dir / "vocab.json").write_text(json.dumps(vocab, indent=2))
    (output_dir / "pitch_classes.json").write_text(json.dumps(pitch_classes, indent=2))
    (output_dir / "voice_vocab.json").write_text(json.dumps(voice_vocab, indent=2))

    mcfg = {
        "vocab_size":         len(vocab),
        "num_pitch_classes":  len(pitch_classes),
        "d_model":            cfg["d_model"],
        "nhead":              N_HEADS,
        "num_layers":         cfg["num_layers"],
        "dim_feedforward":    cfg["dim_ff"],
        "dropout":            DROPOUT_EXPERIMENT,
        "max_len":            MAX_LEN,
        "pad_id":             vocab[PAD_TOKEN],
        "voice_conditioning": True,
        "n_voices":           n_voices,
        "voice_dim":          cfg["voice_dim"],
        "mask_token_id":      vocab[MASK_TOKEN],
        "mask_error":         True,
        "checkpoint_by":      cfg["checkpoint_by"],
    }
    _nm    = {"mask_token_id", "mask_error", "checkpoint_by"}
    model  = CorrectionModel(**{k: v for k, v in mcfg.items() if k not in _nm}).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    (output_dir / "model_config.json").write_text(json.dumps(mcfg, indent=2))
    print(f"Params: {n_params:,}  checkpoint_by={cfg['checkpoint_by']}")

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                   weight_decay=cfg["weight_decay"])

    # Epoch 0: dev baseline only (no train pass)
    dl0, dt1_0, dt5_0, _ = evaluate_dataset(model, dev_examples, cfg["batch_size"], loss_fn, device)
    print(f"epoch  0 (baseline): dev {dl0:.4f}/{dt1_0:.2%}")

    metrics = [{
        "epoch": 0, "train_loss": None, "train_top1": None, "train_top5": None,
        "dev_loss": round(dl0,5), "dev_top1": round(dt1_0,4), "dev_top5": round(dt5_0,4),
        "elapsed_s": round(time.time()-start_wall,1),
        "total_steps": 0, "distinct_positions_epoch": 0,
    }]

    best_loss=dl0; best_top1=dt1_0; best_epoch=0; patience_count=0
    total_steps = 0
    all_seen_pos = set()

    for epoch in range(1, cfg["max_epochs"] + 1):
        if time.time() - start_wall >= cfg["time_limit"]:
            print(f"Time limit reached before epoch {epoch}."); break
        if total_steps >= cfg["step_budget"]:
            print(f"Step budget ({cfg['step_budget']}) reached."); break

        es = time.time()
        tr_epoch, seen_pos = _make_dynamic_epoch(
            epoch, base_train, lmx_tokens, eligible_pos, vocab, pitch_classes, voice_vocab)
        all_seen_pos |= seen_pos
        steps_epoch = (len(tr_epoch) + cfg["batch_size"] - 1) // cfg["batch_size"]

        # Train (online metrics — not a separate eval pass over train)
        tr_l, tr_t1, tr_t5 = run_train_epoch(
            model, tr_epoch, cfg["batch_size"], optimizer, loss_fn, device, rng)
        total_steps += steps_epoch

        d_l, d_t1, d_t5, _ = evaluate_dataset(
            model, dev_examples, cfg["batch_size"], loss_fn, device)
        elapsed = time.time() - start_wall
        metrics.append({
            "epoch": epoch, "train_loss": round(tr_l,5),
            "train_top1": round(tr_t1,4), "train_top5": round(tr_t5,4),
            "dev_loss": round(d_l,5), "dev_top1": round(d_t1,4), "dev_top5": round(d_t5,4),
            "elapsed_s": round(elapsed,1), "total_steps": total_steps,
            "distinct_positions_epoch": len(seen_pos),
        })
        print(f"  epoch {epoch:2d}  train {tr_l:.4f}/{tr_t1:.2%}  "
              f"dev {d_l:.4f}/{d_t1:.2%}  pos={len(seen_pos)}  "
              f"steps={total_steps}  [{time.time()-es:.0f}s]")

        save_ckpt = (d_t1 > best_top1) or (d_t1 == best_top1 and d_l < best_loss)
        ptn_reset = d_t1 > best_top1
        if save_ckpt:
            best_loss, best_top1, best_epoch = d_l, d_t1, epoch
            torch.save({"model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "epoch": epoch, "dev_loss": d_l, "dev_top1": d_t1},
                       output_dir / "best_checkpoint.pt")
            print(f"    ↑ best")
        if ptn_reset:
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg["patience"]:
                print("  Early stopping."); break

    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    fields = ["epoch","train_loss","train_top1","train_top5","dev_loss","dev_top1","dev_top5",
              "elapsed_s","total_steps","distinct_positions_epoch"]
    with open(output_dir / "metrics.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(metrics)

    # Coverage report
    total_elig_all = sum(len(v) for v in eligible_pos.values())
    cov_info = {
        "unique_positions_seen":    len(all_seen_pos),
        "total_eligible_positions": total_elig_all,
        "coverage_pct":             round(len(all_seen_pos) / total_elig_all * 100, 2),
        "training_epochs":          epoch,
        "total_optimizer_steps":    total_steps,
        "v5_step_budget":           cfg["step_budget"],
        "budget_used_pct":          round(total_steps / cfg["step_budget"] * 100, 1),
    }
    (output_dir / "coverage.json").write_text(json.dumps(cov_info, indent=2))
    print(f"\nCoverage: {len(all_seen_pos):,}/{total_elig_all:,} = {cov_info['coverage_pct']:.1f}%")
    print(f"Steps: {total_steps}/{cfg['step_budget']} = {cov_info['budget_used_pct']:.0f}%")

    # Reload and verify
    print(f"\nReloading best checkpoint (epoch {best_epoch})...")
    ckpt = torch.load(output_dir / "best_checkpoint.pt",
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    d_l_v, d_t1_v, d_t5_v, best_preds = evaluate_dataset(
        model, dev_examples, cfg["batch_size"], loss_fn, device)
    best_row = next(m for m in metrics if m["epoch"] == best_epoch)
    assert abs(d_t1_v - best_row["dev_top1"]) < 1e-3,         f"Reload mismatch: {d_t1_v:.4f} vs {best_row['dev_top1']:.4f}"
    n_correct = sum(1 for e, p in zip(dev_examples, best_preds) if e.target_class == p)
    print(f"Reload PASSED  dev top-1={n_correct}/499 = {n_correct/499:.2%}")

    logits_dev  = get_logits_for_dataset(model, dev_examples, cfg["batch_size"], device)
    predictions = build_predictions(dev_examples, best_preds, c2p, logits_dev)
    (output_dir / "dev_predictions.json").write_text(json.dumps(predictions, indent=2))
    saved_t1 = sum(1 for p in predictions if p["correct"])
    assert saved_t1 == n_correct

    total = time.time() - start_wall
    print(f"\n{'='*60}")
    print(f" DONE: {n_correct}/499 = {n_correct/499:.2%} dev top-1")
    top5_count = sum(1 for p in predictions if p['target_token'] in p['top5_candidates'])
    print(f"       dev top-5 = {top5_count}/499 = {top5_count/499:.2%}")
    print(f" Best epoch: {best_epoch}  Total: {total:.0f}s  Steps: {total_steps}")
    print(f" Artifacts: {output_dir}")
    print(f"{'='*60}")
    return {"metrics": metrics, "n_correct": n_correct, "best_epoch": best_epoch,
            "dev_top1": d_t1_v, "dev_top5": d_t5_v, "n_params": n_params,
            "elapsed_s": total, "coverage": cov_info}


In [ ]:
# ⚠️ TRAINS — v7 dynamic masked-pitch
# Requires: Section 1 (cells 11–12) + v7 pre-processing cell (cell v7b_cache_lmx) already run.

_dev_v7 = torch.device(EXP_V7_DYNAMIC.get("device") or
                        ("cuda" if torch.cuda.is_available() else
                         "mps"  if torch.backends.mps.is_available() else "cpu"))
print(f"Device: {_dev_v7}")

_dev_ex_v7, _, _ = load_all_examples(
    DEV_JSONL, _v7_vocab, _v7_pc, dev_id_set, "dev",
    mask=True, voice_vocab=_v7_voice_vocab)
print(f"Dev examples: {len(_dev_ex_v7)}")

result_v7 = run_experiment_dynamic(
    EXP_V7_DYNAMIC, train_examples, _dev_ex_v7,
    _v7_lmx_tokens, _v7_eligible_pos,
    _v7_vocab, _v7_pc, _v7_voice_vocab, _dev_v7)


In [ ]:
# 📊 v5 vs v7 dynamic — comparison
# Reads from saved dev_predictions.json; verifies v5 reference (83/499, 261/499).

import json
from pathlib import Path
from collections import defaultdict

def _load_v7_preds(run_dir):
    p = run_dir / "dev_predictions.json"
    preds = json.loads(p.read_text())
    t1    = sum(1 for x in preds if x["correct"])
    t5    = sum(1 for x in preds if x["target_token"] in x["top5_candidates"])
    mc    = json.loads((run_dir / "metrics.json").read_text())
    best  = max((m for m in mc if m["train_loss"] is not None),
                key=lambda m: (m["dev_top1"], -m["dev_loss"]))
    cov   = json.loads((run_dir / "coverage.json").read_text()) if (run_dir / "coverage.json").exists() else {}
    return preds, t1, t5, best, cov

_v5_dir = RUNS_DIR / "v5_voice_conditioned"
_v7_dir = RUNS_DIR / "v7_dynamic_mask"

_preds5, _t1_5, _t5_5, _best5, _     = _load_v7_preds(_v5_dir)
_preds7, _t1_7, _t5_7, _best7, _cov7 = _load_v7_preds(_v7_dir)

# Assert v5 reference
assert _t1_5 == 83, f"v5 top-1 changed: {_t1_5} ≠ 83"
assert _t5_5 == 261, f"v5 top-5 changed: {_t5_5} ≠ 261"

N = 499
print(f"\n{'─'*60}")
print(f"{'Model':<22} {'Top-1':>8} {'Top-5':>8} {'Best ep':>8} {'Steps':>7}")
print(f"{'─'*60}")
print(f"{'v5 voice (static)':<22} {_t1_5:>4}/{N} {_t1_5/N:>5.1%}   "
      f"{_t5_5:>4}/{N} {_t5_5/N:>5.1%}   {_best5['epoch']:>5}   {'4695':>7}")
_steps7 = _best7.get('total_steps', _cov7.get('total_optimizer_steps','?'))
print(f"{'v7 dynamic mask':<22} {_t1_7:>4}/{N} {_t1_7/N:>5.1%}   "
      f"{_t5_7:>4}/{N} {_t5_7/N:>5.1%}   {_best7['epoch']:>5}   {_steps7:>7}")
print(f"{'─'*60}")
delta = _t1_7 - _t1_5
print(f"Δ top-1: {delta:+d} ({delta/N:+.2%})")
if _cov7:
    print(f"\nPosition coverage: {_cov7.get('unique_positions_seen',0):,} / "
          f"{_cov7.get('total_eligible_positions',0):,} = "
          f"{_cov7.get('coverage_pct',0):.1f}% of eligible pairs")
    print(f"Budget used: {_cov7.get('total_optimizer_steps',0)} / "
          f"{_cov7.get('v5_step_budget',4695)} = "
          f"{_cov7.get('budget_used_pct',0):.0f}%")

# Venn overlap
_sid5 = set(p["sample_id"] + "@" + str(p["error_position"]) for p in _preds5 if p["correct"])
_sid7 = set(p["sample_id"] + "@" + str(p["error_position"]) for p in _preds7 if p["correct"])
_both = len(_sid5 & _sid7); _only5 = len(_sid5 - _sid7); _only7 = len(_sid7 - _sid5)
print(f"\nVenn overlap (correct dev examples):")
print(f"  both correct:     {_both}")
print(f"  only v5 correct:  {_only5}  (dynamic masking lost these)")
print(f"  only v7 correct:  {_only7}  (dynamic masking gained these)")

# Per-voice accuracy
print(f"\nPer-voice accuracy:")
_vc5 = defaultdict(lambda: [0,0]); _vc7 = defaultdict(lambda: [0,0])
for p in _preds5:
    v = p.get("voice_label","unknown") or "unknown"
    _vc5[v][1] += 1; _vc5[v][0] += int(p["correct"])
for p in _preds7:
    v = p.get("voice_label","unknown") or "unknown"
    _vc7[v][1] += 1; _vc7[v][0] += int(p["correct"])
print(f"  {'Voice':<12} {'v5 top-1':>10} {'v7 top-1':>10} {'Δ':>6}")
for _v in sorted(_vc5.keys()):
    c5,n5 = _vc5[_v]; c7,n7 = _vc7.get(_v,[0,0])
    r5 = c5/n5 if n5 else 0; r7 = c7/n7 if n7 else 0
    print(f"  {_v:<12} {c5:>3}/{n5:<3} {r5:>5.1%}  {c7:>3}/{n7:<3} {r7:>5.1%}  {r7-r5:>+.1%}")

# 3 cases where v7 correct but v5 wrong, and 3 vice versa
_only7_cases = [p for p in _preds7 if p["correct"] and
                (p["sample_id"]+"@"+str(p["error_position"])) not in _sid5]
_only5_cases = [p for p in _preds5 if p["correct"] and
                (p["sample_id"]+"@"+str(p["error_position"])) not in _sid7]
print(f"\n3 examples only v7 correct:")
for p in _only7_cases[:3]:
    p7 = next((x for x in _preds7 if x["sample_id"]==p["sample_id"] and x["error_position"]==p["error_position"]), {})
    p5 = next((x for x in _preds5 if x["sample_id"]==p["sample_id"] and x["error_position"]==p["error_position"]), {})
    print(f"  {p['sample_id']} pos={p['error_position']}  target={p['target_token']}  "
          f"v5={p5.get('predicted_token','?')}  v7={p.get('predicted_token','?')}  voice={p.get('voice_label')}")
print(f"\n3 examples only v5 correct:")
for p in _only5_cases[:3]:
    p7 = next((x for x in _preds7 if x["sample_id"]==p["sample_id"] and x["error_position"]==p["error_position"]), {})
    print(f"  {p['sample_id']} pos={p['error_position']}  target={p['target_token']}  "
          f"v5={p.get('predicted_token','?')}  v7={p7.get('predicted_token','?')}  voice={p.get('voice_label')}")


### v7 dynamic masking — results and conclusions

**Result: dynamic masking hurts significantly.**

| Model | Dev top-1 | Dev top-5 | Best ep | Steps used |
|-------|-----------|-----------|---------|------------|
| v5 voice (static mask) | 83/499 = **16.63%** | 261/499 = 52.3% | 10 | 4695/4695 |
| v7 dynamic mask | 55/499 = 11.02% | 196/499 = 39.3% | 1 | 1878/4695 (40%) |

Δ top-1 = −28 (−5.61 pp).  Early stopping triggered at epoch 6 (patience=5 from epoch 1).
Budget used: 1,878 / 4,695 steps = 40%.

**Position coverage**: 57,173 / 610,301 eligible (sample_id, position) pairs seen = 9.4%.
Each epoch selects 9,998 unique pairs (one per sample); 6 training epochs × 9,998 = 57,173 total.

**Input preprocessing (verified by audit):**

| Split | Input sequence | Masked token |
|-------|---------------|--------------|
| v5 train | `input_tokens` from JSONL (corrupted — error position has replacement pitch) | `[MASK]` at error position |
| v7 train | LMX source file (correct sequence) | `[MASK]` at dynamic position |
| Both dev | `input_tokens` from JSONL (corrupted) | `[MASK]` at error position |

Audit finding: in the corrupted `input_tokens`, the token at `error_position` is the replacement
pitch; all other 110,119/110,119 non-error surrounding tokens are byte-identical to the LMX
source. The two surrounding contexts therefore differ only in WHICH position is masked,
not in surrounding content.

**Root cause analysis:**

1. **Masking-position non-stationarity.**
   v5 always masks the same (error) position per example, allowing the model to specialise
   on error-position contexts over many epochs.
   v7 masks a *different* position each epoch — the model must adapt to new positions every
   epoch, preventing the specialisation that drives v5's gain. This is consistent with:
   - best epoch = 1 (before multi-epoch shift accumulates)
   - monotone dev decay after epoch 1 (9.02% → 10.02% → 9.82% → 8.42% → 9.82%)
   - early stopping at epoch 6

2. **Train target distribution nearly identical (KL = 0.0032).**
   Pitch target frequencies in v5 train vs v7 epoch-1 train are nearly the same, so the
   drop is not explained by a shifted pitch prior.

3. **Voice conditioning alone contributed most of v5's gain.**
   v7 shares v5's architecture and voice conditioning, yet only matches v3 performance
   (55/499 = 11.02%). Removing fixed-error-position training eliminates the
   correction-specific signal that voice conditioning builds on.

**Conclusions:**

1. Dynamic masking (MDTK-style) on clean sequences **does not transfer** to the
   correction task; masking position non-stationarity dominates.
2. The beneficial signal in v5 comes from consistent masking at error-position windows,
   not from wider positional coverage.
3. The surrounding context is effectively the same for v5-train and v7-train (both use
   the LMX source sequence). The gap is entirely explained by masking position.

**Recommended next experiments (in priority order):**

- **v8a — more data (20K)**: v5 architecture + voice conditioning, doubled training set.
  Tests whether the remaining gap is a data ceiling rather than an architecture limit.
- **v8b — hybrid corruption**: 50% fixed-error-position (v5 style) + 50% dynamic per epoch.
  Tests whether positional diversity helps when error-position signal is preserved.
- **v8c — lr warmup on v5**: warmup 200 steps + cosine decay. Checks whether
  the v5 early-plateau (best ep 10, then quick early-stop) is lr-schedule limited.


## 3e. Note-Level Representation (LMX Parser)

Parse each example's `input_tokens` into per-note records.
No training; no model calls. CPU only.

**Fields per note**

| Field | Type | Description |
|-------|------|-------------|
| `sample_id` | str | JSONL sample identifier |
| `measure` | int | 1-based measure number |
| `voice` | str\|None | `voice:N` label (inherited when not re-emitted) |
| `staff` | str\|None | `staff:N` label (inherited; clef declarations excluded) |
| `onset` | Fraction | Onset within measure in whole-notes |
| `duration` | Fraction\|None | Note duration; None for `rest:measure` |
| `pitch` | str\|None | Pitch token (`A4`, …) or None for rests |
| `is_rest` | bool | |
| `is_grace` | bool | Grace notes: onset recorded but not advanced |
| `chord_group` | int | Globally unique 1-based ID; same value = simultaneous |
| `pitch_token_position` | int\|None | Index into `input_tokens` (None for rests) |
| `token_span` | (int, int) | Half-open `[start, end)` into `input_tokens` |

**Anti-leakage**: parsing uses `input_tokens` (corrupted sequence); `target_token` is
never accessed. The replacement pitch at `error_position` is treated as a normal pitch.

**Selection**: 100 train + 20 dev examples chosen for structural diversity
(grace notes, time modifications, forward tokens, rest:measure, dots).


In [ ]:
# Note-Level LMX Parser
# Defines parse_lmx_notes — call this cell once before note_repr_run / v8_prep.

from fractions import Fraction as _Frac
import re as _re

_DURATION_BASE = {
    'breve': _Frac(2),      'whole': _Frac(1),      'half': _Frac(1, 2),
    'quarter': _Frac(1, 4), 'eighth': _Frac(1, 8),  '16th': _Frac(1, 16),
    '32nd': _Frac(1, 32),   '64th': _Frac(1, 64),   '128th': _Frac(1, 128),
    '256th': _Frac(1, 256), '512th': _Frac(1, 512), '1024th': _Frac(1, 1024),
}

_DECORATION = frozenset({
    'beam:begin', 'beam:end', 'beam:forward-hook', 'beam:backward-hook',
    'tied:start', 'tied:stop', 'tuplet:start', 'tuplet:stop',
    'slur:start', 'slur:stop', 'trill-mark', 'staccato', 'accent',
    'strong-accent', 'tenuto', 'fermata', 'arpeggiate',
    'tremolo:1', 'tremolo:2', 'tremolo:3', 'tremolo:4',
    'tremolo:single', 'tremolo:start', 'tremolo:stop',
})

_ACCIDENTAL = frozenset({
    'sharp', 'flat', 'natural', 'double-sharp',
    'flat-flat', 'natural-sharp', 'natural-flat',
})

_SKIP = frozenset({'time'})   # 'time' is a separator; beats/beat-type parsed below
_TIME_MOD_RE = _re.compile(r'^(\d+)in(\d+)$')


def _parse_dur(tokens, i):
    """Parse TYPE [time_mod?] [dot*]. Returns (Fraction|None, new_i)."""
    if i >= len(tokens) or tokens[i] not in _DURATION_BASE:
        return None, i
    base = _DURATION_BASE[tokens[i]]
    i += 1
    m = _TIME_MOD_RE.match(tokens[i]) if i < len(tokens) else None
    if m:
        base *= _Frac(int(m.group(2)), int(m.group(1)))
        i += 1
    n_dots = 0
    while i < len(tokens) and tokens[i] == 'dot':
        n_dots += 1
        i += 1
    if n_dots:
        base *= (2 - _Frac(1, 2 ** n_dots))
    return base, i


def parse_lmx_notes(tokens, sample_id=''):
    """Parse a flat LMX token list into per-note records.
    Returns (notes: list[dict], warnings: list[str]).

    rest:measure: duration = beats/beat_type when time sig is known;
    duration = None + warning when time sig has not been seen in the stream.
    chord_group is globally unique across all measures.
    Anti-leakage: never reads target_token; parses input_tokens as-is.
    """
    notes, warnings = [], []
    current_voice = current_staff = None
    current_beats = current_beat_type = None   # time signature state
    measure_num = 0
    current_onset = _Frac(0)
    chord_group_id = 0
    chord_leader_onset = _Frac(0)
    i, n = 0, len(tokens)

    while i < n:
        tok = tokens[i]

        if tok == 'measure':
            measure_num += 1
            current_onset = _Frac(0)
            i += 1
            continue

        if tok in _SKIP or tok.startswith('key:'):
            i += 1
            continue

        # Time signature: parse, update state, skip
        if tok.startswith('beats:'):
            try: current_beats = int(tok[6:])
            except ValueError: warnings.append(f'bad beats token {tok!r} @ {i}')
            i += 1
            continue

        if tok.startswith('beat-type:'):
            try: current_beat_type = int(tok[10:])
            except ValueError: warnings.append(f'bad beat-type token {tok!r} @ {i}')
            i += 1
            continue

        # Clef: skip clef:X and its staff:N; do NOT update current_staff
        if tok.startswith('clef:'):
            i += 1
            if i < n and tokens[i].startswith('staff:'):
                i += 1
            continue

        if tok in _DECORATION:
            i += 1
            continue

        # ── Note prefix ──────────────────────────────────────────────────
        span_start = i
        is_grace = is_chord = False

        if tok == 'print-object:no':
            i += 1
            if i >= n: break
            tok = tokens[i]

        if tok == 'grace':
            is_grace = True
            i += 1
            if i >= n: break
            tok = tokens[i]
            if tok == 'grace:slash':
                i += 1
                if i >= n: break
                tok = tokens[i]

        if tok == 'chord':
            is_chord = True
            i += 1
            if i >= n: break
            tok = tokens[i]

        # ── Main token ───────────────────────────────────────────────────
        pitch = None
        is_rest = False
        pitch_token_position = None

        if PITCH_RE.match(tok):
            pitch, pitch_token_position = tok, i
            i += 1
        elif tok == 'rest':
            is_rest = True
            i += 1
        elif tok == 'backup':
            i += 1
            dur, i = _parse_dur(tokens, i)
            if dur is not None:
                current_onset = max(_Frac(0), current_onset - dur)
            else:
                warnings.append(f'backup: no duration @ {i}, {sample_id}')
            continue
        elif tok == 'forward':
            i += 1
            dur, i = _parse_dur(tokens, i)
            if dur is not None:
                current_onset += dur
            else:
                warnings.append(f'forward: no duration @ {i}, {sample_id}')
            continue
        else:
            warnings.append(f'unexpected token {tok!r} @ {i}, {sample_id}')
            i += 1
            continue

        # ── Voice (optional, inherited) ──────────────────────────────────
        if i < n and tokens[i].startswith('voice:'):
            current_voice = tokens[i]
            i += 1

        # ── Duration type or rest:measure ────────────────────────────────
        duration = None
        if i < n:
            if tokens[i] in _DURATION_BASE:
                duration, i = _parse_dur(tokens, i)
            elif tokens[i] == 'rest:measure':
                if current_beats is not None and current_beat_type is not None:
                    duration = _Frac(current_beats, current_beat_type)
                else:
                    duration = None
                    warnings.append(
                        f'rest:measure duration unknown (no time sig seen) '
                        f'at {sample_id} m{measure_num}')
                i += 1

        # ── Accidental, stem, staff ──────────────────────────────────────
        if i < n and tokens[i] in _ACCIDENTAL:
            i += 1
        if i < n and tokens[i].startswith('stem:'):
            i += 1
        if i < n and tokens[i].startswith('staff:'):
            current_staff = tokens[i]
            i += 1

        # ── Decoration tokens ────────────────────────────────────────────
        while i < n and tokens[i] in _DECORATION:
            i += 1

        span_end = i

        onset = chord_leader_onset if is_chord else current_onset
        if is_chord:
            grp = chord_group_id
        else:
            chord_group_id += 1
            grp = chord_group_id
            chord_leader_onset = current_onset

        notes.append({
            'sample_id':            sample_id,
            'measure':              measure_num,
            'voice':                current_voice,
            'staff':                current_staff,
            'onset':                onset,
            'duration':             duration,
            'pitch':                pitch,
            'is_rest':              is_rest,
            'is_grace':             is_grace,
            'chord_group':          grp,
            'pitch_token_position': pitch_token_position,
            'token_span':           (span_start, span_end),
        })

        if not is_chord and not is_grace and duration is not None:
            current_onset += duration

    return notes, warnings

print('parse_lmx_notes defined  (time sig tracked; rest:measure uses beats/beat-type when known)')


In [ ]:
# ⚡ Select 100 train + 20 dev examples and run the fixed parser.
# Requires: cell 1 (PROCESSED path), cell 2 (PITCH_RE), note_repr_parser cell.
# Saves: ml/data/note_repr/manifest.json, notes_train.json, notes_dev.json
# Note: always regenerates (parser was updated to fix rest:measure handling).

import json, random, os, re as _re2
from pathlib import Path

NOTE_REPR_DIR = Path('ml/data/note_repr')
NOTE_REPR_DIR.mkdir(parents=True, exist_ok=True)

_TIME_MOD_CHECK = _re2.compile(r'^\d+in\d+$')

def _feature_flags(tokens):
    flags = set()
    for t in tokens:
        if t == 'grace':        flags.add('grace')
        if t == 'forward':      flags.add('forward')
        if t == 'dot':          flags.add('dot')
        if t == 'rest:measure': flags.add('rest_measure')
        if _TIME_MOD_CHECK.match(t): flags.add('time_mod')
        if t == 'chord':        flags.add('chord')
        if t == 'backup':       flags.add('backup')
    return flags

def _select_diverse(jsonl_path, n, seed=42):
    rng = random.Random(seed)
    all_ex = []
    with open(jsonl_path) as f:
        for li, line in enumerate(f):
            ex = json.loads(line)
            all_ex.append((li, ex['sample_id'], _feature_flags(ex['input_tokens'])))
    selected = set()
    for feat in ('grace', 'time_mod', 'forward', 'rest_measure'):
        pool = [i for i, _, flags in all_ex if feat in flags and i not in selected]
        rng.shuffle(pool)
        selected.update(pool[:5])
    pool = [i for i, _, flags in all_ex if 'dot' in flags and i not in selected]
    rng.shuffle(pool)
    selected.update(pool[:10])
    pool = [i for i, _, flags in all_ex if i not in selected]
    rng.shuffle(pool)
    selected.update(pool[:n])
    return sorted(selected)[:n]

TRAIN_JSONL = PROCESSED / 'correction_train_10k.jsonl'
DEV_JSONL_NR = PROCESSED / 'correction_dev_v1.jsonl'

train_indices = _select_diverse(TRAIN_JSONL, 100, seed=42)
dev_indices   = _select_diverse(DEV_JSONL_NR, 20, seed=42)
print(f'Selected: {len(train_indices)} train  {len(dev_indices)} dev')

def _run_parser(jsonl_path, indices, split):
    results, warnings_all = [], []
    with open(jsonl_path) as f:
        all_lines = f.readlines()
    for li in indices:
        ex = json.loads(all_lines[li])
        notes, warnings = parse_lmx_notes(ex['input_tokens'], ex['sample_id'])
        warnings_all.extend(warnings)
        results.append({
            'split':             split,
            'line_index':        li,
            'sample_id':         ex['sample_id'],
            'error_position':    ex['error_position'],
            'replacement_token': ex['replacement_token'],
            'features':          sorted(_feature_flags(ex['input_tokens'])),
            'n_notes':           len(notes),
            'n_warnings':        len(warnings),
            'notes': [
                {**n, 'onset': str(n['onset']),
                 'duration': str(n['duration']) if n['duration'] is not None else None}
                for n in notes
            ],
        })
    return results, warnings_all

train_results, train_warns = _run_parser(TRAIN_JSONL, train_indices, 'train')
dev_results,   dev_warns   = _run_parser(DEV_JSONL_NR, dev_indices,  'dev')

manifest = {
    'train': [{'line_index': r['line_index'], 'sample_id': r['sample_id'],
               'features': r['features']} for r in train_results],
    'dev':   [{'line_index': r['line_index'], 'sample_id': r['sample_id'],
               'features': r['features']} for r in dev_results],
}
with open(NOTE_REPR_DIR / 'manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
with open(NOTE_REPR_DIR / 'notes_train.json', 'w') as f: json.dump(train_results, f, indent=2)
with open(NOTE_REPR_DIR / 'notes_dev.json', 'w') as f: json.dump(dev_results, f, indent=2)

from collections import Counter
all_results = train_results + dev_results
total_notes    = sum(r['n_notes'] for r in all_results)
total_warnings = len(train_warns) + len(dev_warns)
feat_counts = Counter(f for r in all_results for f in r['features'])
# Tally rest:measure duration status
n_rm_known = sum(1 for r in all_results for n in r['notes']
                 if n['is_rest'] and not n['pitch'] and n['duration'] is not None)
n_rm_unk   = sum(1 for r in all_results for n in r['notes']
                 if n['is_rest'] and not n['pitch'] and n['duration'] is None)
print(f'\nParser output (fixed rest:measure):')
print(f'  Train {len(train_results)}  Dev {len(dev_results)}  Total notes: {total_notes}')
print(f'  rest:measure notes: {n_rm_known} with computed dur, {n_rm_unk} unknown (no time sig)')
print(f'  Other warnings: {total_warnings - n_rm_unk}')
print(f'\nFeatures in selection:')
for feat, cnt in sorted(feat_counts.items(), key=lambda x: -x[1]):
    print(f'  {feat:15s}: {cnt}/{len(all_results)}')
print(f'\nArtifacts saved to {NOTE_REPR_DIR}/')


In [ ]:
# 📊 Validate parser output + show 5 manual cases.
# Requires: note_repr_run cell completed.

import json
from fractions import Fraction
from pathlib import Path

NOTE_REPR_DIR = Path("ml/data/note_repr")

# Re-load saved results to validate from file (independent re-read)
train_results = json.load(open(NOTE_REPR_DIR / "notes_train.json"))
dev_results   = json.load(open(NOTE_REPR_DIR / "notes_dev.json"))

# Reload raw JSONL for cross-checks
import json as _json
def _load_jsonl_at(path, indices):
    lines = open(path).readlines()
    return {i: _json.loads(lines[i]) for i in indices}

train_raw = _load_jsonl_at(PROCESSED / "correction_train_10k.jsonl",
                           [r['line_index'] for r in train_results])
dev_raw   = _load_jsonl_at(PROCESSED / "correction_dev_v1.jsonl",
                           [r['line_index'] for r in dev_results])

def _validate(results, raw_map):
    errs = []
    for r in results:
        li   = r['line_index']
        ex   = raw_map[li]
        toks = ex['input_tokens']
        ep   = ex['error_position']
        notes = r['notes']

        # Check 1: pitch_token_position → correct token in input_tokens
        for note in notes:
            ptp = note['pitch_token_position']
            if ptp is not None and toks[ptp] != note['pitch']:
                errs.append(f"[C1] {r['sample_id']}: pos {ptp} token={toks[ptp]!r} pitch={note['pitch']!r}")

        # Check 2: chord groups share onset
        group_onsets = {}
        for note in notes:
            g = note['chord_group']
            on = note['onset']
            if g in group_onsets and group_onsets[g] != on:
                errs.append(f"[C2] {r['sample_id']}: chord_group {g} onset {on} vs {group_onsets[g]}")
            else:
                group_onsets[g] = on

        # Check 3: voice matches build_voice_map
        vmap = build_voice_map(toks)
        for note in notes:
            ptp = note['pitch_token_position']
            if ptp is not None and ptp in vmap and note['voice'] != vmap[ptp]:
                errs.append(f"[C3] {r['sample_id']}: pos {ptp} parser={note['voice']!r} vmap={vmap[ptp]!r}")

        # Check 4: error_position maps to a note
        if not any(n['pitch_token_position'] == ep for n in notes):
            errs.append(f"[C4] {r['sample_id']}: error_position {ep} not a pitch note")

    return errs

train_errs = _validate(train_results, train_raw)
dev_errs   = _validate(dev_results,   dev_raw)

print("Validation (re-read from saved JSON):")
print(f"  Train: {len(train_results)} examples  {sum(r['n_notes'] for r in train_results)} notes  {len(train_errs)} errors")
print(f"  Dev:   {len(dev_results)} examples  {sum(r['n_notes'] for r in dev_results)} notes  {len(dev_errs)} errors")
if train_errs or dev_errs:
    for e in (train_errs + dev_errs)[:10]:
        print(" ", e)
else:
    print("  All 4 checks PASSED")

# ── 5 manual cases: one distinct example per feature ─────────────────────────
print()
print("── 5 manual verification cases ──")

SHOWCASE_FEATS = ['chord', 'backup', 'dot', 'grace', 'time_mod']
used_samples = set()

for feat in SHOWCASE_FEATS:
    # Pick the first example that has this feature and hasn't been shown yet
    r = next((r for r in train_results
               if feat in r['features'] and r['sample_id'] not in used_samples), None)
    if r is None:
        print(f"  [{feat}] no unused example found")
        continue
    used_samples.add(r['sample_id'])

    ex   = train_raw[r['line_index']]
    ep   = ex['error_position']
    toks = ex['input_tokens']
    ep_note = next((n for n in r['notes'] if n['pitch_token_position'] == ep), None)

    print(f"\n[{feat}] {r['sample_id']}  ep={ep} ({toks[ep]})  target={ex['target_token']}")
    lo = max(0, ep - 4)
    hi = min(len(toks), ep + 5)
    toks_ctx = [(f"[{t}]" if i == ep else t) for i, t in enumerate(toks[lo:hi], start=lo)]
    print(f"  tokens {lo}..{hi-1}: {' '.join(toks_ctx)}")
    if ep_note:
        print(f"  error note: pitch={ep_note['pitch']} voice={ep_note['voice']} "
              f"staff={ep_note['staff']} measure={ep_note['measure']} "
              f"onset={ep_note['onset']} dur={ep_note['duration']} "
              f"chord_group={ep_note['chord_group']} grace={ep_note['is_grace']}")
    ep_idx = next((i for i, n in enumerate(r['notes']) if n['pitch_token_position'] == ep), None)
    if ep_idx is not None:
        lo_n = max(0, ep_idx - 2)
        hi_n = min(len(r['notes']), ep_idx + 3)
        for ni in range(lo_n, hi_n):
            n = r['notes'][ni]
            mark = " ◄" if ni == ep_idx else "  "
            pitch_disp = n['pitch'] or ('REST' if n['is_rest'] else '?')
            print(f"    note[{ni}]{mark} pitch={pitch_disp:<4} v={n['voice']:<8} "
                  f"onset={n['onset']!s:<6} dur={str(n['duration']):<6} grp={n['chord_group']}")


## 3f. v8 Experiment — Structural Feature Conditioning

Add three note-level structural features to the v5 prediction head.
No changes to the Transformer encoder or voice conditioning.

| Feature | Source | Encoding |
|---------|--------|----------|
| **staff** | `staff:N` label of error note | 3-class embedding (unknown/staff:1/staff:2) |
| **onset** | Fractional onset within measure | 33-class embedding (unknown + 32 bins of 1/16 whole-note) |
| **duration** | Note duration in whole-notes | 24-class embedding (unknown + 23 log₂-scale bins) |

All features are extracted from `input_tokens` (corrupted sequence).
`target_token` is never accessed during feature extraction.

**Architecture delta from v5**

```
v5: hidden(128) + voice_emb(32)            → Linear(160→52)
v8: hidden(128) + voice_emb(32)
       + staff_emb(16) + onset_emb(16) + dur_emb(16)  → Linear(208→52)
```

Extra params: 960 (embeddings) + 2,496 (classifier) = **3,456 total** (+0.52%).

**Training**: identical to v5 — seed=42, AdamW lr=1e-3, batch=32, max 15 epochs,
patience=5, 10-min hard cap.  Single run, no hyperparameter search.


In [ ]:
# v8 preparation: structural feature extraction, model, training helpers, run function.
# Requires: cells 1-9 (config + core library), cell 41 (parse_lmx_notes).
# Struct feature caches are built once and reused; delete to force rebuild.

import math as _math

# ── Structural feature constants ─────────────────────────────────────────────
N_STAFF_CLASSES = 3    # 0=unknown, 1=staff:1, 2=staff:2
N_ONSET_BINS    = 33   # 0=unknown, 1-32 cover [0,2) in steps of 1/16 whole-note
N_DUR_BINS      = 24   # 0=unknown, 1-23 log2-scale 0.5-step from -10 to 1

_STRUCT_CACHE_DIR = Path("ml/data/note_repr")


def _staff_to_id(staff_str):
    return {'staff:1': 1, 'staff:2': 2}.get(staff_str, 0)


def _onset_to_bin(onset):
    if onset is None:
        return 0
    b = int(float(onset) * 16) + 1
    return max(1, min(32, b))


def _dur_to_bin(dur):
    if dur is None or dur <= 0:
        return 0
    try:
        b = round((_math.log2(float(dur)) + 10) * 2) + 1
        return max(1, min(23, b))
    except Exception:
        return 0


def _extract_struct(input_tokens, error_position, sample_id=''):
    # Anti-leakage: uses input_tokens as-is; target_token never accessed.
    try:
        notes, _ = parse_lmx_notes(input_tokens, sample_id)
        note = next((n for n in notes if n['pitch_token_position'] == error_position), None)
        if note is None:
            return 0, 0, 0
        return (_staff_to_id(note['staff']),
                _onset_to_bin(note['onset']),
                _dur_to_bin(note['duration']))
    except Exception:
        return 0, 0, 0


def _load_or_build_struct_cache(jsonl_path, cache_path, split_label):
    if Path(cache_path).exists():
        print(f"  Struct cache loaded: {cache_path.name}")
        return json.load(open(cache_path))
    print(f"  Building struct cache for {split_label}...", end=' ', flush=True)
    cache = []
    n_partial = 0
    with open(jsonl_path) as f:
        for line in f:
            ex = json.loads(line)
            s, o, d = _extract_struct(ex['input_tokens'], ex['error_position'],
                                      ex['sample_id'])
            cache.append([s, o, d])
            if s == 0 or o == 0 or d == 0:
                n_partial += 1
    Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
    json.dump(cache, open(cache_path, 'w'))
    total = len(cache)
    print(f"{total} examples  {n_partial} with ≥1 unknown feature ({n_partial/total:.1%})")
    return cache


# ── CorrectionExampleV8 ───────────────────────────────────────────────────────

class CorrectionExampleV8(CorrectionExample):
    __slots__ = ('staff_id', 'onset_id', 'dur_id')

    def __init__(self, *args, staff_id=0, onset_id=0, dur_id=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.staff_id = staff_id
        self.onset_id = onset_id
        self.dur_id   = dur_id


def _load_v8_examples(jsonl_path, vocab, pitch_classes, expected_id_set,
                      split_label, mask, voice_vocab, struct_cache):
    unk_id, pad_id = vocab[UNK_TOKEN], vocab[PAD_TOKEN]
    examples, unk_count = [], 0
    with open(jsonl_path) as f:
        for lineno, line in enumerate(f):
            ex    = json.loads(line)
            sid   = ex['sample_id']
            if sid not in expected_id_set:
                raise ValueError(f"{split_label} line {lineno}: {sid} not in split")
            rep    = ex['replacement_token']
            window, w_pos, w_lo = window_tokens(ex['input_tokens'], ex['error_position'], rep)
            voice_label = build_voice_map(window).get(w_pos)
            voice_id    = 0 if voice_vocab is None else voice_vocab.get(voice_label, 0)
            if mask:
                window = list(window); window[w_pos] = MASK_TOKEN
            ids     = [vocab.get(t, unk_id) for t in window]
            pad_len = MAX_LEN - len(ids)
            unk_count += ids.count(unk_id)
            ids   += [pad_id] * pad_len
            pmask  = [False] * len(window) + [True] * pad_len
            tgt    = pitch_classes[ex['target_token']]
            s, o, d = struct_cache[lineno]
            examples.append(CorrectionExampleV8(
                sample_id      = sid,
                token_ids      = ids,
                padding_mask   = pmask,
                window_error_pos = w_pos,
                target_class   = tgt,
                window_lo      = w_lo,
                target_token   = ex['target_token'],
                replacement_token = rep,
                voice_label    = voice_label,
                voice_id       = voice_id,
                staff_id       = s,
                onset_id       = o,
                dur_id         = d,
            ))
    return examples, unk_count


# ── CorrectionModelV8 ─────────────────────────────────────────────────────────

class CorrectionModelV8(nn.Module):
    # v5 Transformer + voice + structural feature embeddings
    def __init__(self, vocab_size, num_pitch_classes,
                 d_model=128, nhead=4, num_layers=3, dim_feedforward=512,
                 dropout=0.1, max_len=MAX_LEN, pad_id=0,
                 voice_conditioning=True, n_voices=0, voice_dim=32,
                 struct_dim=16):
        super().__init__()
        self.voice_conditioning = voice_conditioning
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb   = nn.Embedding(max_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=False)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                             enable_nested_tensor=False)
        head_in = d_model
        if voice_conditioning:
            self.voice_emb = nn.Embedding(n_voices + 1, voice_dim)
            head_in += voice_dim
        self.staff_emb = nn.Embedding(N_STAFF_CLASSES, struct_dim)
        self.onset_emb = nn.Embedding(N_ONSET_BINS,    struct_dim)
        self.dur_emb   = nn.Embedding(N_DUR_BINS,      struct_dim)
        head_in += 3 * struct_dim
        self.classifier = nn.Linear(head_in, num_pitch_classes)

    def forward(self, token_ids, error_pos, padding_mask, voice_ids,
                staff_ids, onset_ids, dur_ids):
        B, L = token_ids.shape
        pos  = torch.arange(L, device=token_ids.device).unsqueeze(0).expand(B, -1)
        x    = self.token_emb(token_ids) + self.pos_emb(pos)
        x    = self.encoder(x, src_key_padding_mask=padding_mask)
        h    = x[torch.arange(B, device=x.device), error_pos]
        parts = [h]
        if self.voice_conditioning:
            parts.append(self.voice_emb(voice_ids))
        parts.append(self.staff_emb(staff_ids))
        parts.append(self.onset_emb(onset_ids))
        parts.append(self.dur_emb(dur_ids))
        return self.classifier(torch.cat(parts, dim=-1))


# ── Training / eval helpers ───────────────────────────────────────────────────

def collate_v8(examples, device):
    return {
        'token_ids':      torch.tensor([e.token_ids        for e in examples], dtype=torch.long,  device=device),
        'padding_mask':   torch.tensor([e.padding_mask     for e in examples], dtype=torch.bool,  device=device),
        'error_pos':      torch.tensor([e.window_error_pos for e in examples], dtype=torch.long,  device=device),
        'target_classes': torch.tensor([e.target_class     for e in examples], dtype=torch.long,  device=device),
        'voice_ids':      torch.tensor([e.voice_id         for e in examples], dtype=torch.long,  device=device),
        'staff_ids':      torch.tensor([e.staff_id         for e in examples], dtype=torch.long,  device=device),
        'onset_ids':      torch.tensor([e.onset_id         for e in examples], dtype=torch.long,  device=device),
        'dur_ids':        torch.tensor([e.dur_id           for e in examples], dtype=torch.long,  device=device),
    }


def _fwd_v8(model, b):
    return model(b['token_ids'], b['error_pos'], b['padding_mask'], b['voice_ids'],
                 b['staff_ids'], b['onset_ids'], b['dur_ids'])


def run_train_epoch_v8(model, examples, batch_size, optimizer, loss_fn, device, rng):
    model.train()
    shuffled = list(examples); rng.shuffle(shuffled)
    loss_sum = top1 = top5 = n = 0
    for i in range(0, len(shuffled), batch_size):
        b    = collate_v8(shuffled[i:i+batch_size], device)
        bs   = b['token_ids'].size(0)
        optimizer.zero_grad()
        lg   = _fwd_v8(model, b)
        loss = loss_fn(lg, b['target_classes'])
        loss.backward(); optimizer.step()
        tgt       = b['target_classes']
        loss_sum += loss.item() * bs
        top1     += lg.topk(1, dim=-1)[1].squeeze(1).eq(tgt).sum().item()
        top5     += lg.topk(5, dim=-1)[1].eq(tgt.unsqueeze(1)).any(1).sum().item()
        n        += bs
    return loss_sum / n, top1 / n, top5 / n


@torch.no_grad()
def evaluate_dataset_v8(model, examples, batch_size, loss_fn, device):
    model.eval()
    loss_sum = top1 = top5 = n = 0; all_preds = []
    for i in range(0, len(examples), batch_size):
        b    = collate_v8(examples[i:i+batch_size], device)
        bs   = b['token_ids'].size(0)
        lg   = _fwd_v8(model, b)
        loss = loss_fn(lg, b['target_classes'])
        tgt  = b['target_classes']
        loss_sum += loss.item() * bs
        top1     += lg.topk(1, dim=-1)[1].squeeze(1).eq(tgt).sum().item()
        top5     += lg.topk(5, dim=-1)[1].eq(tgt.unsqueeze(1)).any(1).sum().item()
        n        += bs
        all_preds.extend(lg.argmax(dim=-1).tolist())
    return loss_sum / n, top1 / n, top5 / n, all_preds


# ── run_experiment_v8 ─────────────────────────────────────────────────────────

def run_experiment_v8(cfg, train_ids, dev_ids, device):
    output_dir = cfg['output_dir']
    if output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(f"Output dir not empty: {output_dir}\nDelete or rename.")
    output_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(cfg['seed']); random.seed(cfg['seed'])
    rng = random.Random(cfg['seed'])
    start_wall = time.time()

    # Vocab + pitch classes
    use_mask  = cfg['use_mask']
    v_base    = build_vocab_from_train(DATASET_ROOT, train_ids)
    vocab_exp = add_mask_to_vocab(v_base) if use_mask else v_base
    pc_exp    = build_pitch_classes(train_ids, DATASET_ROOT)
    c2p       = {v: k for k, v in pc_exp.items()}
    (output_dir / 'vocab.json').write_text(json.dumps(vocab_exp, indent=2))
    (output_dir / 'pitch_classes.json').write_text(json.dumps(pc_exp, indent=2))

    # Struct caches
    train_struct = _load_or_build_struct_cache(
        cfg['train_jsonl'],
        _STRUCT_CACHE_DIR / 'struct_cache_train.json', 'train')
    dev_struct = _load_or_build_struct_cache(
        DEV_JSONL, _STRUCT_CACHE_DIR / 'struct_cache_dev.json', 'dev')

    # Voice conditioning
    use_voice = cfg.get('voice_conditioning', True)
    voice_dim = cfg.get('voice_dim', 32)

    # Load train (no voice_vocab yet)
    tr, n_unk_tr = _load_v8_examples(
        cfg['train_jsonl'], vocab_exp, pc_exp, set(train_ids),
        'train', mask=use_mask, voice_vocab=None, struct_cache=train_struct)
    if use_voice:
        voice_vocab = build_voice_vocab(tr)
        n_voices    = len(voice_vocab)
        for e in tr: e.voice_id = voice_vocab.get(e.voice_label, 0)
        (output_dir / 'voice_vocab.json').write_text(json.dumps(voice_vocab, indent=2))
    else:
        voice_vocab = None; n_voices = 0

    dv, n_unk_dv = _load_v8_examples(
        DEV_JSONL, vocab_exp, pc_exp, set(dev_ids),
        'dev', mask=use_mask, voice_vocab=voice_vocab, struct_cache=dev_struct)

    # Unknown struct feature counts
    n_tr_unk = sum(1 for e in tr if e.staff_id==0 or e.onset_id==0 or e.dur_id==0)
    n_dv_unk = sum(1 for e in dv if e.staff_id==0 or e.onset_id==0 or e.dur_id==0)
    print(f"Train {len(tr)} (tok-unk={n_unk_tr}, struct-unk={n_tr_unk})"
          f"  Dev {len(dv)} (struct-unk={n_dv_unk})")

    # Model
    struct_dim = cfg.get('struct_dim', 16)
    mcfg = {
        'vocab_size':        len(vocab_exp),
        'num_pitch_classes': len(pc_exp),
        'd_model':           cfg['d_model'],
        'nhead':             N_HEADS,
        'num_layers':        cfg['num_layers'],
        'dim_feedforward':   cfg['dim_ff'],
        'dropout':           DROPOUT_EXPERIMENT,
        'max_len':           MAX_LEN,
        'pad_id':            vocab_exp[PAD_TOKEN],
        'voice_conditioning':use_voice,
        'n_voices':          n_voices,
        'voice_dim':         voice_dim,
        'struct_dim':        struct_dim,
    }
    (output_dir / 'model_config.json').write_text(json.dumps(mcfg, indent=2))
    model     = CorrectionModelV8(**mcfg).to(device)
    n_params  = sum(p.numel() for p in model.parameters())
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                  weight_decay=cfg['weight_decay'])
    print(f"Params: {n_params:,}  struct_dim={struct_dim}")

    # Epoch 0 baseline
    tl0, tt1_0, tt5_0, _ = evaluate_dataset_v8(model, tr, cfg['batch_size'], loss_fn, device)
    dl0, dt1_0, dt5_0, _ = evaluate_dataset_v8(model, dv, cfg['batch_size'], loss_fn, device)
    print(f"epoch 0: train {tl0:.4f}/{tt1_0:.2%}  dev {dl0:.4f}/{dt1_0:.2%}")

    metrics = [{'epoch':0,'train_loss':round(tl0,5),'train_top1':round(tt1_0,4),
                'train_top5':round(tt5_0,4),'dev_loss':round(dl0,5),
                'dev_top1':round(dt1_0,4),'dev_top5':round(dt5_0,4),
                'elapsed_s':round(time.time()-start_wall,1)}]

    best_loss=dl0; best_top1=dt1_0; best_epoch=0; patience_count=0

    for epoch in range(1, cfg['max_epochs']+1):
        if time.time() - start_wall >= cfg['time_limit']:
            print(f"Time limit ({cfg['time_limit']}s) reached before epoch {epoch}. "
                  f"Saving progress and stopping."); break
        es = time.time()
        tr_l, tr_t1, tr_t5 = run_train_epoch_v8(
            model, tr, cfg['batch_size'], optimizer, loss_fn, device, rng)
        d_l, d_t1, d_t5, _ = evaluate_dataset_v8(
            model, dv, cfg['batch_size'], loss_fn, device)
        elapsed = time.time() - start_wall
        metrics.append({'epoch':epoch,'train_loss':round(tr_l,5),
                        'train_top1':round(tr_t1,4),'train_top5':round(tr_t5,4),
                        'dev_loss':round(d_l,5),'dev_top1':round(d_t1,4),
                        'dev_top5':round(d_t5,4),'elapsed_s':round(elapsed,1)})
        print(f"  epoch {epoch:2d}  train {tr_l:.4f}/{tr_t1:.2%}  "
              f"dev {d_l:.4f}/{d_t1:.2%}  [{time.time()-es:.0f}s]")

        save_ckpt = (d_t1 > best_top1) or (d_t1 == best_top1 and d_l < best_loss)
        ptn_reset = d_t1 > best_top1
        if save_ckpt:
            best_loss=d_l; best_top1=d_t1; best_epoch=epoch
            torch.save({'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'epoch': epoch, 'dev_loss': d_l, 'dev_top1': d_t1},
                       output_dir / 'best_checkpoint.pt')
            print(f"    ↑ best")
        if ptn_reset:
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg['patience']:
                print("  Early stopping."); break

    (output_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    fields = ['epoch','train_loss','train_top1','train_top5','dev_loss',
              'dev_top1','dev_top5','elapsed_s']
    with open(output_dir / 'metrics.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(metrics)

    # Reload best checkpoint
    print(f"\nReloading best checkpoint (epoch {best_epoch})...")
    ckpt = torch.load(output_dir / 'best_checkpoint.pt',
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    d_l_v, d_t1_v, d_t5_v, best_preds = evaluate_dataset_v8(
        model, dv, cfg['batch_size'], loss_fn, device)
    best_row = next(m for m in metrics if m['epoch'] == best_epoch)
    assert abs(d_t1_v - best_row['dev_top1']) < 1e-3
    n_correct = sum(1 for e, p in zip(dv, best_preds) if e.target_class == p)
    print(f"Reload PASSED  dev top-1={n_correct}/{len(dv)} = {n_correct/len(dv):.2%}")

    logits_dev  = torch.cat([
        _fwd_v8(model, collate_v8(dv[i:i+cfg['batch_size']], device)).cpu()
        for i in range(0, len(dv), cfg['batch_size'])])
    predictions = build_predictions(dv, best_preds, c2p, logits_dev)
    (output_dir / 'dev_predictions.json').write_text(json.dumps(predictions, indent=2))

    # Unknown struct count in dev
    dv_struct_unk = {
        'staff': sum(1 for e in dv if e.staff_id==0),
        'onset': sum(1 for e in dv if e.onset_id==0),
        'dur':   sum(1 for e in dv if e.dur_id==0),
    }
    (output_dir / 'struct_unknowns.json').write_text(json.dumps(dv_struct_unk, indent=2))

    total = time.time() - start_wall
    print(f"\n{'='*60}")
    print(f" DONE: {n_correct}/{len(dv)} = {n_correct/len(dv):.2%} dev top-1")
    print(f" Best epoch: {best_epoch}  Total: {total:.0f}s")
    print(f" Dev struct unknowns: {dv_struct_unk}")
    print(f"{'='*60}")
    return {'metrics': metrics, 'n_correct': n_correct, 'best_epoch': best_epoch,
            'dev_top1': d_t1_v, 'dev_top5': d_t5_v, 'n_params': n_params,
            'elapsed_s': total, 'struct_unknowns': dv_struct_unk}

print("v8 helpers loaded: CorrectionExampleV8, CorrectionModelV8, run_experiment_v8")


In [ ]:
# ⚙️ v8 experiment config — v5 settings + structural features
EXP_V8 = {
    'name':              'v8_struct_features',
    'train_jsonl':       TRAIN_10K,
    'use_mask':          True,
    'voice_conditioning':True,
    'voice_dim':         32,
    'd_model':           128,
    'num_layers':        3,
    'dim_ff':            512,
    'struct_dim':        16,
    'checkpoint_by':    'top1',
    'seed':              42,
    'batch_size':        32,
    'lr':                1e-3,
    'weight_decay':      0.01,
    'max_epochs':        15,
    'patience':          5,
    'time_limit':        600,   # 10-minute hard cap
    'device':            None,  # auto-detect
}
EXP_V8['output_dir'] = RUNS_DIR / EXP_V8['name']
print(f"v8 output: {EXP_V8['output_dir']}")


In [ ]:
# ⚠️ TRAINS — v8 structural features
# Requires: Section 1 (cells 11-12) + v8_prep + v8_config cells.

_dev_v8 = (torch.device(EXP_V8['device']) if EXP_V8['device'] else
           torch.device('cuda' if torch.cuda.is_available() else
                        'mps'  if torch.backends.mps.is_available() else 'cpu'))
result_v8 = run_experiment_v8(EXP_V8, train_ids, dev_ids, _dev_v8)


In [ ]:
# 📊 v5 vs v8 comparison — reads from saved artifacts only.
# Requires: v8_train completed.

import json
from pathlib import Path

V5_DIR = RUNS_DIR / 'v5_voice_conditioned'
V8_DIR = RUNS_DIR / 'v8_struct_features'

def _load_run_data(run_dir):
    rd = Path(run_dir)
    if not rd.exists():
        return None
    preds   = json.load(open(rd / 'dev_predictions.json'))
    metrics = json.load(open(rd / 'metrics.json'))
    n_ex    = len(preds)
    n_t1    = sum(1 for p in preds if p['correct'])
    n_t5    = sum(1 for p in preds if p['target_token'] in p['top5_candidates'])
    best    = max((m for m in metrics if m['epoch'] > 0), key=lambda m: m['dev_top1'])
    total_t = metrics[-1]['elapsed_s']
    return {'n_t1': n_t1, 'n_t5': n_t5, 'n_ex': n_ex,
            'best_ep': best['epoch'], 'elapsed_s': total_t}

v5r = _load_run_data(V5_DIR)
v8r = _load_run_data(V8_DIR)

# Param counts from model configs (use correct class per run)
import torch.nn as nn
def _count_params_v5(cfg):
    skip = {'mask_token_id','mask_error','checkpoint_by'}
    return CorrectionModel(**{k:v for k,v in cfg.items() if k not in skip})
def _count_params_v8(cfg):
    skip = {'mask_token_id','mask_error','checkpoint_by'}
    return CorrectionModelV8(**{k:v for k,v in cfg.items() if k not in skip})

try:
    v5_params = sum(p.numel() for p in _count_params_v5(
        json.load(open(V5_DIR / 'model_config.json'))).parameters())
except Exception as e:
    v5_params = 659924  # fallback: computed earlier
try:
    v8_params = sum(p.numel() for p in _count_params_v8(
        json.load(open(V8_DIR / 'model_config.json'))).parameters())
except Exception as e:
    v8_params = None

# Struct unknowns
unk_f = V8_DIR / 'struct_unknowns.json'
struct_unk = json.load(open(unk_f)) if unk_f.exists() else {}

print("v5 vs v8 — dev accuracy (499 examples, same split)")
print(f"{'Metric':<28} {'v5 (MASK+voice)':<22} {'v8 (+staff/onset/dur)':<22}")
print('-' * 72)

rows = [
    ('dev top-1  (n/499)',   lambda r: f"{r['n_t1']}/499" if r else 'N/A'),
    ('dev top-1  (%)',       lambda r: f"{r['n_t1']/r['n_ex']:.2%}" if r else 'N/A'),
    ('dev top-5  (n/499)',   lambda r: f"{r['n_t5']}/499" if r else 'N/A'),
    ('dev top-5  (%)',       lambda r: f"{r['n_t5']/r['n_ex']:.2%}" if r else 'N/A'),
    ('best epoch',           lambda r: str(r['best_ep']) if r else 'N/A'),
    ('train time (s)',       lambda r: f"{r['elapsed_s']:.0f}" if r else 'N/A'),
    ('params',               None),
]

for label, fn in rows:
    if fn is None:
        v5v = f"{v5_params:,}" if v5_params else 'N/A'
        v8v = f"{v8_params:,}" if v8_params else 'N/A'
    else:
        v5v = fn(v5r)
        v8v = fn(v8r)
    print(f"  {label:<26} {v5v:<22} {v8v:<22}")

print()
print(f"v8 dev struct unknowns (bin=0): "
      f"staff={struct_unk.get('staff',0)}  "
      f"onset={struct_unk.get('onset',0)}  "
      f"dur={struct_unk.get('dur',0)}  (out of 499)")
print()
if v5r and v8r:
    delta = v8r['n_t1'] - v5r['n_t1']
    sign  = '+' if delta >= 0 else ''
    print(f"Δ dev top-1: {sign}{delta} examples  ({sign}{delta/v5r['n_ex']*100:.2f}pp)")
    print()
print("Interpretation:")
print("  v5 = historical checkpoint (seed=42, MASK+voice, 15 epochs, best ep 10).")
print("  v8 = this session's run (same seed/optimizer/budget, +structural features).")
print("  Adding staff/onset/duration did NOT improve over v5 in this single run.")
print("  No conclusion on root cause of low accuracy can be drawn from this alone.")


## 3g. v5 Train-Sample vs Dev Inference

Load v5 best checkpoint, evaluate 1,000 randomly-sampled train examples (seed=42) and compare against the full dev set (499). No weight updates. Dev top-1 / top-5 reused from saved `dev_predictions.json`; dev CE reused from `metrics.json` epoch 10.

In [ ]:
# 📊 v5 train-sample vs dev inference — no weight updates.
# Requires: cells 1–12 executed (train_examples, CorrectionModel, DEVICE in scope).

import random, torch, torch.nn as nn

V5_DIR = RUNS_DIR / 'v5_voice_conditioned'
_BATCH = 256   # inference-only

# ─── 1. Sample 1,000 train indices (seed=42) ────────────────────────────────
rng          = random.Random(42)
_sample_idx  = sorted(rng.sample(range(len(train_examples)), 1_000))
_sample_sids = [train_examples[i].sample_id for i in _sample_idx]
(V5_DIR / 'train_eval_sample_ids.json').write_text(json.dumps(_sample_sids, indent=1))
print(f'Saved {len(_sample_idx)} sample IDs -> train_eval_sample_ids.json')

# ─── 2. Build masked copies with v5 voice vocab ─────────────────────────────
_v5_cfg  = json.load(open(V5_DIR / 'model_config.json'))
_v5_vv   = json.load(open(V5_DIR / 'voice_vocab.json'))
_mask_id = _v5_cfg['mask_token_id']   # 183

def _masked_copy(e, vv, mid):
    ids = list(e.token_ids)
    ids[e.window_error_pos] = mid
    return CorrectionExample(
        sample_id=e.sample_id, token_ids=ids, padding_mask=e.padding_mask,
        window_error_pos=e.window_error_pos, target_class=e.target_class,
        window_lo=e.window_lo, target_token=e.target_token,
        replacement_token=e.replacement_token, voice_label=e.voice_label,
        voice_id=vv.get(e.voice_label, 0),
    )

train_1k = [_masked_copy(train_examples[i], _v5_vv, _mask_id) for i in _sample_idx]
assert all(e.token_ids[e.window_error_pos] == _mask_id for e in train_1k)

# ─── 3. Load v5 best checkpoint and move to DEVICE ──────────────────────────
_skip  = {'mask_token_id', 'mask_error', 'checkpoint_by'}
_model = CorrectionModel(**{k: v for k, v in _v5_cfg.items() if k not in _skip})
_ckpt  = torch.load(V5_DIR / 'best_checkpoint.pt', map_location='cpu', weights_only=False)
_model.load_state_dict(_ckpt['model_state_dict'])
_model.to(DEVICE)
_model.eval()
print(f'v5 checkpoint epoch {_ckpt["epoch"]} loaded -> {DEVICE}')

# ─── 4. Evaluate 1K train sample ────────────────────────────────────────────
_loss_fn = nn.CrossEntropyLoss()
with torch.inference_mode():
    tr_loss, tr_t1f, tr_t5f, _ = evaluate_dataset(_model, train_1k, _BATCH, _loss_fn, DEVICE)
tr_t1 = round(tr_t1f * 1000)
tr_t5 = round(tr_t5f * 1000)

# ─── 5. Reuse saved dev metrics ─────────────────────────────────────────────
_dev_preds = json.load(open(V5_DIR / 'dev_predictions.json'))
dv_n  = len(_dev_preds)
dv_t1 = sum(1 for p in _dev_preds if p['correct'])
dv_t5 = sum(1 for p in _dev_preds if p['target_token'] in p['top5_candidates'])
_met  = json.load(open(V5_DIR / 'metrics.json'))
_bm   = max(_met, key=lambda m: m['dev_top1'])
dv_loss = _bm['dev_loss']
best_ep = _bm['epoch']

# ─── 6. Comparison table ─────────────────────────────────────────────────────
print()
print(f'v5 best checkpoint (epoch {best_ep}) | MASK + voice conditioning')
print(f'{"Metric":<24} {"Train 1K  (fresh)":<26} {"Dev 499  (saved)"}')
print('─' * 68)
for lbl, tv, dv in [
    ('top-1  (n)',  f'{tr_t1}/1000',  f'{dv_t1}/499'),
    ('top-1  (%)',  f'{tr_t1f:.2%}',  f'{dv_t1/dv_n:.2%}'),
    ('top-5  (n)',  f'{tr_t5}/1000',  f'{dv_t5}/499'),
    ('top-5  (%)',  f'{tr_t5f:.2%}',  f'{dv_t5/dv_n:.2%}'),
    ('avg CE loss', f'{tr_loss:.4f}', f'{dv_loss:.4f} *'),
]:
    print(f'  {lbl:<22} {tv:<26} {dv}')
print()
print(f'* Dev CE from metrics.json epoch {best_ep} (model.eval(), same conditions as training eval).')

# ─── 7. Gap + training-curve context ─────────────────────────────────────────
delta_t1 = tr_t1f - dv_t1 / dv_n
delta_ce = dv_loss - tr_loss
print()
print(f'Delta top-1 (train-dev): {delta_t1:+.2%}  |  Delta CE (dev-train): {delta_ce:+.4f}')
print()

_ep10 = next(m for m in _met if m['epoch'] == best_ep)
_last = _met[-1]
print('Training curve reference (v5 metrics.json, online train top-1):')
print(f'  ep {best_ep} (checkpoint): train_top1_online={_ep10["train_top1"]:.4f}  '
      f'train_top1_offline={tr_t1f:.4f}  dev_top1={_ep10["dev_top1"]:.4f}')
print(f'  ep {_last["epoch"]} (final):      train_top1_online={_last["train_top1"]:.4f}  '
      f'dev_top1={_last["dev_top1"]:.4f}')
print()
print('Observations:')
print(f'  • Online train top-1 at ep {best_ep} was {_ep10["train_top1"]:.2%}; '
      f'offline eval of the same checkpoint gives {tr_t1f:.2%}.')
print(f'    Online metrics are pre-gradient-step per batch, so they underestimate '
      f'the final weights.')
print(f'  • Train-dev gap is {delta_t1:+.2%} top-1, {delta_ce:+.4f} CE.')
print(f'    Mild overfitting: model fits seen examples slightly better than unseen.')
print(f'    But dev top-1 ({dv_t1/dv_n:.2%}) is not catastrophically below train ({tr_t1f:.2%}),')
print(f'    suggesting capacity/task difficulty also limits performance.')
print()
print('Next hypotheses worth testing from existing artifacts (no new inference needed):')
print('  1. Per-class accuracy: compute from dev_predictions.json.')
print('     Are certain pitch classes predicted much more reliably? '
      '(Reveals class-level ceiling.)')
print('  2. Truncated vs full-window accuracy: 232/499 dev examples are truncated.')
print('     Compute accuracy split by (error_position > MAX_LEN//2). '
      '(Tests window-coverage hypothesis.)')
print('  3. Training CE curves: metrics.json has dev_loss per epoch.')
print(f'     Dev CE at ep {best_ep}: {dv_loss:.4f}. Plot to confirm at which epoch '
      'dev CE starts rising while train CE keeps falling.')


## 4. Error Analysis 📊

Analyse prediction errors for a saved run: octave accuracy, pitch-class accuracy,
semitone distance distribution.


In [ ]:
# 📊 Error analysis for a saved run
ANALYSIS_RUN = "v4_mask_top1"  # change to any key in RUNS

preds_file = RUNS[ANALYSIS_RUN] / "dev_predictions.json"
if not preds_file.exists():
    print(f"No predictions file for {ANALYSIS_RUN}")
else:
    preds = json.loads(preds_file.read_text())

    # Pitch → (octave, pitch_class) helpers
    NOTE_ORDER = ["C","D","E","F","G","A","B"]
    SEMITONES  = {"C":0,"D":2,"E":4,"F":5,"G":7,"A":9,"B":11}

    def pitch_to_midi(tok):
        """Approximate MIDI pitch (no accidentals; A4=69 reference)."""
        note, octave = tok[0], int(tok[1])
        return SEMITONES[note] + (octave + 1) * 12

    wrong = [p for p in preds if not p["correct"]]
    n_total = len(preds)
    n_wrong = len(wrong)

    # Octave accuracy
    oct_correct = sum(1 for p in wrong
                      if p["target_token"][1] == p["predicted_token"][1])
    # Pitch-class accuracy (correct note letter, wrong octave)
    pc_correct  = sum(1 for p in wrong
                      if p["target_token"][0] == p["predicted_token"][0])
    # Semitone distances
    semi_dist   = [abs(pitch_to_midi(p["target_token"]) - pitch_to_midi(p["predicted_token"]))
                   for p in wrong]

    print(f"Error analysis: {ANALYSIS_RUN}")
    print(f"  {n_total - n_wrong}/{n_total} correct  ({(n_total-n_wrong)/n_total:.2%})")
    print(f"\nAmong {n_wrong} wrong predictions:")
    print(f"  Correct octave, wrong note  : {oct_correct}/{n_wrong} = {oct_correct/n_wrong:.2%}")
    print(f"  Correct note letter, wrong oct: {pc_correct}/{n_wrong} = {pc_correct/n_wrong:.2%}")
    print(f"  Mean semitone distance : {sum(semi_dist)/len(semi_dist):.2f}")
    print(f"  Median semitone distance: {sorted(semi_dist)[len(semi_dist)//2]}")

    # Distribution
    from collections import Counter
    dist_counter = Counter(semi_dist)
    print(f"\n  Semitone distance distribution (top 10):")
    for d, cnt in sorted(dist_counter.items())[:10]:
        bar = "█" * (cnt // 2 + 1)
        print(f"    {d:3d} semitones : {cnt:3d}  {bar}")


## 5. Data Preparation ⚠️ WRITES

Generate JSONL correction datasets from OLiMPiC LMX files.

**The datasets already exist** — only run the build cell to regenerate or create new splits.

Existing files:
- `correction_train_10k.jsonl` — 10 000 synthetic errors from train split
- `correction_dev_v1.jsonl` — 499 examples from dev split
- `correction_smoke_train.jsonl` — 16 examples for sanity check


In [ ]:
# Dataset building functions (from build_correction_dataset.py)

def collect_pitch_vocab_for_build(dataset_root, split_ids):
    """Collect all pitch tokens from split_ids LMX files. Returns sorted list."""
    pitches = set()
    for sid in split_ids:
        p = dataset_root / f"{sid}.lmx"
        if p.exists():
            pitches.update(t for t in p.read_text().split() if PITCH_RE.match(t))
    return sorted(pitches)


def build_correction_dataset(dataset_root, split_ids, pitch_vocab,
                              limit, seed, output_path, overwrite=False):
    """
    ⚠️ WRITES — Generate JSONL of synthetic pitch-replacement errors.
    For each sampled score, one pitch token is replaced by a different pitch
    drawn uniformly from pitch_vocab.
    """
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"{output_path} already exists. Pass overwrite=True.")
    import random as _r
    rng   = _r.Random(seed)
    rows  = []
    skips = 0
    pids  = [s for s in split_ids if (dataset_root / f"{s}.lmx").exists()]
    rng.shuffle(pids)
    for sid in pids:
        if len(rows) >= limit: break
        tokens     = (dataset_root / f"{sid}.lmx").read_text().split()
        pitch_pos  = [i for i, t in enumerate(tokens) if PITCH_RE.match(t)]
        if not pitch_pos: skips += 1; continue
        error_pos  = rng.choice(pitch_pos)
        target_tok = tokens[error_pos]
        candidates = [p for p in pitch_vocab if p != target_tok]
        if not candidates: skips += 1; continue
        replacement = rng.choice(candidates)
        corrupted   = list(tokens); corrupted[error_pos] = replacement
        rows.append({
            "sample_id":         sid,
            "input_tokens":      corrupted,
            "error_position":    error_pos,
            "target_token":      target_tok,
            "replacement_token": replacement,
        })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        for row in rows: f.write(json.dumps(row) + "\n")
    print(f"Wrote {len(rows)} examples → {output_path}  (skipped {skips})")
    return rows


In [ ]:
# ⚠️ WRITES FILES — only run to regenerate datasets.
# The files below already exist. Uncomment to rebuild.

print(f"DEV_JSONL  exists={DEV_JSONL.exists()}  ({DEV_JSONL})")
print(f"TRAIN_10K  exists={TRAIN_10K.exists()}  ({TRAIN_10K})")
print(f"TRAIN_3K   exists={TRAIN_3K.exists()}   ({TRAIN_3K})")

# # Example: rebuild 10K train set
# pitch_vocab = collect_pitch_vocab_for_build(DATASET_ROOT, train_ids)
# build_correction_dataset(
#     DATASET_ROOT, train_ids, pitch_vocab,
#     limit=10_000, seed=42,
#     output_path=TRAIN_10K,
#     overwrite=False,  # set True to overwrite existing file
# )


---
# Part B: M2BERT Fine-Tuning, Duration Correction & Joint Model

**Source:** `m2bert_finetune_comparison.ipynb` (37 cells, pre-cleanup snapshot)

Covers: M2BERT backbone loading, pitch fine-tuning (epochs 1–20), pitch demo
with MusicXML/MIDI export, duration data construction and training, joint
pitch+duration model training, unified demo, and standalone inference interface.

**Joint model results (best checkpoint, epoch 2):**
- Pitch top-1: 310/499 (62.12%)
- Duration top-1: 438/499 (87.78%)

# M2BERT Fine-tuning for Pitch Correction

Compare pretrained **M2BERT** (ModernBERT-base backbone, CP compound-token representation)
against the custom Transformer **v5** on the OLiMPiC pitch-correction task.

**Setup**: freeze all but last 2 transformer layers (10, 11) + new 52-class head.  
**Budget**: ≤ 3 epochs **or** ≤ 10 minutes.  
**Baseline**: v5 dev top-1 = 83/499 (16.63%), top-5 = 261/499 (52.30%).


In [ ]:
import sys, json, re, random, time, pickle
from pathlib import Path
from fractions import Fraction as _Frac
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ModernBertConfig, ModernBertModel


In [ ]:
ML_DIR       = Path('/Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml')
PROCESSED    = ML_DIR / 'data' / 'processed'
PRETRAINED   = ML_DIR / 'data' / 'pretrained' / 'm2bert'
M2_RUN_DIR   = ML_DIR / 'data' / 'runs' / 'm2bert_finetune'
DATASET_ROOT = ML_DIR / 'dataset' / 'olimpic-1.0-synthetic'
M2_RUN_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cuda') if torch.cuda.is_available() else
          torch.device('cpu'))
print(f'DEVICE: {DEVICE}')

# ── CP vocab (from M2BERT pretrained) ──────────────────────────
with open(PRETRAINED / 'CP.pkl', 'rb') as f:
    _cp = pickle.load(f)

# _cp[0] = {attr_name: {token_str: id}}
# _cp[1] = {attr_idx: {id: token_str}}
CP_TOK2ID = _cp[0]
print('CP attribute sizes:', {k: len(v) for k, v in CP_TOK2ID.items()})

BAR_PAD    = CP_TOK2ID['Bar']['Bar <PAD>']           # 2
BAR_MASK   = CP_TOK2ID['Bar']['Bar <MASK>']          # 3
POS_PAD    = CP_TOK2ID['Position']['Position <PAD>'] # 16
PITCH_PAD  = CP_TOK2ID['Pitch']['Pitch <PAD>']       # 86
PITCH_MASK = CP_TOK2ID['Pitch']['Pitch <MASK>']      # 87
DUR_PAD    = CP_TOK2ID['Duration']['Duration <PAD>'] # 64

CP_PAD = [BAR_PAD, POS_PAD, PITCH_PAD, DUR_PAD]
print(f'CP_PAD: {CP_PAD}   PITCH_MASK: {PITCH_MASK}')


In [ ]:
PITCH_RE = re.compile(r'^[A-G][0-9]$')

def _lmx_path(sid):
    parts = sid.split('/')
    return DATASET_ROOT / 'samples' / parts[1] / (parts[2] + '.lmx')

# 52-class pitch mapping — identical to main notebook's build_pitch_classes()
pitches = set()
for sid in (DATASET_ROOT / 'samples.train.txt').read_text().split():
    p = _lmx_path(sid)
    if p.exists():
        pitches.update(t for t in p.read_text().split() if PITCH_RE.match(t))
pitch_classes = {p: i for i, p in enumerate(sorted(pitches))}
id2pitch      = {i: p for p, i in pitch_classes.items()}
N_CLASSES     = len(pitch_classes)
print(f'Pitch classes: {N_CLASSES}  (classes: {sorted(pitch_classes)[:5]} ... {sorted(pitch_classes)[-3:]})')

with open(PROCESSED / 'correction_train_10k.jsonl') as f:
    train_raw = [json.loads(l) for l in f]
with open(PROCESSED / 'correction_dev_v1.jsonl') as f:
    dev_raw = [json.loads(l) for l in f]
print(f'Train: {len(train_raw)}  Dev: {len(dev_raw)}')


## LMX → CP Conversion

Each example's full token sequence is parsed into per-note dicts by `parse_lmx_notes`.
For each note we compute four CP indices:

| Attribute | Formula |
|-----------|---------|
| Bar       | 0 (Bar New) if note starts a new measure vs previous note, else 1 (Bar Continue) |
| Position  | `min(15, int(onset_in_measure_whole_notes × 16))` |
| Pitch     | `MIDI - 22` (0–85); 87 = PITCH_MASK for error note; 86 = PITCH_PAD for rest |
| Duration  | `min(63, int(duration_whole_notes × 32))`; 64 = DUR_PAD if unknown |

The full note sequence (no windowing needed; max 216 notes across all examples) is used.


In [ ]:
# Note-Level LMX Parser — copied verbatim from correction_experiments.ipynb cell 41.
import re as _re

_DURATION_BASE = {
    'breve': _Frac(2),      'whole': _Frac(1),      'half': _Frac(1, 2),
    'quarter': _Frac(1, 4), 'eighth': _Frac(1, 8),  '16th': _Frac(1, 16),
    '32nd': _Frac(1, 32),   '64th': _Frac(1, 64),   '128th': _Frac(1, 128),
    '256th': _Frac(1, 256), '512th': _Frac(1, 512), '1024th': _Frac(1, 1024),
}
_DECORATION = frozenset({
    'beam:begin', 'beam:end', 'beam:forward-hook', 'beam:backward-hook',
    'tied:start', 'tied:stop', 'tuplet:start', 'tuplet:stop',
    'slur:start', 'slur:stop', 'trill-mark', 'staccato', 'accent',
    'strong-accent', 'tenuto', 'fermata', 'arpeggiate',
    'tremolo:1', 'tremolo:2', 'tremolo:3', 'tremolo:4',
    'tremolo:single', 'tremolo:start', 'tremolo:stop',
})
_ACCIDENTAL = frozenset({
    'sharp', 'flat', 'natural', 'double-sharp',
    'flat-flat', 'natural-sharp', 'natural-flat',
})
_SKIP = frozenset({'time'})
_TIME_MOD_RE = _re.compile(r'^(\d+)in(\d+)$')

def _parse_dur(tokens, i):
    if i >= len(tokens) or tokens[i] not in _DURATION_BASE:
        return None, i
    base = _DURATION_BASE[tokens[i]]
    i += 1
    m = _TIME_MOD_RE.match(tokens[i]) if i < len(tokens) else None
    if m:
        base *= _Frac(int(m.group(2)), int(m.group(1)))
        i += 1
    n_dots = 0
    while i < len(tokens) and tokens[i] == 'dot':
        n_dots += 1
        i += 1
    if n_dots:
        base *= (2 - _Frac(1, 2 ** n_dots))
    return base, i

def parse_lmx_notes(tokens, sample_id=''):
    notes, warnings = [], []
    current_voice = current_staff = None
    current_beats = current_beat_type = None
    measure_num = 0
    current_onset = _Frac(0)
    chord_group_id = 0
    chord_leader_onset = _Frac(0)
    i, n = 0, len(tokens)
    while i < n:
        tok = tokens[i]
        if tok == 'measure':
            measure_num += 1
            current_onset = _Frac(0)
            i += 1; continue
        if tok in _SKIP or tok.startswith('key:'):
            i += 1; continue
        if tok.startswith('beats:'):
            try: current_beats = int(tok[6:])
            except ValueError: warnings.append(f'bad beats {tok!r}')
            i += 1; continue
        if tok.startswith('beat-type:'):
            try: current_beat_type = int(tok[10:])
            except ValueError: warnings.append(f'bad beat-type {tok!r}')
            i += 1; continue
        if tok.startswith('clef:'):
            i += 1
            if i < n and tokens[i].startswith('staff:'): i += 1
            continue
        if tok in _DECORATION:
            i += 1; continue
        span_start = i
        is_grace = is_chord = False
        if tok == 'print-object:no':
            i += 1
            if i >= n: break
            tok = tokens[i]
        if tok == 'grace':
            is_grace = True; i += 1
            if i >= n: break
            tok = tokens[i]
            if tok == 'grace:slash':
                i += 1
                if i >= n: break
                tok = tokens[i]
        if tok == 'chord':
            is_chord = True; i += 1
            if i >= n: break
            tok = tokens[i]
        pitch = None; is_rest = False; pitch_token_position = None
        if PITCH_RE.match(tok):
            pitch, pitch_token_position = tok, i; i += 1
        elif tok == 'rest':
            is_rest = True; i += 1
        elif tok == 'backup':
            i += 1; dur, i = _parse_dur(tokens, i)
            if dur is not None: current_onset = max(_Frac(0), current_onset - dur)
            else: warnings.append(f'backup no dur @ {i}')
            continue
        elif tok == 'forward':
            i += 1; dur, i = _parse_dur(tokens, i)
            if dur is not None: current_onset += dur
            else: warnings.append(f'forward no dur @ {i}')
            continue
        else:
            warnings.append(f'unexpected {tok!r} @ {i}'); i += 1; continue
        if i < n and tokens[i].startswith('voice:'):
            current_voice = tokens[i]; i += 1
        duration = None
        if i < n:
            if tokens[i] in _DURATION_BASE:
                duration, i = _parse_dur(tokens, i)
            elif tokens[i] == 'rest:measure':
                if current_beats is not None and current_beat_type is not None:
                    duration = _Frac(current_beats, current_beat_type)
                else:
                    duration = None
                    warnings.append(f'rest:measure unknown dur @ {sample_id}')
                i += 1
        if i < n and tokens[i] in _ACCIDENTAL: i += 1
        if i < n and tokens[i].startswith('stem:'): i += 1
        if i < n and tokens[i].startswith('staff:'):
            current_staff = tokens[i]; i += 1
        while i < n and tokens[i] in _DECORATION: i += 1
        onset = chord_leader_onset if is_chord else current_onset
        if is_chord: grp = chord_group_id
        else:
            chord_group_id += 1; grp = chord_group_id
            chord_leader_onset = current_onset
        notes.append({'measure': measure_num, 'voice': current_voice, 'staff': current_staff,
                      'onset': onset, 'duration': duration, 'pitch': pitch,
                      'is_rest': is_rest, 'is_grace': is_grace,
                      'chord_group': grp, 'pitch_token_position': pitch_token_position,
                      'token_span': (span_start, i)})
        if not is_chord and not is_grace and duration is not None:
            current_onset += duration
    return notes, warnings

print('parse_lmx_notes defined')


In [ ]:
_SEMITONES = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}

def _pitch_to_cp_idx(tok):
    midi = (int(tok[1]) + 1) * 12 + _SEMITONES[tok[0]]
    idx  = midi - 22
    return idx if 0 <= idx <= 85 else None

def lmx_to_cp_example(ex, pitch_classes):
    """
    Convert one JSONL example to (cp_ids, error_note_idx, target_class).
    cp_ids: list[list[int]] of shape [n_notes, 4] = [bar, pos, pitch, dur].
    Error note pitch is replaced with PITCH_MASK (87).
    Returns None only if error note position not found (0 in practice).
    """
    target = ex['target_token']
    if target not in pitch_classes:
        return None

    notes, _ = parse_lmx_notes(ex['input_tokens'], ex.get('sample_id', ''))
    err_pos   = ex['error_position']
    err_idx   = next((i for i, n in enumerate(notes)
                      if n['pitch_token_position'] == err_pos), None)
    if err_idx is None:
        return None

    cp_ids    = []
    prev_meas = -1
    for i, note in enumerate(notes):
        bar   = 0 if note['measure'] != prev_meas else 1
        prev_meas = note['measure']
        pos   = min(15, int(float(note['onset']) * 16))
        if i == err_idx:
            pitch = PITCH_MASK
        elif note['pitch'] and PITCH_RE.match(note['pitch']):
            p = _pitch_to_cp_idx(note['pitch'])
            pitch = p if p is not None else PITCH_PAD
        else:
            pitch = PITCH_PAD   # rest
        if note['duration'] is not None:
            dur = min(63, int(float(note['duration']) * 32))
        else:
            dur = DUR_PAD
        cp_ids.append([bar, pos, pitch, dur])

    return cp_ids, err_idx, pitch_classes[target]


In [ ]:
def build_cp_examples(raw_list, pitch_classes, split_name):
    examples, skipped = [], 0
    for ex in raw_list:
        res = lmx_to_cp_example(ex, pitch_classes)
        if res is None:
            skipped += 1; continue
        cp_ids, err_idx, tgt = res
        examples.append({'cp_ids': cp_ids, 'error_note_idx': err_idx,
                         'target_class': tgt, 'sample_id': ex['sample_id']})
    lens = [len(e['cp_ids']) for e in examples]
    print(f'{split_name}: {len(examples)} ok, {skipped} skipped  '
          f'seq_len min={min(lens)} max={max(lens)} mean={sum(lens)/len(lens):.1f}')
    return examples

print('Building CP examples...')
t0 = time.time()
train_cp = build_cp_examples(train_raw, pitch_classes, 'train')
dev_cp   = build_cp_examples(dev_raw,   pitch_classes, 'dev')
print(f'Done in {time.time()-t0:.1f}s')


## M2BERT Model

- **Backbone** (`MidiModernBERT`): 4 × Embedding(vocab_size, 256) → Linear(1024, 768)
  → ModernBERT-base (12 layers, 768 hidden). Loaded from `model_best.ckpt` with `strict=True`.
- **Head**: Dropout(0.1) → Linear(768, 52). Takes hidden state at the masked note position.
- **Frozen**: all backbone except layers 10, 11 (last 2 transformer blocks).


In [ ]:
class LUT(nn.Module):
    """Lookup-table wrapper matching M2BERT checkpoint key `word_emb.N.lut.weight`."""
    def __init__(self, n, d, pad=None):
        super().__init__()
        self.lut = nn.Embedding(n, d, padding_idx=pad)
    def forward(self, x):
        return self.lut(x)


class MidiModernBERT(nn.Module):
    """M2BERT backbone: CP embeddings → ModernBERT-12L."""
    def __init__(self):
        super().__init__()
        self.word_emb = nn.ModuleList([
            LUT(4,  256, pad=BAR_PAD),    # Bar
            LUT(18, 256, pad=POS_PAD),    # Position
            LUT(88, 256, pad=PITCH_PAD),  # Pitch
            LUT(66, 256, pad=DUR_PAD),    # Duration
        ])
        self.in_linear = nn.Linear(1024, 768)
        _cfg = ModernBertConfig(
            vocab_size=10, hidden_size=768, num_hidden_layers=12,
            num_attention_heads=12, intermediate_size=1152,
            max_position_embeddings=4096, pad_token_id=0,
        )
        self.note_bert1 = ModernBertModel(_cfg)

    def forward(self, cp_ids, attention_mask=None):
        # cp_ids: [B, S, 4]
        embs = torch.cat([self.word_emb[i](cp_ids[:, :, i]) for i in range(4)], dim=-1)
        embs = self.in_linear(embs)           # [B, S, 768]
        out  = self.note_bert1(inputs_embeds=embs, attention_mask=attention_mask)
        return out.last_hidden_state          # [B, S, 768]


class M2BERTPitchCorrection(nn.Module):
    """M2BERT backbone + classification head for pitch correction."""
    def __init__(self, backbone, n_classes):
        super().__init__()
        self.backbone   = backbone
        self.classifier = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_classes))

    def forward(self, cp_ids, error_note_idx, attention_mask=None):
        hidden  = self.backbone(cp_ids, attention_mask)                   # [B, S, 768]
        err_h   = hidden[torch.arange(hidden.size(0)), error_note_idx]    # [B, 768]
        return self.classifier(err_h)                                     # [B, N_CLASSES]


In [ ]:
backbone = MidiModernBERT()
_ckpt    = torch.load(PRETRAINED / 'model_best.ckpt', map_location='cpu', weights_only=False)
backbone.load_state_dict(_ckpt['state_dict'], strict=True)
print(f'Backbone: strict=True load OK  '
      f'(pretrained epoch {_ckpt["epoch"]}, best_acc={_ckpt["best_acc"]:.4f})')

model = M2BERTPitchCorrection(backbone, N_CLASSES)
model.to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_p:,}  N_CLASSES: {N_CLASSES}')


In [ ]:
# Freeze all backbone parameters
for p in model.parameters():
    p.requires_grad_(False)

# Unfreeze last 2 transformer layers (10 & 11) and classifier head
for layer_idx in (10, 11):
    for p in model.backbone.note_bert1.layers[layer_idx].parameters():
        p.requires_grad_(True)
for p in model.classifier.parameters():
    p.requires_grad_(True)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frz   = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Trainable: {n_train:,}  Frozen: {n_frz:,}')

optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
loss_fn   = nn.CrossEntropyLoss()


In [ ]:
class CPDataset(Dataset):
    def __init__(self, examples): self.examples = examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

_pad_t = torch.tensor(CP_PAD, dtype=torch.long)

def collate_cp(batch):
    max_len    = max(len(e['cp_ids']) for e in batch)
    B          = len(batch)
    cp_ids_t   = _pad_t.unsqueeze(0).expand(max_len, -1).unsqueeze(0).expand(B, -1, -1).clone()
    attn_mask  = torch.zeros(B, max_len, dtype=torch.long)
    err_idx_t  = torch.zeros(B, dtype=torch.long)
    tgt_cls_t  = torch.zeros(B, dtype=torch.long)
    for i, e in enumerate(batch):
        L = len(e['cp_ids'])
        cp_ids_t[i, :L] = torch.tensor(e['cp_ids'], dtype=torch.long)
        attn_mask[i, :L] = 1
        err_idx_t[i]     = e['error_note_idx']
        tgt_cls_t[i]     = e['target_class']
    return cp_ids_t, err_idx_t, tgt_cls_t, attn_mask

# Benchmark single batch forward pass
_bmark_loader = DataLoader(CPDataset(train_cp[:64]), batch_size=64,
                            shuffle=False, collate_fn=collate_cp)
_cp, _ei, _tc, _am = next(iter(_bmark_loader))
model.eval()
with torch.no_grad():
    t0 = time.time()
    _ = model(_cp.to(DEVICE), _ei.to(DEVICE), _am.to(DEVICE))
    torch.mps.synchronize() if DEVICE.type == 'mps' else None
    print(f'Single batch (B=64): {(time.time()-t0)*1000:.1f}ms  '
          f'seq_len≈{_am.sum(1).float().mean().item():.0f}')
del _bmark_loader, _cp, _ei, _tc, _am


## Training ⚠️ TRAINS

Budget: ≤ 3 epochs **or** ≤ 10 minutes. Best checkpoint saved by dev top-1.


In [ ]:
# Continue M2BERT fine-tuning up to TOTAL epoch 6.
# Requires the previous model/data/optimizer setup cells.

import json
import time
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

MAX_EPOCHS = 6
MAX_SECONDS = 180
PATIENCE = 2
BATCH_SIZE = 32

# ---------- Restore the previous best ----------

source_dir = Path(M2_RUN_DIR)
source_path = source_dir / "best_checkpoint.pt"
ckpt = torch.load(source_path, map_location="cpu", weights_only=True)

history = list(ckpt.get("metrics", []))

if "epoch" in ckpt:
    start_epoch = int(ckpt["epoch"])
elif history:
    start_epoch = int(
        max(history, key=lambda r: r["dev_top1"])["epoch"]
    )
else:
    raise ValueError(
        "Checkpoint has no epoch information; identify its epoch first."
    )

if start_epoch >= MAX_EPOCHS:
    raise ValueError(
        f"Checkpoint is already at epoch {start_epoch}; "
        "no additional training needed."
    )

model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.to(DEVICE)

if "optimizer_state_dict" in ckpt:
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    print("Restored model and optimizer.")
else:
    optimizer.state.clear()
    print("Restored model; optimizer state initialized fresh.")

best_dev_top1 = float(ckpt["best_dev_top1"])
best_epoch = start_epoch
metrics = [
    dict(r) for r in history
    if int(r["epoch"]) <= start_epoch
]
del ckpt

# New directory: preserve all original artifacts.
run_dir = source_dir.parent / (
    source_dir.name
    + "_resume_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
run_dir.mkdir(parents=True, exist_ok=False)
best_path = run_dir / "best_checkpoint.pt"

train_loader = DataLoader(
    CPDataset(train_cp),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_cp,
    num_workers=0,
)

dev_loader = DataLoader(
    CPDataset(dev_cp),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_cp,
    num_workers=0,
)

# ---------- Evaluation and checkpoint helpers ----------

def eval_metrics(model, loader):
    model.eval()
    n, top1, top5, ce_sum = 0, 0, 0, 0.0

    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            cp_ids, err_idx, tgt, attn = [
                x.to(DEVICE)
                for x in (cp_ids, err_idx, tgt, attn)
            ]

            logits = model(cp_ids, err_idx, attn)

            top1 += (logits.argmax(1) == tgt).sum().item()

            candidates = logits.topk(
                min(5, logits.size(1)), dim=1
            ).indices
            top5 += (
                candidates == tgt[:, None]
            ).any(dim=1).sum().item()

            ce_sum += F.cross_entropy(
                logits, tgt, reduction="sum"
            ).item()
            n += tgt.size(0)

    return {
        "n": n,
        "correct_top1": top1,
        "correct_top5": top5,
        "top1": top1 / n,
        "top5": top5 / n,
        "ce": ce_sum / n,
    }


def save_best(epoch, epoch_complete=True):
    # Save immediately so later training cannot change this snapshot.
    torch.save(
        {
            "epoch": epoch,
            "epoch_complete": epoch_complete,
            "model_state_dict": {
                k: v.detach().cpu()
                for k, v in model.state_dict().items()
            },
            "optimizer_state_dict": optimizer.state_dict(),
            "best_dev_top1": best_dev_top1,
            "metrics": list(metrics),
            "n_classes": N_CLASSES,
            "pitch_classes": pitch_classes,
            "source_checkpoint": str(source_path),
        },
        best_path,
    )


# The old best remains the baseline, even if continuation gets worse.
save_best(start_epoch)

print(
    f"Starting at epoch {start_epoch}; "
    f"best dev top-1={best_dev_top1:.2%}"
)
print(f"Output: {run_dir}")

# ---------- Continue training ----------

bad_epochs = 0
budget_hit = False
t_start = time.monotonic()

for epoch in range(start_epoch + 1, MAX_EPOCHS + 1):
    if time.monotonic() - t_start >= MAX_SECONDS:
        print("Training budget reached.")
        break

    model.train()
    loss_sum, n_train = 0.0, 0

    for cp_ids, err_idx, tgt, attn in train_loader:
        if time.monotonic() - t_start >= MAX_SECONDS:
            budget_hit = True
            break

        cp_ids, err_idx, tgt, attn = [
            x.to(DEVICE)
            for x in (cp_ids, err_idx, tgt, attn)
        ]

        optimizer.zero_grad(set_to_none=True)
        logits = model(cp_ids, err_idx, attn)
        loss = loss_fn(logits, tgt)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * tgt.size(0)
        n_train += tgt.size(0)

    if n_train == 0:
        break

    epoch_complete = n_train == len(train_loader.dataset)
    dev = eval_metrics(model, dev_loader)
    improved = dev["top1"] > best_dev_top1

    row = {
        "epoch": epoch,
        "epoch_complete": epoch_complete,
        "train_examples_seen": n_train,
        "train_loss": loss_sum / n_train,
        "dev_top1": dev["top1"],
        "dev_top5": dev["top5"],
        "dev_ce": dev["ce"],
        "correct_top1": dev["correct_top1"],
        "correct_top5": dev["correct_top5"],
        "dev_n": dev["n"],
        "elapsed_s": time.monotonic() - t_start,
    }
    metrics.append(row)

    if improved:
        best_dev_top1 = dev["top1"]
        best_epoch = epoch
        bad_epochs = 0
        save_best(epoch, epoch_complete)
    else:
        bad_epochs += 1

    with open(run_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    status = "complete" if epoch_complete else "partial"
    print(
        f"Epoch {epoch} ({status}): "
        f"loss={row['train_loss']:.4f} "
        f"dev={dev['correct_top1']}/{dev['n']} "
        f"({dev['top1']:.2%}) "
        f"top-5={dev['top5']:.2%}"
        + ("  *best*" if improved else "")
    )

    if budget_hit or time.monotonic() - t_start >= MAX_SECONDS:
        print("Training budget reached; stopping.")
        break

    if bad_epochs >= PATIENCE:
        print(
            f"Early stopping: {PATIENCE} epochs "
            "without improvement."
        )
        break

# ---------- Restore and evaluate the selected best ----------

selected = torch.load(
    best_path, map_location="cpu", weights_only=True
)
model.load_state_dict(selected["model_state_dict"], strict=True)
optimizer.load_state_dict(selected["optimizer_state_dict"])
del selected
model.eval()

final = eval_metrics(model, dev_loader)

with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open(run_dir / "final_dev_metrics.json", "w") as f:
    json.dump({"best_epoch": best_epoch, **final}, f, indent=2)

# Following cells use this run; the original directory stays intact.
M2_RUN_DIR = run_dir

print(f"\nBest epoch: {best_epoch}")
print(
    f"Top-1: {final['correct_top1']}/{final['n']} "
    f"= {final['top1']:.2%}"
)
print(
    f"Top-5: {final['correct_top5']}/{final['n']} "
    f"= {final['top5']:.2%}"
)
print(f"Saved to: {M2_RUN_DIR}")

## Results — M2BERT vs v5 📊

Load best checkpoint, evaluate on all 499 dev examples, compare with v5.


In [ ]:
# 📊 Evaluate M2BERT on dev + compare with v5.
# Requires: best_checkpoint.pt saved by the TRAINS cell.

# Load best checkpoint
_saved = torch.load(M2_RUN_DIR / 'best_checkpoint.pt', map_location='cpu', weights_only=False)
model.load_state_dict(_saved['model_state_dict'])
model.to(DEVICE)
model.eval()
print(f'Loaded checkpoint: best_dev_top1={_saved["best_dev_top1"]:.4f}')

# Full dev evaluation with per-example predictions
all_preds = []
with torch.inference_mode():
    for cp_ids, err_idx, tgt, attn in dev_loader:
        logits   = model(cp_ids.to(DEVICE), err_idx.to(DEVICE), attn.to(DEVICE))
        probs    = logits.softmax(dim=-1).cpu()
        top5_ids = probs.topk(5, dim=-1).indices.tolist()
        for b in range(tgt.size(0)):
            pred_id  = top5_ids[b][0]
            tgt_id   = tgt[b].item()
            all_preds.append({
                'sample_id':      dev_cp[len(all_preds)]['sample_id'],
                'target_token':   id2pitch[tgt_id],
                'predicted_token': id2pitch[pred_id],
                'correct':        (pred_id == tgt_id),
                'top5_candidates': [id2pitch[x] for x in top5_ids[b]],
            })

with open(M2_RUN_DIR / 'dev_predictions.json', 'w') as f:
    json.dump(all_preds, f, indent=2)

m2_t1 = sum(1 for p in all_preds if p['correct'])
m2_t5 = sum(1 for p in all_preds if p['target_token'] in p['top5_candidates'])
n_dev = len(all_preds)
print(f'M2BERT  dev: top-1 = {m2_t1}/{n_dev} ({m2_t1/n_dev:.2%})  '
      f'top-5 = {m2_t5}/{n_dev} ({m2_t5/n_dev:.2%})')

# ── Load v5 for comparison ─────────────────────────────────────
V5_DIR     = ML_DIR / 'data' / 'runs' / 'v5_voice_conditioned'
_v5_preds  = json.load(open(V5_DIR / 'dev_predictions.json'))
v5_t1 = sum(1 for p in _v5_preds if p['correct'])
v5_t5 = sum(1 for p in _v5_preds if p['target_token'] in p['top5_candidates'])
print(f'v5      dev: top-1 = {v5_t1}/{n_dev} ({v5_t1/n_dev:.2%})  '
      f'top-5 = {v5_t5}/{n_dev} ({v5_t5/n_dev:.2%})')

# ── Venn: overlap ──────────────────────────────────────────────
m2_ids = {p['sample_id'] for p in all_preds if p['correct']}
v5_ids = {p['sample_id'] for p in _v5_preds if p['correct']}
both  = len(m2_ids & v5_ids)
only_m2 = len(m2_ids - v5_ids)
only_v5 = len(v5_ids - m2_ids)
print(f'\nVenn overlap  both={both}  only_M2BERT={only_m2}  only_v5={only_v5}')

# ── Summary table ─────────────────────────────────────────────
_met = json.load(open(M2_RUN_DIR / 'metrics.json'))
n_ep = len(_met)
partial = ' (partial)' if n_ep < 3 else ''
print(f'\n{"Model":<22} {"Top-1":>10} {"Top-5":>10} {"Epochs":>8}')
print('─' * 55)
print(f'{"v5 (voice-conditioned)":<22} '
      f'{v5_t1}/{n_dev} ({v5_t1/n_dev:.2%}) {" ":>4} '
      f'{v5_t5}/{n_dev} ({v5_t5/n_dev:.2%}) '
      f'{"10":>8}')
print(f'{"M2BERT fine-tuned":<22} '
      f'{m2_t1}/{n_dev} ({m2_t1/n_dev:.2%}) {" ":>4} '
      f'{m2_t5}/{n_dev} ({m2_t5/n_dev:.2%}) '
      f'{str(n_ep)+partial:>8}')


## Extended Training (epochs 5–20) ⚠️ TRAINS

Resume from epoch 4 best checkpoint (304/499 = 60.92% top-1, 417/499 = 83.57% top-5).

| Parameter | Value |
|-----------|-------|
| Source checkpoint | `m2bert_finetune_resume_20261028_101120_113800/best_checkpoint.pt` (epoch 4) |
| Resume from | epoch 5 |
| Max epochs | 20 |
| Early stopping | patience = 4 (no dev top-1 improvement) |
| Budget | 600 s |
| LR / batch / freeze | unchanged (1e-4, 32, layers 10–11 + head) |

**⚠️ Do not run the next cell until the machine is confirmed safe.**


In [ ]:
# ⚠️ TRAINS — Extended fine-tuning (epochs 5–20, patience=4, 600 s budget).
# Requires: all setup cells (1–12) executed first.
# Loads epoch-4 best checkpoint; saves to a new run directory.

from datetime import datetime
import torch.nn.functional as F

# ── Configuration ──────────────────────────────────────────────
RESUME_DIR   = ML_DIR / "data" / "runs" / "m2bert_finetune_resume_20261028_101120_113800"
START_EPOCH  = 4          # checkpoint epoch (training resumes at START_EPOCH + 1)
MAX_EPOCH    = 20
PATIENCE     = 4          # early stopping on dev top-1
MAX_SECONDS  = 600
BATCH_SIZE   = 32

# ── Load epoch-4 checkpoint ───────────────────────────────────
_ckpt = torch.load(RESUME_DIR / "best_checkpoint.pt", map_location="cpu", weights_only=False)
model.load_state_dict(_ckpt["model_state_dict"], strict=True)
model.to(DEVICE)
optimizer.load_state_dict(_ckpt["optimizer_state_dict"])
baseline_top1 = _ckpt["best_dev_top1"]          # 0.6092 = 304/499
print(f"Loaded epoch {START_EPOCH}: dev top-1 = {baseline_top1:.4f}")
del _ckpt

# ── Output directory (preserves original) ─────────────────────
EXT_RUN_DIR = ML_DIR / "data" / "runs" / (
    "m2bert_finetune_extended_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
EXT_RUN_DIR.mkdir(parents=True, exist_ok=False)
print(f"Output: {EXT_RUN_DIR}")

# ── DataLoaders ───────────────────────────────────────────────
train_loader = DataLoader(CPDataset(train_cp), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_cp, num_workers=0)
dev_loader   = DataLoader(CPDataset(dev_cp), batch_size=64,
                          shuffle=False, collate_fn=collate_cp, num_workers=0)

# ── Evaluation helper ─────────────────────────────────────────
def _eval_dev(model, loader, device):
    model.eval()
    n = top1 = top5 = 0; ce_sum = 0.0
    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            cp_ids, err_idx, tgt, attn = (
                cp_ids.to(device), err_idx.to(device),
                tgt.to(device), attn.to(device)
            )
            logits = model(cp_ids, err_idx, attn)
            top1 += (logits.argmax(1) == tgt).sum().item()
            top5 += (logits.topk(min(5, logits.size(1)), dim=1)
                      .indices == tgt[:, None]).any(1).sum().item()
            ce_sum += F.cross_entropy(logits, tgt, reduction="sum").item()
            n += tgt.size(0)
    return {"n": n, "top1": top1/n, "top5": top5/n, "ce": ce_sum/n,
            "correct_top1": top1, "correct_top5": top5}

# ── Training loop ─────────────────────────────────────────────
metrics      = []
best_top1    = baseline_top1
best_epoch   = START_EPOCH
best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
best_optim   = optimizer.state_dict()
bad_epochs   = 0
t_start      = time.monotonic()

for epoch in range(START_EPOCH + 1, MAX_EPOCH + 1):
    elapsed = time.monotonic() - t_start
    if elapsed >= MAX_SECONDS:
        print(f"Budget reached before epoch {epoch} ({elapsed:.0f}s)"); break

    model.train()
    loss_sum, n_seen = 0.0, 0
    for cp_ids, err_idx, tgt, attn in train_loader:
        if time.monotonic() - t_start >= MAX_SECONDS:
            break
        cp_ids, err_idx, tgt, attn = (
            cp_ids.to(DEVICE), err_idx.to(DEVICE),
            tgt.to(DEVICE), attn.to(DEVICE)
        )
        optimizer.zero_grad(set_to_none=True)
        logits = model(cp_ids, err_idx, attn)
        loss = loss_fn(logits, tgt)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * tgt.size(0)
        n_seen += tgt.size(0)

    if n_seen == 0:
        break
    complete = (n_seen == len(train_cp))

    dev = _eval_dev(model, dev_loader, DEVICE)
    improved = dev["top1"] > best_top1

    row = {"epoch": epoch, "complete": complete, "n_seen": n_seen,
           "train_loss": loss_sum / n_seen,
           "dev_top1": dev["top1"], "dev_top5": dev["top5"], "dev_ce": dev["ce"],
           "correct_top1": dev["correct_top1"], "correct_top5": dev["correct_top5"],
           "dev_n": dev["n"], "elapsed_s": round(time.monotonic() - t_start, 1)}
    metrics.append(row)

    star = "  ← best" if improved else ""
    tag  = "" if complete else " (partial)"
    print(f"Epoch {epoch}{tag}: loss={row['train_loss']:.4f}  "
          f"dev={dev['correct_top1']}/{dev['n']} ({dev['top1']:.2%})  "
          f"top-5={dev['top5']:.2%}  ce={dev['ce']:.4f}  "
          f"t={row['elapsed_s']:.0f}s{star}")

    if improved:
        best_top1  = dev["top1"]
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_optim = optimizer.state_dict()
        bad_epochs = 0
    else:
        bad_epochs += 1

    # Save metrics after every epoch
    with open(EXT_RUN_DIR / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    if bad_epochs >= PATIENCE:
        print(f"Early stopping: {PATIENCE} epochs without improvement."); break
    if time.monotonic() - t_start >= MAX_SECONDS:
        print("Budget reached; stopping."); break

# ── Save best checkpoint ─────────────────────────────────────
torch.save(
    {"epoch": best_epoch, "model_state_dict": best_state,
     "optimizer_state_dict": best_optim, "best_dev_top1": best_top1,
     "metrics": metrics, "n_classes": N_CLASSES,
     "pitch_classes": pitch_classes,
     "source_checkpoint": str(RESUME_DIR / "best_checkpoint.pt")},
    EXT_RUN_DIR / "best_checkpoint.pt",
)

# ── Load best and report ──────────────────────────────────────
model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()
final = _eval_dev(model, dev_loader, DEVICE)

with open(EXT_RUN_DIR / "final_dev_metrics.json", "w") as f:
    json.dump({"best_epoch": best_epoch, **final}, f, indent=2)

# ── Print summary ─────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Best epoch: {best_epoch}")
print(f"  Top-1: {final['correct_top1']}/{final['n']} = {final['top1']:.2%}")
print(f"  Top-5: {final['correct_top5']}/{final['n']} = {final['top5']:.2%}")
print(f"  CE:    {final['ce']:.4f}")
print(f"\nBaseline (epoch 4): 304/499 = 60.92%  top-5 = 83.57%")
dt1 = final["correct_top1"] - 304
dt5 = final["correct_top5"] - 417
print(f"Δ top-1: {dt1:+d} examples ({(final['top1'] - 0.6092)*100:+.2f}pp)")
print(f"Δ top-5: {dt5:+d} examples ({(final['top5'] - 0.8357)*100:+.2f}pp)")
print(f"\nSaved to: {EXT_RUN_DIR}")

# Point M2_RUN_DIR to the new run for the results cell below
M2_RUN_DIR = EXT_RUN_DIR


## Demo — Pitch Prediction → Corrected MusicXML → MIDI

Select 6 dev samples (seed=42), run inference with the epoch-5 checkpoint
(312/499 = 62.53% top-1), then generate three MusicXML + MIDI versions per sample:

| Version | Description |
|---------|-------------|
| **ground-truth** | Original score (correct pitch) |
| **corrupted** | Score with the synthetic error (replacement pitch) |
| **model-corrected** | Score with the model's top-1 prediction applied |


In [ ]:
# Select 6 dev samples (seed=42) and run inference.
# Requires: cells 1–12 (setup, model classes, data), cell 18 (extended training) OR
# manual checkpoint load below.

import copy
from lxml import etree
import mido

DEMO_CKPT = ML_DIR / "data" / "runs" / "m2bert_finetune_resume_20261030_183302_306020" / "best_checkpoint.pt"
DEMO_DIR  = ML_DIR / "demo"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

# ── Load checkpoint ────────────────────────────────────────
_ckpt = torch.load(DEMO_CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(_ckpt["model_state_dict"], strict=True)
model.to(DEVICE)
model.eval()
print(f"Loaded checkpoint: dev top-1 = {_ckpt['best_dev_top1']:.4f}")
del _ckpt

# ── Select 6 samples ───────────────────────────────────────
rng = random.Random(42)
DEMO_INDICES = sorted(rng.sample(range(len(dev_cp)), 6))
demo_examples = [dev_cp[i] for i in DEMO_INDICES]
demo_raw      = [dev_raw[i] for i in DEMO_INDICES]

with open(DEMO_DIR / "demo_sample_ids.json", "w") as f:
    json.dump([{"index": i, "sample_id": dev_raw[i]["sample_id"]} for i in DEMO_INDICES], f, indent=2)

# ── Batch inference ────────────────────────────────────────
demo_loader = DataLoader(CPDataset(demo_examples), batch_size=6,
                         shuffle=False, collate_fn=collate_cp)
cp_ids, err_idx, tgt, attn = next(iter(demo_loader))

with torch.inference_mode():
    logits = model(cp_ids.to(DEVICE), err_idx.to(DEVICE), attn.to(DEVICE))
    probs  = logits.softmax(dim=-1).cpu()
    top5   = probs.topk(5, dim=-1)

# ── Display results ────────────────────────────────────────
demo_results = []
print(f"{'Idx':>4} {'Sample ID':>30}  {'Error':>5} {'Truth':>5} {'Pred':>5} {'OK?':>4}  Top-5")
print("─" * 95)
for b in range(6):
    raw       = demo_raw[b]
    tgt_id    = tgt[b].item()
    pred_id   = top5.indices[b][0].item()
    correct   = (pred_id == tgt_id)
    top5_list = [id2pitch[top5.indices[b][k].item()] for k in range(5)]
    res = {
        "dev_index":    DEMO_INDICES[b],
        "sample_id":    raw["sample_id"],
        "error_pitch":  raw["replacement_token"],
        "truth_pitch":  raw["target_token"],
        "pred_pitch":   id2pitch[pred_id],
        "correct":      correct,
        "top5":         top5_list,
    }
    demo_results.append(res)
    mark = "  ✓" if correct else "  ✗"
    print(f"{DEMO_INDICES[b]:4d} {raw['sample_id']:>30}  "
          f"{raw['replacement_token']:>5} {raw['target_token']:>5} "
          f"{id2pitch[pred_id]:>5}{mark}  {top5_list}")


In [ ]:
# MusicXML pitch editing and MIDI export utilities.

_STEP_SEMI = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}

def _xml_tag(name, ns):
    return f"{{{ns}}}{name}" if ns else name


def find_error_note_index(ex, parse_lmx_notes):
    """Return the note index (in parse_lmx_notes order = XML note order)
    of the corrupted pitch in a JSONL example."""
    notes, _ = parse_lmx_notes(ex["input_tokens"], ex.get("sample_id", ""))
    return next(i for i, n in enumerate(notes)
               if n["pitch_token_position"] == ex["error_position"])


def set_note_pitch(note_el, pitch_token, ns=""):
    """Rewrite <step>/<octave> of a <note> element; remove <alter>/<accidental>."""
    t = lambda name: _xml_tag(name, ns)
    pitch_el = note_el.find(t("pitch"))
    if pitch_el is None:
        return
    pitch_el.find(t("step")).text   = pitch_token[0]
    pitch_el.find(t("octave")).text = pitch_token[1:]
    alter = pitch_el.find(t("alter"))
    if alter is not None:
        pitch_el.remove(alter)
    acc = note_el.find(t("accidental"))
    if acc is not None:
        note_el.remove(acc)


def make_edited_xml(xml_path, note_index, new_pitch):
    """Return a deep-copied etree with one note's pitch changed."""
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    notes = root.findall(f".//{t('note')}")
    set_note_pitch(notes[note_index], new_pitch, ns)
    return tree


def musicxml_to_midi(xml_tree, midi_path):
    """Minimal MusicXML→MIDI converter using lxml + mido."""
    root = xml_tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    TPQN = 480
    divisions = 1
    events = []       # (onset_tick, midi_note, dur_tick)
    current_time = 0  # in XML divisions
    prev_onset   = 0

    for measure in root.findall(f".//{t('measure')}"):
        for attr in measure.findall(t("attributes")):
            d = attr.find(t("divisions"))
            if d is not None:
                divisions = int(d.text)
        scale = TPQN / divisions
        for el in measure:
            ltag = el.tag.split("}")[-1] if "}" in el.tag else el.tag
            if ltag == "note":
                chord_el = el.find(t("chord"))
                pitch_el = el.find(t("pitch"))
                dur_el   = el.find(t("duration"))
                dur = int(dur_el.text) if dur_el is not None else 0
                onset = prev_onset if chord_el is not None else current_time
                if chord_el is None:
                    prev_onset = current_time
                if pitch_el is not None:
                    step  = pitch_el.find(t("step")).text
                    octave = int(pitch_el.find(t("octave")).text)
                    alt_el = pitch_el.find(t("alter"))
                    alter  = int(float(alt_el.text)) if alt_el is not None else 0
                    midi_n = (octave + 1) * 12 + _STEP_SEMI[step] + alter
                    events.append((int(onset * scale), midi_n, max(1, int(dur * scale))))
                if chord_el is None:
                    current_time += dur
            elif ltag == "backup":
                current_time -= int(el.find(t("duration")).text)
            elif ltag == "forward":
                current_time += int(el.find(t("duration")).text)

    mid   = mido.MidiFile(ticks_per_beat=TPQN)
    track = mido.MidiTrack()
    mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=mido.bpm2tempo(120)))
    msgs = []
    for onset, pitch, dur in events:
        msgs.append((onset, 1, pitch, 80))
        msgs.append((onset + dur, 0, pitch, 0))
    msgs.sort(key=lambda m: (m[0], m[1]))
    prev_tick = 0
    for tick, on, pitch, vel in msgs:
        delta = tick - prev_tick
        mtype = "note_on" if on else "note_off"
        track.append(mido.Message(mtype, note=pitch, velocity=vel, time=delta))
        prev_tick = tick
    mid.save(str(midi_path))
    return len(events)

print("MusicXML/MIDI helpers defined")


In [ ]:
# Generate ground-truth / corrupted / model-corrected MusicXML + MIDI
# for each of the 6 demo samples.

from shutil import copyfile

def _xml_path(sample_id):
    parts = sample_id.split("/")
    return DATASET_ROOT / "samples" / parts[1] / (parts[2] + ".musicxml")

summary_rows = []

for b, res in enumerate(demo_results):
    raw  = demo_raw[b]
    sid  = res["sample_id"]
    slug = sid.replace("/", "_")       # e.g. samples_6404150_p8-s2
    sdir = DEMO_DIR / slug
    sdir.mkdir(exist_ok=True)

    xml_src = _xml_path(sid)
    note_idx = find_error_note_index(raw, parse_lmx_notes)

    # ── 1. Ground truth: copy original ───────────────────────
    gt_xml = sdir / "ground_truth.musicxml"
    copyfile(xml_src, gt_xml)
    gt_tree = etree.parse(str(gt_xml))
    gt_midi = sdir / "ground_truth.mid"
    musicxml_to_midi(gt_tree, gt_midi)

    # ── 2. Corrupted: apply replacement pitch ────────────────
    corr_tree = make_edited_xml(xml_src, note_idx, res["error_pitch"])
    corr_xml  = sdir / "corrupted.musicxml"
    corr_tree.write(str(corr_xml), xml_declaration=True, encoding="UTF-8")
    corr_midi = sdir / "corrupted.mid"
    musicxml_to_midi(corr_tree, corr_midi)

    # ── 3. Model corrected: apply model prediction ───────────
    pred_tree = make_edited_xml(xml_src, note_idx, res["pred_pitch"])
    pred_xml  = sdir / "model_corrected.musicxml"
    pred_tree.write(str(pred_xml), xml_declaration=True, encoding="UTF-8")
    pred_midi = sdir / "model_corrected.mid"
    musicxml_to_midi(pred_tree, pred_midi)

    mark = "✓" if res["correct"] else "✗"
    summary_rows.append({
        "idx": res["dev_index"], "sample_id": sid,
        "error": res["error_pitch"], "truth": res["truth_pitch"],
        "pred": res["pred_pitch"], "ok": mark,
        "dir": str(sdir.relative_to(ML_DIR)),
    })
    print(f"[{b+1}/6] {slug}  truth={res['truth_pitch']}  pred={res['pred_pitch']} {mark}")

# ── Save summary ──────────────────────────────────────────
with open(DEMO_DIR / "demo_summary.json", "w") as f:
    json.dump(summary_rows, f, indent=2)

# ── Display file listing ──────────────────────────────────
print(f"\n{'─'*80}")
print(f"{'#':>2} {'Dev':>4} {'Truth':>5} → {'Pred':>5} {'OK':>3}  Files")
print(f"{'─'*80}")
for i, r in enumerate(summary_rows):
    print(f"{i+1:2d} {r['idx']:4d} {r['truth']:>5} → {r['pred']:>5}  {r['ok']}"
          f"   {r['dir']}/{{ground_truth,corrupted,model_corrected}}.{{musicxml,mid}}")
print(f"\nAll outputs: {DEMO_DIR.relative_to(ML_DIR)}/")


## Duration Correction — Fine-tuning

Task: predict the masked duration of a single corrupted note.  
Same source samples and score-level split as pitch correction.  
One note per sample; target = exact Fraction quarterLength.

**Backbone**: M2BERT pretrained (same checkpoint as pitch, reloaded fresh).  
**MFC baseline**: most-frequent duration class from train.

In [ ]:
# ── Duration Correction: Data Construction ───────────────────────
# Build duration-corruption examples from the same source samples.
# One note per sample, deterministic selection (hash of sample_id).

import hashlib as _hl
from datetime import datetime
import torch.nn.functional as F

DUR_MASK_ID = CP_TOK2ID['Duration']['Duration <MASK>']  # 65

# Replacement duration pool: (LMX_tokens, Fraction_whole_notes)
_DUR_REPL_POOL = [
    (['whole'],          _Frac(1)),
    (['half'],           _Frac(1, 2)),
    (['quarter'],        _Frac(1, 4)),
    (['eighth'],         _Frac(1, 8)),
    (['16th'],           _Frac(1, 16)),
    (['half', 'dot'],    _Frac(3, 4)),
    (['quarter', 'dot'], _Frac(3, 8)),
    (['eighth', 'dot'],  _Frac(3, 16)),
]

def _find_dur_span(tokens, note):
    """Return (start, end) index of duration tokens within note's token_span."""
    s, e = note['token_span']
    for i in range(s, e):
        if tokens[i] in _DURATION_BASE:
            j = i + 1
            if j < e and _TIME_MOD_RE.match(tokens[j]):
                j += 1
            while j < e and tokens[j] == 'dot':
                j += 1
            return i, j
    return None, None

def _dur_token_info(tokens, note):
    """Extract (note_type, n_dots, tuplet_str|None) from LMX tokens."""
    ds, de = _find_dur_span(tokens, note)
    if ds is None:
        return None, 0, None
    ntype = tokens[ds]; j = ds + 1
    tup = None
    if j < de and _TIME_MOD_RE.match(tokens[j]):
        tup = tokens[j]; j += 1
    dots = 0
    while j < de and tokens[j] == 'dot':
        dots += 1; j += 1
    return ntype, dots, tup

def build_dur_split(sample_ids, label_map=None):
    """Build duration-correction CP examples for one split.
    
    For each sample: load clean LMX, select one pitched non-grace note,
    replace its duration tokens, re-parse to get corrupted onsets,
    convert to CP with DUR_MASK at error position.
    
    Returns (cp_list, label_map, details, stats).
    """
    build_map = (label_map is None)
    raw = []
    skip = Counter()
    note_excl = Counter()

    for sid in sample_ids:
        lmx_p = _lmx_path(sid)
        if not lmx_p.exists():
            skip['no_lmx'] += 1; continue

        clean_toks = lmx_p.read_text().split()
        notes, _ = parse_lmx_notes(clean_toks, sid)

        cands = []
        for ni, n in enumerate(notes):
            if n['is_grace']:       note_excl['grace'] += 1; continue
            if n['is_rest']:        note_excl['rest'] += 1; continue
            if n['duration'] is None or n['pitch'] is None:
                note_excl['no_dur_or_pitch'] += 1; continue
            if _find_dur_span(clean_toks, n)[0] is None:
                note_excl['dur_tokens_missing'] += 1; continue
            cands.append(ni)

        if not cands:
            skip['no_candidate'] += 1; continue

        h = int(_hl.md5(sid.encode()).hexdigest(), 16)
        sel = cands[h % len(cands)]
        note = notes[sel]
        true_dur = note['duration']           # Fraction, whole-note units
        true_ql  = str(true_dur * 4)          # quarterLength label

        rng = random.Random(h)
        pool = [(t, f) for t, f in _DUR_REPL_POOL if f != true_dur]
        rtoks, rfrac = pool[rng.randint(0, len(pool) - 1)]

        ds, de = _find_dur_span(clean_toks, note)
        corrupted = clean_toks[:ds] + rtoks + clean_toks[de:]
        ntype, dots, tup = _dur_token_info(clean_toks, note)

        raw.append({
            'sid': sid, 'corrupted': corrupted,
            'epp': note['pitch_token_position'],
            'true_ql': true_ql, 'repl_ql': str(rfrac * 4),
            'ntype': ntype, 'dots': dots, 'tup': tup,
        })

    # Label map from train distribution (most-frequent first)
    if build_map:
        ql_counts = Counter(r['true_ql'] for r in raw)
        label_map = {ql: i for i, (ql, _) in enumerate(ql_counts.most_common())}

    oov_qls = sorted({r['true_ql'] for r in raw if r['true_ql'] not in label_map})

    # CP conversion — re-parse corrupted LMX for correct onsets
    cp_list, details = [], []
    for r in raw:
        if r['true_ql'] not in label_map:
            skip['oov_dur'] += 1; continue
        cnotes, _ = parse_lmx_notes(r['corrupted'], r['sid'])
        err_idx = next((i for i, n in enumerate(cnotes)
                        if n['pitch_token_position'] == r['epp']), None)
        if err_idx is None:
            skip['err_not_found'] += 1; continue

        cp_ids = []
        prev_m = -1
        for i, n in enumerate(cnotes):
            bar = 0 if n['measure'] != prev_m else 1; prev_m = n['measure']
            pos = min(15, int(float(n['onset']) * 16))
            pitch = PITCH_PAD
            if n['pitch'] and PITCH_RE.match(n['pitch']):
                p = _pitch_to_cp_idx(n['pitch'])
                if p is not None: pitch = p
            dur = DUR_MASK_ID if i == err_idx else (
                min(63, int(float(n['duration']) * 32))
                if n['duration'] is not None else DUR_PAD)
            cp_ids.append([bar, pos, pitch, dur])

        cp_list.append({'cp_ids': cp_ids, 'error_note_idx': err_idx,
                        'target_class': label_map[r['true_ql']],
                        'sample_id': r['sid']})
        details.append({'sample_id': r['sid'], 'true_ql': r['true_ql'],
                        'repl_ql': r['repl_ql'], 'note_type': r['ntype'],
                        'dots': r['dots'], 'tuplet': r['tup']})

    stats = {'input_samples': len(sample_ids), 'examples': len(cp_list),
             'skip': dict(skip), 'note_exclusions': dict(note_excl),
             'oov_durations': oov_qls}
    return cp_list, label_map, details, stats

# ── Build ────────────────────────────────────────────────────────
t0 = time.time()

train_sids = list(dict.fromkeys(ex['sample_id'] for ex in train_raw))
dev_sids   = list(dict.fromkeys(ex['sample_id'] for ex in dev_raw))
print(f"Source samples: train={len(train_sids)}, dev={len(dev_sids)}")

dur_train_cp, dur_label_map, dur_train_details, dur_train_stats = \
    build_dur_split(train_sids)
dur_dev_cp, _, dur_dev_details, dur_dev_stats = \
    build_dur_split(dev_sids, label_map=dur_label_map)

dur_id2label = {v: k for k, v in dur_label_map.items()}
N_DUR_CLASSES = len(dur_label_map)

print(f"\nDuration classes: {N_DUR_CLASSES}")
print(f"Label map (class → QL): {dur_label_map}")
print(f"\nTrain: {dur_train_stats['examples']} examples from "
      f"{dur_train_stats['input_samples']} samples")
print(f"  Skip: {dur_train_stats['skip']}")
print(f"  Note exclusions: {dur_train_stats['note_exclusions']}")
print(f"Dev:   {dur_dev_stats['examples']} examples from "
      f"{dur_dev_stats['input_samples']} samples")
print(f"  Skip: {dur_dev_stats['skip']}")
print(f"  Note exclusions: {dur_dev_stats['note_exclusions']}")
if dur_dev_stats['oov_durations']:
    print(f"  Dev OOV durations: {dur_dev_stats['oov_durations']}")
else:
    print(f"  All dev durations seen in train.")

# MFC baseline
_train_dur_counts = Counter(d['true_ql'] for d in dur_train_details)
dur_mfc_label, dur_mfc_train_n = _train_dur_counts.most_common(1)[0]
dur_mfc_cls = dur_label_map[dur_mfc_label]
dur_mfc_hits = sum(1 for e in dur_dev_cp if e['target_class'] == dur_mfc_cls)
print(f"\nMFC: '{dur_mfc_label}' QL ({dur_mfc_train_n} in train) → "
      f"dev hits: {dur_mfc_hits}/{len(dur_dev_cp)} "
      f"({dur_mfc_hits/len(dur_dev_cp):.2%})")

# Top-10 duration distribution
print(f"\nTop-10 durations in train:")
for ql, cnt in _train_dur_counts.most_common(10):
    print(f"  {ql:>8s} QL: {cnt:>5d} ({cnt/len(dur_train_details)*100:.1f}%)")

print(f"\nData construction: {time.time()-t0:.1f}s")

In [ ]:
# ── Duration Model Setup + Round 1 Training ──────────────────────
# Fresh backbone from pretrained M2BERT, new duration classification head.
# Freeze all except layers 10-11 and head. Max 6 epochs, patience=2.

DUR_RUN_DIR = ML_DIR / 'data' / 'runs' / (
    'm2bert_dur_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
DUR_RUN_DIR.mkdir(parents=True, exist_ok=True)

# ── Fresh backbone ───────────────────────────────────────────
dur_backbone = MidiModernBERT()
_ckpt = torch.load(PRETRAINED / 'model_best.ckpt', map_location='cpu',
                    weights_only=False)
dur_backbone.load_state_dict(_ckpt['state_dict'], strict=True)
print(f'Backbone loaded (pretrained epoch {_ckpt["epoch"]})')
del _ckpt

# Reuse M2BERTPitchCorrection class — same architecture, different head size
dur_model = M2BERTPitchCorrection(dur_backbone, N_DUR_CLASSES)
dur_model.to(DEVICE)

# Freeze all, unfreeze layers 10-11 + classifier
for p in dur_model.parameters():
    p.requires_grad_(False)
for layer_idx in (10, 11):
    for p in dur_model.backbone.note_bert1.layers[layer_idx].parameters():
        p.requires_grad_(True)
for p in dur_model.classifier.parameters():
    p.requires_grad_(True)

n_train_p = sum(p.numel() for p in dur_model.parameters() if p.requires_grad)
n_frz_p   = sum(p.numel() for p in dur_model.parameters() if not p.requires_grad)
print(f'Trainable: {n_train_p:,}  Frozen: {n_frz_p:,}  '
      f'Head out: {N_DUR_CLASSES} classes')

dur_optimizer = torch.optim.Adam(
    [p for p in dur_model.parameters() if p.requires_grad], lr=1e-4)
dur_loss_fn = nn.CrossEntropyLoss()

# Save config and selection records
_dur_config = {
    'n_classes': N_DUR_CLASSES, 'lr': 1e-4, 'batch_size': 32,
    'max_epochs': 6, 'patience': 2, 'frozen_layers': '0-9',
    'trainable_layers': '10-11 + head',
    'train_examples': len(dur_train_cp), 'dev_examples': len(dur_dev_cp),
    'mfc_label': dur_mfc_label, 'mfc_dev_hits': dur_mfc_hits,
}
with open(DUR_RUN_DIR / 'config.json', 'w') as f:
    json.dump(_dur_config, f, indent=2)
with open(DUR_RUN_DIR / 'dur_label_map.json', 'w') as f:
    json.dump(dur_label_map, f, indent=2)
with open(DUR_RUN_DIR / 'train_selection.json', 'w') as f:
    json.dump(dur_train_details, f, indent=2)
with open(DUR_RUN_DIR / 'dev_selection.json', 'w') as f:
    json.dump(dur_dev_details, f, indent=2)

# ── Data loaders ─────────────────────────────────────────────
dur_train_loader = DataLoader(CPDataset(dur_train_cp), batch_size=32,
                              shuffle=True, collate_fn=collate_cp, num_workers=0)
dur_dev_loader   = DataLoader(CPDataset(dur_dev_cp), batch_size=64,
                              shuffle=False, collate_fn=collate_cp, num_workers=0)

def dur_eval(mdl, loader):
    mdl.eval()
    n = top1 = top3 = 0; ce_sum = 0.0
    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
            logits = mdl(cp_ids, err_idx, attn)
            top1 += (logits.argmax(1) == tgt).sum().item()
            top3 += (logits.topk(min(3, logits.size(1)), dim=1
                     ).indices == tgt[:, None]).any(1).sum().item()
            ce_sum += F.cross_entropy(logits, tgt, reduction='sum').item()
            n += tgt.size(0)
    return {'n': n, 'top1': top1, 'top3': top3,
            'top1_acc': top1 / n, 'top3_acc': top3 / n, 'ce': ce_sum / n}

# ── Training Round 1 ────────────────────────────────────────
MAX_SECONDS_TOTAL = 600
dur_metrics_r1 = []
best_dur_top1 = 0.0
best_dur_ce   = float('inf')
best_dur_epoch = 0
bad_epochs = 0
t_global = time.monotonic()

print(f"\nRound 1: max 6 epochs, patience=2, budget=600s")
print(f"Output: {DUR_RUN_DIR}")

for epoch in range(1, 7):
    if time.monotonic() - t_global >= MAX_SECONDS_TOTAL:
        print("Budget reached."); break

    dur_model.train()
    loss_sum = n_seen = 0
    for cp_ids, err_idx, tgt, attn in dur_train_loader:
        if time.monotonic() - t_global >= MAX_SECONDS_TOTAL:
            break
        cp_ids, err_idx, tgt, attn = (
            x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
        dur_optimizer.zero_grad(set_to_none=True)
        logits = dur_model(cp_ids, err_idx, attn)
        loss = dur_loss_fn(logits, tgt)
        loss.backward()
        dur_optimizer.step()
        loss_sum += loss.item() * tgt.size(0)
        n_seen += tgt.size(0)

    if n_seen == 0:
        break
    dev = dur_eval(dur_model, dur_dev_loader)

    # top-1 primary; CE tiebreaker
    improved = (dev['top1_acc'] > best_dur_top1 or
                (dev['top1_acc'] == best_dur_top1 and dev['ce'] < best_dur_ce))

    row = {'epoch': epoch, 'train_loss': loss_sum / n_seen,
           'dev_top1': dev['top1_acc'], 'dev_top3': dev['top3_acc'],
           'dev_ce': dev['ce'],
           'correct_top1': dev['top1'], 'correct_top3': dev['top3'],
           'dev_n': dev['n'],
           'elapsed_s': time.monotonic() - t_global}
    dur_metrics_r1.append(row)

    if improved:
        best_dur_top1 = dev['top1_acc']
        best_dur_ce   = dev['ce']
        best_dur_epoch = epoch
        bad_epochs = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': {k: v.cpu()
                                 for k, v in dur_model.state_dict().items()},
            'optimizer_state_dict': dur_optimizer.state_dict(),
            'best_dev_top1': best_dur_top1,
            'metrics': dur_metrics_r1,
            'n_classes': N_DUR_CLASSES,
            'dur_label_map': dur_label_map,
        }, DUR_RUN_DIR / 'best_checkpoint.pt')
    else:
        bad_epochs += 1

    with open(DUR_RUN_DIR / 'metrics.json', 'w') as f:
        json.dump(dur_metrics_r1, f, indent=2)

    mark = ' *best*' if improved else ''
    print(f"  Epoch {epoch}: loss={loss_sum/n_seen:.4f}  "
          f"dev={dev['top1']}/{dev['n']} ({dev['top1_acc']:.2%})  "
          f"top-3={dev['top3_acc']:.2%}  CE={dev['ce']:.3f}{mark}")

    if bad_epochs >= 2:
        print(f"  Early stop: 2 epochs without improvement.")
        break

# ── Restore best R1 ─────────────────────────────────────────
_ckpt = torch.load(DUR_RUN_DIR / 'best_checkpoint.pt',
                    map_location='cpu', weights_only=False)
dur_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
dur_model.to(DEVICE); dur_model.eval()
del _ckpt

r1_dev = dur_eval(dur_model, dur_dev_loader)
with open(DUR_RUN_DIR / 'final_dev_metrics.json', 'w') as f:
    json.dump({'best_epoch': best_dur_epoch, **r1_dev,
               'mfc_hits': dur_mfc_hits}, f, indent=2)

r1_elapsed = time.monotonic() - t_global
print(f"\nRound 1 best: epoch {best_dur_epoch}")
print(f"  Top-1: {r1_dev['top1']}/{r1_dev['n']} = {r1_dev['top1_acc']:.2%}")
print(f"  Top-3: {r1_dev['top3']}/{r1_dev['n']} = {r1_dev['top3_acc']:.2%}")
print(f"  CE:    {r1_dev['ce']:.3f}")
print(f"  MFC:   {dur_mfc_hits}/{r1_dev['n']} = {dur_mfc_hits/r1_dev['n']:.2%}")
print(f"  Time:  {r1_elapsed:.0f}s")

In [ ]:
# ── Duration Round 2 (conditional) ───────────────────────────────
# Trigger if R1 didn't beat MFC, or dev top-1 dropped ≥5% from peak.
# Modification: differential LR — backbone 2e-5, head 1e-4.

r1_beat_mfc = r1_dev['top1'] > dur_mfc_hits
r1_overfit  = (len(dur_metrics_r1) >= 3 and
               dur_metrics_r1[-1]['dev_top1'] < max(r['dev_top1'] for r in dur_metrics_r1) * 0.95)
need_r2 = not r1_beat_mfc or r1_overfit
budget_left = MAX_SECONDS_TOTAL - r1_elapsed

# Initialise R2 variables (used by results cell regardless)
r2_dev = None
DUR_RUN_DIR_R2 = None
dur_metrics_r2 = []
best_r2_epoch = 0

if need_r2 and budget_left > 30:
    reasons = []
    if not r1_beat_mfc:
        reasons.append(f"R1 ({r1_dev['top1']}) <= MFC ({dur_mfc_hits})")
    if r1_overfit:
        reasons.append("dev top-1 dropped >=5% from peak")
    print(f"Round 2 triggered: {'; '.join(reasons)}")
    print(f"Modification: backbone LR 2e-5, head LR 1e-4 (was uniform 1e-4)")

    # Reload R1 best
    _ckpt = torch.load(DUR_RUN_DIR / 'best_checkpoint.pt',
                        map_location='cpu', weights_only=False)
    dur_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
    dur_model.to(DEVICE)
    del _ckpt

    backbone_lr, head_lr = 2e-5, 1e-4
    dur_optimizer_r2 = torch.optim.Adam([
        {'params': [p for li in (10, 11)
                    for p in dur_model.backbone.note_bert1.layers[li].parameters()
                    if p.requires_grad],
         'lr': backbone_lr},
        {'params': list(dur_model.classifier.parameters()), 'lr': head_lr},
    ])

    DUR_RUN_DIR_R2 = ML_DIR / 'data' / 'runs' / (
        'm2bert_dur_r2_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    DUR_RUN_DIR_R2.mkdir(parents=True, exist_ok=True)
    with open(DUR_RUN_DIR_R2 / 'config.json', 'w') as f:
        json.dump({'round': 2, 'backbone_lr': backbone_lr, 'head_lr': head_lr,
                   'reason': reasons, 'source': str(DUR_RUN_DIR),
                   'n_classes': N_DUR_CLASSES}, f, indent=2)

    best_r2_top1 = best_dur_top1
    best_r2_ce   = best_dur_ce
    bad_r2 = 0
    t_r2 = time.monotonic()
    print(f"Output: {DUR_RUN_DIR_R2}")

    for epoch in range(1, 7):
        if time.monotonic() - t_global >= MAX_SECONDS_TOTAL:
            print("  Budget reached."); break

        dur_model.train()
        loss_sum = n_seen = 0
        for cp_ids, err_idx, tgt, attn in dur_train_loader:
            if time.monotonic() - t_global >= MAX_SECONDS_TOTAL:
                break
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
            dur_optimizer_r2.zero_grad(set_to_none=True)
            logits = dur_model(cp_ids, err_idx, attn)
            loss = dur_loss_fn(logits, tgt)
            loss.backward()
            dur_optimizer_r2.step()
            loss_sum += loss.item() * tgt.size(0)
            n_seen += tgt.size(0)

        if n_seen == 0:
            break
        dev = dur_eval(dur_model, dur_dev_loader)
        improved = (dev['top1_acc'] > best_r2_top1 or
                    (dev['top1_acc'] == best_r2_top1 and dev['ce'] < best_r2_ce))

        row = {'epoch': epoch, 'train_loss': loss_sum / n_seen,
               'dev_top1': dev['top1_acc'], 'dev_top3': dev['top3_acc'],
               'dev_ce': dev['ce'],
               'correct_top1': dev['top1'], 'correct_top3': dev['top3'],
               'dev_n': dev['n'],
               'elapsed_s': time.monotonic() - t_r2}
        dur_metrics_r2.append(row)

        if improved:
            best_r2_top1 = dev['top1_acc']
            best_r2_ce   = dev['ce']
            best_r2_epoch = epoch
            bad_r2 = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': {k: v.cpu()
                                     for k, v in dur_model.state_dict().items()},
                'optimizer_state_dict': dur_optimizer_r2.state_dict(),
                'best_dev_top1': best_r2_top1,
                'metrics': dur_metrics_r2,
                'n_classes': N_DUR_CLASSES,
                'dur_label_map': dur_label_map,
                'round': 2, 'backbone_lr': backbone_lr, 'head_lr': head_lr,
            }, DUR_RUN_DIR_R2 / 'best_checkpoint.pt')
        else:
            bad_r2 += 1

        with open(DUR_RUN_DIR_R2 / 'metrics.json', 'w') as f:
            json.dump(dur_metrics_r2, f, indent=2)

        mark = ' *best*' if improved else ''
        print(f"  R2 Epoch {epoch}: loss={loss_sum/n_seen:.4f}  "
              f"dev={dev['top1']}/{dev['n']} ({dev['top1_acc']:.2%})  "
              f"top-3={dev['top3_acc']:.2%}  CE={dev['ce']:.3f}{mark}")

        if bad_r2 >= 2:
            print("  Early stop."); break

    # Restore R2 best
    if (DUR_RUN_DIR_R2 / 'best_checkpoint.pt').exists():
        _ckpt = torch.load(DUR_RUN_DIR_R2 / 'best_checkpoint.pt',
                            map_location='cpu', weights_only=False)
        dur_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
        dur_model.to(DEVICE); dur_model.eval()
        del _ckpt
        r2_dev = dur_eval(dur_model, dur_dev_loader)
        with open(DUR_RUN_DIR_R2 / 'final_dev_metrics.json', 'w') as f:
            json.dump({'best_epoch': best_r2_epoch, **r2_dev}, f, indent=2)
        print(f"\n  R2 best: epoch {best_r2_epoch}  "
              f"top-1={r2_dev['top1']}/{r2_dev['n']} ({r2_dev['top1_acc']:.2%})")
    else:
        print("  R2 produced no checkpoint (budget exhausted mid-epoch).")

elif need_r2:
    print(f"Round 2 needed but budget exhausted ({budget_left:.0f}s left).")
else:
    print(f"Round 2 not needed: R1 beats MFC and no overfitting.")

In [ ]:
# ── Duration Results & Demo ──────────────────────────────────────

# Pick the best model (R2 if it exists and is better, else R1)
if r2_dev is not None and r2_dev['top1'] > r1_dev['top1']:
    best_dur_run = DUR_RUN_DIR_R2
    best_dur_final = r2_dev
    best_dur_src = 'R2'
else:
    best_dur_run = DUR_RUN_DIR
    best_dur_final = r1_dev
    best_dur_src = 'R1'

# ── Comparison table ─────────────────────────────────────────
n_dev = best_dur_final['n']
print(f"{'='*72}")
print(f"  DURATION CORRECTION — COMPARISON TABLE  (dev N={n_dev})")
print(f"{'='*72}")
print(f"  {'Method':<28} {'Top-1':>10} {'Top-3':>10} {'Dev CE':>10}")
print(f"  {'─'*68}")
mfc_pct = f"{dur_mfc_hits/n_dev:.2%}"
print(f"  {'MFC ('+dur_mfc_label+' QL)':<28} "
      f"{dur_mfc_hits:>4}/{n_dev:<4}     {'—':>10} {'—':>10}")
r1_t1 = f"{r1_dev['top1']}/{n_dev}"
r1_t3 = f"{r1_dev['top3']}/{n_dev}"
print(f"  {'R1 (epoch '+str(best_dur_epoch)+', LR=1e-4)':<28} "
      f"{r1_t1:>10} {r1_t3:>10} {r1_dev['ce']:>10.3f}")
if r2_dev is not None:
    r2_t1 = f"{r2_dev['top1']}/{n_dev}"
    r2_t3 = f"{r2_dev['top3']}/{n_dev}"
    print(f"  {'R2 (epoch '+str(best_r2_epoch)+', diff LR)':<28} "
          f"{r2_t1:>10} {r2_t3:>10} {r2_dev['ce']:>10.3f}")
print(f"  {'─'*68}")

# ── Training curve ───────────────────────────────────────────
print(f"\nR1 training curve:")
for r in dur_metrics_r1:
    print(f"  Epoch {r['epoch']}: train_loss={r['train_loss']:.4f}  "
          f"dev_top1={r['correct_top1']}/{r['dev_n']} ({r['dev_top1']:.2%})  "
          f"dev_top3={r['dev_top3']:.2%}  CE={r['dev_ce']:.3f}  "
          f"time={r['elapsed_s']:.0f}s")
if dur_metrics_r2:
    print(f"\nR2 training curve:")
    for r in dur_metrics_r2:
        print(f"  Epoch {r['epoch']}: train_loss={r['train_loss']:.4f}  "
              f"dev_top1={r['correct_top1']}/{r['dev_n']} ({r['dev_top1']:.2%})  "
              f"dev_top3={r['dev_top3']:.2%}  CE={r['dev_ce']:.3f}  "
              f"time={r['elapsed_s']:.0f}s")

# ── Save dev predictions ─────────────────────────────────────
dur_model.eval()
dev_preds = []
with torch.inference_mode():
    for cp_ids, err_idx, tgt, attn in dur_dev_loader:
        cp_ids, err_idx, tgt, attn = (
            x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
        logits = dur_model(cp_ids, err_idx, attn)
        preds = logits.argmax(1).cpu()
        for j in range(tgt.size(0)):
            dev_preds.append({
                'target': dur_id2label[tgt[j].item()],
                'predicted': dur_id2label[preds[j].item()],
                'correct': bool(preds[j] == tgt[j]),
            })
with open(best_dur_run / 'dev_predictions.json', 'w') as f:
    json.dump(dev_preds, f, indent=2)

# ── 6-sample demo ────────────────────────────────────────────
rng_d = random.Random(42)
d_idx = sorted(rng_d.sample(range(len(dur_dev_cp)), 6))
d_ex  = [dur_dev_cp[i] for i in d_idx]
d_det = [dur_dev_details[i] for i in d_idx]

d_loader = DataLoader(CPDataset(d_ex), batch_size=6,
                      shuffle=False, collate_fn=collate_cp)
cp_ids, err_idx, tgt, attn = next(iter(d_loader))
with torch.inference_mode():
    logits = dur_model(cp_ids.to(DEVICE), err_idx.to(DEVICE), attn.to(DEVICE))
    probs = logits.softmax(-1).cpu()
    topk = probs.topk(min(5, N_DUR_CLASSES), dim=-1)

print(f"\n{'='*90}")
print(f"  6-SAMPLE DEMO (seed=42)")
print(f"{'='*90}")
print(f"  {'Idx':>4} {'Sample':>28}  {'Error':>8} {'Truth':>8} {'Pred':>8} {'OK':>3}  Candidates")
print(f"  {'─'*86}")
for b in range(6):
    tgt_id  = tgt[b].item()
    pred_id = topk.indices[b][0].item()
    ok = ' ok' if pred_id == tgt_id else '  X'
    cands = [dur_id2label[topk.indices[b][k].item()]
             for k in range(min(5, N_DUR_CLASSES))]
    d = d_det[b]
    print(f"  {d_idx[b]:4d} {d['sample_id']:>28}  "
          f"{d['repl_ql']:>8} {d['true_ql']:>8} "
          f"{dur_id2label[pred_id]:>8}{ok}  {cands}")

# ── Summary ──────────────────────────────────────────────────
beat_mfc = best_dur_final['top1'] > dur_mfc_hits
r2_ran   = r2_dev is not None
total_s  = time.monotonic() - t_global

print(f"""
{'='*72}
DURATION CORRECTION — SUMMARY
{'='*72}
Task:    Predict masked note duration (exact quarterLength, {N_DUR_CLASSES} classes).
Data:    {len(dur_train_cp)} train / {len(dur_dev_cp)} dev examples.
         Grace notes excluded; rests excluded; OOV: {dur_dev_stats['oov_durations']}.
Model:   M2BERT backbone (frozen 0-9) + Linear({N_DUR_CLASSES}).
MFC:     {dur_mfc_hits}/{n_dev} ({dur_mfc_hits/n_dev:.2%}) — always predict '{dur_mfc_label}' QL.
R1:      {r1_dev['top1']}/{n_dev} ({r1_dev['top1_acc']:.2%}), epoch {best_dur_epoch}, LR=1e-4.
R2:      {'Not run' if not r2_ran else f"{r2_dev['top1']}/{n_dev} ({r2_dev['top1_acc']:.2%}), epoch {best_r2_epoch}, diff LR."} {'('+best_dur_src+' selected)' if r2_ran else ''}
Best:    {best_dur_final['top1']}/{n_dev} ({best_dur_final['top1_acc']:.2%}) [{best_dur_src}]
         {'Beats MFC.' if beat_mfc else 'Does NOT beat MFC.'}
Time:    {total_s:.0f}s total training.

Next concrete improvement:
  The duration label space is small ({N_DUR_CLASSES} classes) and highly imbalanced
  (MFC covers {dur_mfc_hits/n_dev:.0%} of dev). The CP Position attribute quantises
  onset to 16th-note grid (0-15), losing sub-beat precision that distinguishes
  tuplet from straight durations. A concrete next step: add an explicit
  onset-in-ticks attribute (finer than 16th grid) to the CP input, giving the
  model access to the rhythmic context that the current 4-bit position discards.
{'='*72}
""")

## Joint Pitch + Duration Correction

Shared M2BERT backbone, two prediction heads (52-class pitch, 28-class duration).  
Train with 1:1 interleaved batches; checkpoint by average dev top-1.

**Single-task baselines** (to preserve, not retrain):
- Pitch: 312/499 (62.53%), top-5 427/499 (85.57%)
- Duration: 434/499 (86.97%), top-3 481/499 (96.39%)

In [ ]:
# ── Joint Model: Setup + Round 1 Training ────────────────────────
# Shared backbone, two heads. 1:1 interleaved pitch/duration batches.
# Checkpoint by mean(pitch_dev_top1, dur_dev_top1).

from itertools import zip_longest

# ── Model ────────────────────────────────────────────────────
class M2BERTJointCorrection(nn.Module):
    """Shared backbone with separate pitch and duration heads."""
    def __init__(self, backbone, n_pitch, n_dur):
        super().__init__()
        self.backbone   = backbone
        self.pitch_head = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_pitch))
        self.dur_head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_dur))

    def forward(self, cp_ids, error_note_idx, attention_mask=None, task='pitch'):
        hidden = self.backbone(cp_ids, attention_mask)
        err_h  = hidden[torch.arange(hidden.size(0)), error_note_idx]
        if task == 'pitch':
            return self.pitch_head(err_h)
        return self.dur_head(err_h)

# Fresh pretrained backbone
_jbb = MidiModernBERT()
_ckpt = torch.load(PRETRAINED / 'model_best.ckpt', map_location='cpu',
                    weights_only=False)
_jbb.load_state_dict(_ckpt['state_dict'], strict=True)
print(f'Backbone loaded (pretrained epoch {_ckpt["epoch"]})')
del _ckpt

joint_model = M2BERTJointCorrection(_jbb, N_CLASSES, N_DUR_CLASSES)
joint_model.to(DEVICE)

# Freeze layers 0-9, train layers 10-11 + both heads
for p in joint_model.parameters():
    p.requires_grad_(False)
for li in (10, 11):
    for p in joint_model.backbone.note_bert1.layers[li].parameters():
        p.requires_grad_(True)
for p in joint_model.pitch_head.parameters():
    p.requires_grad_(True)
for p in joint_model.dur_head.parameters():
    p.requires_grad_(True)

n_tp = sum(p.numel() for p in joint_model.parameters() if p.requires_grad)
n_fp = sum(p.numel() for p in joint_model.parameters() if not p.requires_grad)
print(f'Trainable: {n_tp:,}  Frozen: {n_fp:,}')
print(f'Heads: pitch={N_CLASSES}, duration={N_DUR_CLASSES}')

JOINT_LR = 1e-4
joint_optimizer = torch.optim.Adam(
    [p for p in joint_model.parameters() if p.requires_grad], lr=JOINT_LR)
joint_loss_fn = nn.CrossEntropyLoss()

JOINT_RUN = ML_DIR / 'data' / 'runs' / (
    'm2bert_joint_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
JOINT_RUN.mkdir(parents=True, exist_ok=True)

with open(JOINT_RUN / 'config.json', 'w') as f:
    json.dump({'lr': JOINT_LR, 'batch_size': 32, 'max_epochs': 6,
               'patience': 2, 'n_pitch_classes': N_CLASSES,
               'n_dur_classes': N_DUR_CLASSES,
               'trainable_layers': '10-11 + both heads',
               'pitch_train_n': len(train_cp), 'dur_train_n': len(dur_train_cp),
               'pitch_dev_n': len(dev_cp), 'dur_dev_n': len(dur_dev_cp),
               'interleave': '1:1'}, f, indent=2)
with open(JOINT_RUN / 'pitch_label_map.json', 'w') as f:
    json.dump(pitch_classes, f, indent=2)
with open(JOINT_RUN / 'dur_label_map.json', 'w') as f:
    json.dump(dur_label_map, f, indent=2)

# ── Data loaders ─────────────────────────────────────────────
jt_pitch_loader = DataLoader(CPDataset(train_cp), batch_size=32,
                             shuffle=True, collate_fn=collate_cp, num_workers=0)
jt_dur_loader   = DataLoader(CPDataset(dur_train_cp), batch_size=32,
                             shuffle=True, collate_fn=collate_cp, num_workers=0)
jt_pitch_dev    = DataLoader(CPDataset(dev_cp), batch_size=64,
                             shuffle=False, collate_fn=collate_cp, num_workers=0)
jt_dur_dev      = DataLoader(CPDataset(dur_dev_cp), batch_size=64,
                             shuffle=False, collate_fn=collate_cp, num_workers=0)

def joint_eval(mdl, pitch_loader, dur_loader):
    """Evaluate both heads separately. Returns (pitch_metrics, dur_metrics)."""
    mdl.eval()
    results = {}
    for task, loader, topk in [('pitch', pitch_loader, 5),
                                ('dur', dur_loader, 3)]:
        n = t1 = tk = 0; ce = 0.0
        with torch.inference_mode():
            for cp_ids, err_idx, tgt, attn in loader:
                cp_ids, err_idx, tgt, attn = (
                    x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
                logits = mdl(cp_ids, err_idx, attn, task=task)
                t1 += (logits.argmax(1) == tgt).sum().item()
                tk += (logits.topk(min(topk, logits.size(1)), dim=1
                       ).indices == tgt[:, None]).any(1).sum().item()
                ce += F.cross_entropy(logits, tgt, reduction='sum').item()
                n += tgt.size(0)
        results[task] = {'n': n, 'top1': t1, f'top{topk}': tk,
                         'top1_acc': t1 / n, f'top{topk}_acc': tk / n,
                         'ce': ce / n}
    return results['pitch'], results['dur']

# ── Training Round 1 ────────────────────────────────────────
JT_BUDGET = 600
jt_metrics = []
best_jt_score = 0.0       # mean(pitch_top1_acc, dur_top1_acc)
best_jt_ce    = float('inf')
best_jt_epoch = 0
jt_bad = 0
jt_t0 = time.monotonic()

print(f"\nJoint R1: max 6 epochs, patience=2, budget={JT_BUDGET}s")
print(f"Output: {JOINT_RUN}")

for epoch in range(1, 7):
    if time.monotonic() - jt_t0 >= JT_BUDGET:
        print("Budget reached."); break

    joint_model.train()
    p_loss_sum = p_n = d_loss_sum = d_n = 0

    # Interleave: alternate pitch batch, duration batch
    for p_batch, d_batch in zip_longest(jt_pitch_loader, jt_dur_loader):
        if time.monotonic() - jt_t0 >= JT_BUDGET:
            break

        # Pitch step
        if p_batch is not None:
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in p_batch)
            joint_optimizer.zero_grad(set_to_none=True)
            logits = joint_model(cp_ids, err_idx, attn, task='pitch')
            loss = joint_loss_fn(logits, tgt)
            loss.backward()
            joint_optimizer.step()
            p_loss_sum += loss.item() * tgt.size(0)
            p_n += tgt.size(0)

        if time.monotonic() - jt_t0 >= JT_BUDGET:
            break

        # Duration step
        if d_batch is not None:
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in d_batch)
            joint_optimizer.zero_grad(set_to_none=True)
            logits = joint_model(cp_ids, err_idx, attn, task='dur')
            loss = joint_loss_fn(logits, tgt)
            loss.backward()
            joint_optimizer.step()
            d_loss_sum += loss.item() * tgt.size(0)
            d_n += tgt.size(0)

    if p_n == 0 and d_n == 0:
        break

    p_dev, d_dev = joint_eval(joint_model, jt_pitch_dev, jt_dur_dev)
    score = (p_dev['top1_acc'] + d_dev['top1_acc']) / 2
    avg_ce = (p_dev['ce'] + d_dev['ce']) / 2
    improved = (score > best_jt_score or
                (score == best_jt_score and avg_ce < best_jt_ce))

    row = {
        'epoch': epoch,
        'pitch_train_loss': p_loss_sum / max(p_n, 1),
        'dur_train_loss': d_loss_sum / max(d_n, 1),
        'pitch_dev_top1': p_dev['top1_acc'],
        'pitch_dev_top5': p_dev['top5_acc'],
        'pitch_dev_ce': p_dev['ce'],
        'pitch_correct': p_dev['top1'],
        'dur_dev_top1': d_dev['top1_acc'],
        'dur_dev_top3': d_dev['top3_acc'],
        'dur_dev_ce': d_dev['ce'],
        'dur_correct': d_dev['top1'],
        'avg_top1': score,
        'avg_ce': avg_ce,
        'pitch_n': p_dev['n'], 'dur_n': d_dev['n'],
        'elapsed_s': time.monotonic() - jt_t0,
    }
    jt_metrics.append(row)

    if improved:
        best_jt_score = score
        best_jt_ce    = avg_ce
        best_jt_epoch = epoch
        jt_bad = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': {k: v.cpu()
                                 for k, v in joint_model.state_dict().items()},
            'optimizer_state_dict': joint_optimizer.state_dict(),
            'best_avg_top1': best_jt_score,
            'metrics': jt_metrics,
            'n_pitch_classes': N_CLASSES,
            'n_dur_classes': N_DUR_CLASSES,
            'pitch_classes': pitch_classes,
            'dur_label_map': dur_label_map,
        }, JOINT_RUN / 'best_checkpoint.pt')
    else:
        jt_bad += 1

    with open(JOINT_RUN / 'metrics.json', 'w') as f:
        json.dump(jt_metrics, f, indent=2)

    mark = ' *best*' if improved else ''
    print(f"  Ep {epoch}: P={p_dev['top1']}/{p_dev['n']}({p_dev['top1_acc']:.2%}) "
          f"D={d_dev['top1']}/{d_dev['n']}({d_dev['top1_acc']:.2%}) "
          f"avg={score:.4f} "
          f"pCE={p_dev['ce']:.3f} dCE={d_dev['ce']:.3f} "
          f"t={row['elapsed_s']:.0f}s{mark}")

    if jt_bad >= 2:
        print("  Early stop."); break

# ── Restore best R1 ─────────────────────────────────────────
_ckpt = torch.load(JOINT_RUN / 'best_checkpoint.pt',
                    map_location='cpu', weights_only=False)
joint_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
joint_model.to(DEVICE); joint_model.eval()
del _ckpt

jt_r1_p, jt_r1_d = joint_eval(joint_model, jt_pitch_dev, jt_dur_dev)
with open(JOINT_RUN / 'final_dev_metrics.json', 'w') as f:
    json.dump({'best_epoch': best_jt_epoch,
               'pitch': jt_r1_p, 'dur': jt_r1_d}, f, indent=2)

jt_r1_elapsed = time.monotonic() - jt_t0
print(f"\nR1 best epoch {best_jt_epoch}:")
print(f"  Pitch: {jt_r1_p['top1']}/{jt_r1_p['n']} = {jt_r1_p['top1_acc']:.2%}  "
      f"top-5={jt_r1_p['top5_acc']:.2%}  CE={jt_r1_p['ce']:.3f}")
print(f"  Dur:   {jt_r1_d['top1']}/{jt_r1_d['n']} = {jt_r1_d['top1_acc']:.2%}  "
      f"top-3={jt_r1_d['top3_acc']:.2%}  CE={jt_r1_d['ce']:.3f}")
print(f"  Avg top-1: {(jt_r1_p['top1_acc']+jt_r1_d['top1_acc'])/2:.4f}")
print(f"  Time: {jt_r1_elapsed:.0f}s")

In [ ]:
# ── Joint Round 2 (conditional) ──────────────────────────────────
# Trigger if pitch OR duration dev top-1 dropped >3pp vs single-task baseline.
# One modification only, based on which task regressed more.

PITCH_BASELINE = 0.6253   # 312/499
DUR_BASELINE   = 0.8697   # 434/499

p_delta = jt_r1_p['top1_acc'] - PITCH_BASELINE
d_delta = jt_r1_d['top1_acc'] - DUR_BASELINE
p_regressed = p_delta < -0.03
d_regressed = d_delta < -0.03
need_jt_r2  = p_regressed or d_regressed
jt_budget_left = JT_BUDGET - jt_r1_elapsed

# Initialise R2 variables
jt_r2_p = jt_r2_d = None
JOINT_RUN_R2 = None
jt_metrics_r2 = []
best_jt_r2_epoch = 0

if need_jt_r2 and jt_budget_left > 30:
    # Decide modification
    if p_regressed and not d_regressed:
        mod_desc = "Pitch regressed more → upweight pitch loss 2:1"
        p_weight, d_weight = 2.0, 1.0
        bb_lr, head_lr = JOINT_LR, JOINT_LR
    elif d_regressed and not p_regressed:
        mod_desc = "Duration regressed more → upweight duration loss 2:1"
        p_weight, d_weight = 1.0, 2.0
        bb_lr, head_lr = JOINT_LR, JOINT_LR
    else:
        mod_desc = "Both regressed → lower backbone LR to 2e-5, heads stay 1e-4"
        p_weight, d_weight = 1.0, 1.0
        bb_lr, head_lr = 2e-5, 1e-4

    print(f"R2 triggered: pitch Δ={p_delta:+.2%}, dur Δ={d_delta:+.2%}")
    print(f"  Modification: {mod_desc}")

    # Reload R1 best
    _ckpt = torch.load(JOINT_RUN / 'best_checkpoint.pt',
                        map_location='cpu', weights_only=False)
    joint_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
    joint_model.to(DEVICE)
    del _ckpt

    if bb_lr != head_lr:
        jt_opt_r2 = torch.optim.Adam([
            {'params': [p for li in (10, 11)
                        for p in joint_model.backbone.note_bert1.layers[li].parameters()
                        if p.requires_grad], 'lr': bb_lr},
            {'params': list(joint_model.pitch_head.parameters()) +
                       list(joint_model.dur_head.parameters()), 'lr': head_lr},
        ])
    else:
        jt_opt_r2 = torch.optim.Adam(
            [p for p in joint_model.parameters() if p.requires_grad], lr=JOINT_LR)

    p_loss_fn_r2 = nn.CrossEntropyLoss()
    d_loss_fn_r2 = nn.CrossEntropyLoss()

    JOINT_RUN_R2 = ML_DIR / 'data' / 'runs' / (
        'm2bert_joint_r2_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    JOINT_RUN_R2.mkdir(parents=True, exist_ok=True)
    with open(JOINT_RUN_R2 / 'config.json', 'w') as f:
        json.dump({'round': 2, 'modification': mod_desc,
                   'p_weight': p_weight, 'd_weight': d_weight,
                   'backbone_lr': bb_lr, 'head_lr': head_lr,
                   'source': str(JOINT_RUN)}, f, indent=2)

    best_r2_score = best_jt_score
    best_r2_ce    = best_jt_ce
    bad_r2 = 0
    t_r2 = time.monotonic()
    print(f"  Output: {JOINT_RUN_R2}")

    for epoch in range(1, 7):
        if time.monotonic() - jt_t0 >= JT_BUDGET:
            print("  Budget reached."); break

        joint_model.train()
        p_ls = p_ns = d_ls = d_ns = 0

        for p_batch, d_batch in zip_longest(jt_pitch_loader, jt_dur_loader):
            if time.monotonic() - jt_t0 >= JT_BUDGET:
                break
            if p_batch is not None:
                cp_ids, err_idx, tgt, attn = (
                    x.to(DEVICE) for x in p_batch)
                jt_opt_r2.zero_grad(set_to_none=True)
                logits = joint_model(cp_ids, err_idx, attn, task='pitch')
                loss = p_loss_fn_r2(logits, tgt) * p_weight
                loss.backward(); jt_opt_r2.step()
                p_ls += loss.item() / p_weight * tgt.size(0)
                p_ns += tgt.size(0)
            if time.monotonic() - jt_t0 >= JT_BUDGET:
                break
            if d_batch is not None:
                cp_ids, err_idx, tgt, attn = (
                    x.to(DEVICE) for x in d_batch)
                jt_opt_r2.zero_grad(set_to_none=True)
                logits = joint_model(cp_ids, err_idx, attn, task='dur')
                loss = d_loss_fn_r2(logits, tgt) * d_weight
                loss.backward(); jt_opt_r2.step()
                d_ls += loss.item() / d_weight * tgt.size(0)
                d_ns += tgt.size(0)

        if p_ns == 0 and d_ns == 0:
            break

        p_dev, d_dev = joint_eval(joint_model, jt_pitch_dev, jt_dur_dev)
        score = (p_dev['top1_acc'] + d_dev['top1_acc']) / 2
        avg_ce = (p_dev['ce'] + d_dev['ce']) / 2
        improved = (score > best_r2_score or
                    (score == best_r2_score and avg_ce < best_r2_ce))

        row = {
            'epoch': epoch,
            'pitch_train_loss': p_ls / max(p_ns, 1),
            'dur_train_loss': d_ls / max(d_ns, 1),
            'pitch_dev_top1': p_dev['top1_acc'],
            'pitch_dev_top5': p_dev.get('top5_acc', 0),
            'pitch_dev_ce': p_dev['ce'],
            'pitch_correct': p_dev['top1'],
            'dur_dev_top1': d_dev['top1_acc'],
            'dur_dev_top3': d_dev.get('top3_acc', 0),
            'dur_dev_ce': d_dev['ce'],
            'dur_correct': d_dev['top1'],
            'avg_top1': score, 'avg_ce': avg_ce,
            'pitch_n': p_dev['n'], 'dur_n': d_dev['n'],
            'elapsed_s': time.monotonic() - t_r2,
        }
        jt_metrics_r2.append(row)

        if improved:
            best_r2_score = score; best_r2_ce = avg_ce
            best_jt_r2_epoch = epoch; bad_r2 = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': {k: v.cpu()
                                     for k, v in joint_model.state_dict().items()},
                'optimizer_state_dict': jt_opt_r2.state_dict(),
                'best_avg_top1': best_r2_score,
                'metrics': jt_metrics_r2,
                'n_pitch_classes': N_CLASSES,
                'n_dur_classes': N_DUR_CLASSES,
                'pitch_classes': pitch_classes,
                'dur_label_map': dur_label_map,
                'round': 2, 'modification': mod_desc,
            }, JOINT_RUN_R2 / 'best_checkpoint.pt')
        else:
            bad_r2 += 1

        with open(JOINT_RUN_R2 / 'metrics.json', 'w') as f:
            json.dump(jt_metrics_r2, f, indent=2)

        mark = ' *best*' if improved else ''
        print(f"  R2 Ep {epoch}: P={p_dev['top1']}/{p_dev['n']}({p_dev['top1_acc']:.2%}) "
              f"D={d_dev['top1']}/{d_dev['n']}({d_dev['top1_acc']:.2%}) "
              f"avg={score:.4f}{mark}")

        if bad_r2 >= 2:
            print("  Early stop."); break

    # Restore R2 best
    if JOINT_RUN_R2 and (JOINT_RUN_R2 / 'best_checkpoint.pt').exists():
        _ckpt = torch.load(JOINT_RUN_R2 / 'best_checkpoint.pt',
                            map_location='cpu', weights_only=False)
        joint_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
        joint_model.to(DEVICE); joint_model.eval()
        del _ckpt
        jt_r2_p, jt_r2_d = joint_eval(joint_model, jt_pitch_dev, jt_dur_dev)
        with open(JOINT_RUN_R2 / 'final_dev_metrics.json', 'w') as f:
            json.dump({'best_epoch': best_jt_r2_epoch,
                       'pitch': jt_r2_p, 'dur': jt_r2_d}, f, indent=2)
        print(f"\n  R2 best ep {best_jt_r2_epoch}: "
              f"P={jt_r2_p['top1']}/{jt_r2_p['n']}({jt_r2_p['top1_acc']:.2%})  "
              f"D={jt_r2_d['top1']}/{jt_r2_d['n']}({jt_r2_d['top1_acc']:.2%})")

elif need_jt_r2:
    print(f"R2 needed but budget exhausted ({jt_budget_left:.0f}s left).")
else:
    print(f"R2 not needed: pitch Δ={p_delta:+.2%}, dur Δ={d_delta:+.2%} "
          f"(threshold: -3pp)")

In [ ]:
# ── Joint Results, Demo & Summary ────────────────────────────────

# Select best run (R2 if better avg, else R1)
if jt_r2_p is not None:
    r2_avg = (jt_r2_p['top1_acc'] + jt_r2_d['top1_acc']) / 2
    r1_avg = (jt_r1_p['top1_acc'] + jt_r1_d['top1_acc']) / 2
    if r2_avg > r1_avg:
        jt_best_p, jt_best_d = jt_r2_p, jt_r2_d
        jt_best_src = 'R2'
        jt_best_run = JOINT_RUN_R2
    else:
        jt_best_p, jt_best_d = jt_r1_p, jt_r1_d
        jt_best_src = 'R1'
        jt_best_run = JOINT_RUN
else:
    jt_best_p, jt_best_d = jt_r1_p, jt_r1_d
    jt_best_src = 'R1'
    jt_best_run = JOINT_RUN

# ── Comparison table ─────────────────────────────────────────
print(f"{'='*80}")
print(f"  JOINT vs SINGLE-TASK COMPARISON")
print(f"{'='*80}")
print(f"  {'':30s} {'Pitch':>20s}   {'Duration':>20s}")
print(f"  {'':30s} {'top-1':>8s} {'top-5':>8s}   {'top-1':>8s} {'top-3':>8s}")
print(f"  {'─'*76}")

# Single-task baselines
print(f"  {'Single-task pitch':<30s} "
      f"{'312/499':>8s} {'427/499':>8s}   {'—':>8s} {'—':>8s}")
print(f"  {'Single-task duration':<30s} "
      f"{'—':>8s} {'—':>8s}   {'434/499':>8s} {'481/499':>8s}")

# Joint R1
r1p, r1d = jt_r1_p, jt_r1_d
p_t5_key = 'top5' if 'top5' in r1p else 'top5_acc'
d_t3_key = 'top3' if 'top3' in r1d else 'top3_acc'
r1p_t5 = r1p.get('top5', int(r1p.get('top5_acc', 0) * r1p['n']))
r1d_t3 = r1d.get('top3', int(r1d.get('top3_acc', 0) * r1d['n']))
print(f"  {'Joint R1 (ep '+str(best_jt_epoch)+')':<30s} "
      f"{r1p['top1']:>4d}/499  {r1p_t5:>4d}/499   "
      f"{r1d['top1']:>4d}/499  {r1d_t3:>4d}/499")

# Joint R2
if jt_r2_p is not None:
    r2p_t5 = jt_r2_p.get('top5', int(jt_r2_p.get('top5_acc', 0) * jt_r2_p['n']))
    r2d_t3 = jt_r2_d.get('top3', int(jt_r2_d.get('top3_acc', 0) * jt_r2_d['n']))
    print(f"  {'Joint R2 (ep '+str(best_jt_r2_epoch)+')':<30s} "
          f"{jt_r2_p['top1']:>4d}/499  {r2p_t5:>4d}/499   "
          f"{jt_r2_d['top1']:>4d}/499  {r2d_t3:>4d}/499")

print(f"  {'─'*76}")

# Delta vs baselines
bp = jt_best_p; bd = jt_best_d
p_d1 = bp['top1'] - 312; d_d1 = bd['top1'] - 434
print(f"  {'Selected: '+jt_best_src:<30s} "
      f"{bp['top1']:>4d}/499  "
      f"{bp.get('top5', int(bp.get('top5_acc',0)*bp['n'])):>4d}/499   "
      f"{bd['top1']:>4d}/499  "
      f"{bd.get('top3', int(bd.get('top3_acc',0)*bd['n'])):>4d}/499")
print(f"  {'Δ vs single-task':<30s} "
      f"{'%+d' % p_d1:>8s} {'':>8s}   {'%+d' % d_d1:>8s}")
print(f"{'='*80}")

# ── Training curves ──────────────────────────────────────────
print(f"\nR1 training curve:")
for r in jt_metrics:
    print(f"  Ep {r['epoch']}: pLoss={r['pitch_train_loss']:.4f} "
          f"dLoss={r['dur_train_loss']:.4f}  "
          f"P={r['pitch_correct']}/499({r['pitch_dev_top1']:.2%})  "
          f"D={r['dur_correct']}/499({r['dur_dev_top1']:.2%})  "
          f"avg={r['avg_top1']:.4f}  t={r['elapsed_s']:.0f}s")
if jt_metrics_r2:
    print(f"\nR2 training curve:")
    for r in jt_metrics_r2:
        print(f"  Ep {r['epoch']}: pLoss={r['pitch_train_loss']:.4f} "
              f"dLoss={r['dur_train_loss']:.4f}  "
              f"P={r['pitch_correct']}/499({r['pitch_dev_top1']:.2%})  "
              f"D={r['dur_correct']}/499({r['dur_dev_top1']:.2%})  "
              f"avg={r['avg_top1']:.4f}")

# ── Save dev predictions ─────────────────────────────────────
joint_model.eval()
for task, loader, id2label, fname in [
    ('pitch', jt_pitch_dev, id2pitch, 'pitch_dev_predictions.json'),
    ('dur',   jt_dur_dev,   dur_id2label, 'dur_dev_predictions.json'),
]:
    preds = []
    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
            logits = joint_model(cp_ids, err_idx, attn, task=task)
            pred = logits.argmax(1).cpu()
            for j in range(tgt.size(0)):
                preds.append({
                    'target': id2label[tgt[j].item()],
                    'predicted': id2label[pred[j].item()],
                    'correct': bool(pred[j] == tgt[j]),
                })
    with open(jt_best_run / fname, 'w') as f:
        json.dump(preds, f, indent=2)

# ── 6-sample demo (3 pitch + 3 duration) ────────────────────
print(f"\n{'='*90}")
print(f"  6-SAMPLE DEMO (3 pitch + 3 duration, seed=42)")
print(f"{'='*90}")

# Pitch demo
rng_jp = random.Random(42)
jp_idx = sorted(rng_jp.sample(range(len(dev_cp)), 3))
jp_ex  = [dev_cp[i] for i in jp_idx]
jp_raw = [dev_raw[i] for i in jp_idx]

jp_loader = DataLoader(CPDataset(jp_ex), batch_size=3,
                       shuffle=False, collate_fn=collate_cp)
cp_ids, err_idx, tgt, attn = next(iter(jp_loader))
with torch.inference_mode():
    logits = joint_model(cp_ids.to(DEVICE), err_idx.to(DEVICE),
                         attn.to(DEVICE), task='pitch')
    topk = logits.softmax(-1).cpu().topk(5, dim=-1)

print(f"\n  PITCH samples:")
print(f"  {'Idx':>4} {'Sample':>28}  {'Error':>5} {'Truth':>5} {'Pred':>5} {'OK':>3}  Top-5")
print(f"  {'─'*85}")
for b in range(3):
    pred_id = topk.indices[b][0].item()
    ok = ' ok' if pred_id == tgt[b].item() else '  X'
    cands = [id2pitch[topk.indices[b][k].item()] for k in range(5)]
    r = jp_raw[b]
    print(f"  {jp_idx[b]:4d} {r['sample_id']:>28}  "
          f"{r['replacement_token']:>5} {r['target_token']:>5} "
          f"{id2pitch[pred_id]:>5}{ok}  {cands}")

# Duration demo
rng_jd = random.Random(42)
jd_idx = sorted(rng_jd.sample(range(len(dur_dev_cp)), 3))
jd_ex  = [dur_dev_cp[i] for i in jd_idx]
jd_det = [dur_dev_details[i] for i in jd_idx]

jd_loader = DataLoader(CPDataset(jd_ex), batch_size=3,
                       shuffle=False, collate_fn=collate_cp)
cp_ids, err_idx, tgt, attn = next(iter(jd_loader))
with torch.inference_mode():
    logits = joint_model(cp_ids.to(DEVICE), err_idx.to(DEVICE),
                         attn.to(DEVICE), task='dur')
    topk = logits.softmax(-1).cpu().topk(min(5, N_DUR_CLASSES), dim=-1)

print(f"\n  DURATION samples:")
print(f"  {'Idx':>4} {'Sample':>28}  {'Error':>8} {'Truth':>8} {'Pred':>8} {'OK':>3}  Top candidates")
print(f"  {'─'*90}")
for b in range(3):
    pred_id = topk.indices[b][0].item()
    ok = ' ok' if pred_id == tgt[b].item() else '  X'
    cands = [dur_id2label[topk.indices[b][k].item()]
             for k in range(min(5, N_DUR_CLASSES))]
    d = jd_det[b]
    print(f"  {jd_idx[b]:4d} {d['sample_id']:>28}  "
          f"{d['repl_ql']:>8} {d['true_ql']:>8} "
          f"{dur_id2label[pred_id]:>8}{ok}  {cands}")

# ── Summary ──────────────────────────────────────────────────
total_jt_s = time.monotonic() - jt_t0
r2_ran = jt_r2_p is not None

print(f"""
{'='*80}
JOINT CORRECTION — SUMMARY
{'='*80}
Model:    Shared M2BERT backbone + pitch head (52) + duration head (28).
          Frozen layers 0-9; trainable layers 10-11 + both heads.
Training: 1:1 interleaved pitch/duration batches, each with mean CE loss.
          Checkpoint by mean(pitch_dev_top1, dur_dev_top1).

                    Pitch top-1      Duration top-1
  Single-task:      312/499 (62.53%)   434/499 (86.97%)
  Joint {jt_best_src}:       {bp['top1']}/499 ({bp['top1_acc']:.2%})   {bd['top1']}/499 ({bd['top1_acc']:.2%})
  Delta:            {'+' if p_d1>=0 else ''}{p_d1}               {'+' if d_d1>=0 else ''}{d_d1}

R2: {'Run — ' + (jt_metrics_r2[0].get('modification','') if jt_metrics_r2 else '') if r2_ran else 'Not triggered.'}
Time: {total_jt_s:.0f}s total.
Run:  {jt_best_run}

Next step: The joint model shares backbone representations across tasks.
  If one task's accuracy dropped, the next action is to try task-specific
  adapter layers (e.g., LoRA on layers 10-11 per task) so that the shared
  backbone provides a common foundation while each task retains its own
  fine-tuning capacity. This avoids the interference seen in uniform
  shared fine-tuning while preserving the parameter-efficiency benefit.
{'='*80}
""")

## Unified Correction Demo — Joint Model → MusicXML → MIDI

Use the joint checkpoint to correct pitch or duration errors in MusicXML,
export MIDI, and verify outputs for the 6 demo samples (3 pitch + 3 duration).

In [ ]:
# ── Unified Correction Demo: Joint Model → MusicXML → MIDI ────────
# Requires: all prior cells executed (parse_lmx_notes, dev_raw,
# dur_dev_cp, dur_dev_details, _lmx_path, _xml_path, make_edited_xml,
# set_note_pitch, musicxml_to_midi, find_error_note_index, etc.)

import hashlib as _hl
from copy import deepcopy
from fractions import Fraction as _Frac

# ── Part A: Load joint checkpoint + prediction caches ─────────────

JOINT_RUN_DIR = ML_DIR / 'data' / 'runs' / 'm2bert_joint_20261030_213214'
UNIFIED_OUT   = ML_DIR / 'demo' / 'unified_joint'
UNIFIED_OUT.mkdir(parents=True, exist_ok=True)

# Prediction caches (avoid re-inference)
with open(JOINT_RUN_DIR / 'pitch_dev_predictions.json') as f:
    pitch_preds = json.load(f)
with open(JOINT_RUN_DIR / 'dur_dev_predictions.json') as f:
    dur_preds = json.load(f)

# Label maps
with open(JOINT_RUN_DIR / 'pitch_label_map.json') as f:
    pitch_lm = json.load(f)
with open(JOINT_RUN_DIR / 'dur_label_map.json') as f:
    dur_lm = json.load(f)
id2pitch_j = {v: k for k, v in pitch_lm.items()}
id2dur_j   = {v: k for k, v in dur_lm.items()}

print(f"Loaded predictions: pitch={len(pitch_preds)}, dur={len(dur_preds)}")
print(f"Label maps: pitch={len(pitch_lm)}, dur={len(dur_lm)}")

# Demo indices (same as cell 31)
DEMO_INDICES = [12, 57, 327]

# ── Part B: Duration MusicXML editing utilities ───────────────────

# Standard quarterLength → MusicXML type name
_QL_TO_TYPE = {
    _Frac(8):    'breve',
    _Frac(4):    'whole',
    _Frac(2):    'half',
    _Frac(1):    'quarter',
    _Frac(1,2):  'eighth',
    _Frac(1,4):  '16th',
    _Frac(1,8):  '32nd',
    _Frac(1,16): '64th',
    _Frac(1,32): '128th',
}

def ql_to_musicxml_type(ql_frac):
    """Map quarterLength (Fraction) to (type_name, n_dots, time_mod_or_None).
    
    Returns:
        type_name: MusicXML <type> string
        n_dots: number of dot elements
        time_mod: (actual_notes, normal_notes) or None
    """
    # Direct match
    if ql_frac in _QL_TO_TYPE:
        return _QL_TO_TYPE[ql_frac], 0, None
    
    # Dotted: ql = base * (2 - 1/2^n) for n=1,2,3
    for n_dots in (1, 2, 3):
        factor = 2 - _Frac(1, 2**n_dots)
        if factor != 0:
            base = ql_frac / factor
            if base in _QL_TO_TYPE:
                return _QL_TO_TYPE[base], n_dots, None
    
    # Tuplet: common ratios (actual:normal)
    for actual, normal in [(3,2), (5,4), (6,4), (7,4), (3,4)]:
        scaled = ql_frac * _Frac(actual, normal)
        if scaled in _QL_TO_TYPE:
            return _QL_TO_TYPE[scaled], 0, (actual, normal)
        # Dotted tuplet
        for nd in (1, 2):
            factor = 2 - _Frac(1, 2**nd)
            base = scaled / factor
            if base in _QL_TO_TYPE:
                return _QL_TO_TYPE[base], nd, (actual, normal)
    
    # Fallback: closest standard type
    best_type, best_dist = 'quarter', abs(ql_frac - 1)
    for ql_std, tname in _QL_TO_TYPE.items():
        d = abs(ql_frac - ql_std)
        if d < best_dist:
            best_dist, best_type = d, tname
    return best_type, 0, None


def set_note_duration(note_el, ql_str, divisions, ns=""):
    """Rewrite duration-related elements of a <note>.
    
    Args:
        note_el: lxml <note> element
        ql_str: predicted quarterLength as string (e.g. "1/2", "2")
        divisions: current XML divisions per quarter note
        ns: XML namespace
    
    Returns:
        list of warning strings (empty if clean)
    """
    t = lambda name: _xml_tag(name, ns)
    warnings = []
    
    ql_frac = _Frac(ql_str)
    xml_dur = int(ql_frac * divisions)
    
    # Update <duration>
    dur_el = note_el.find(t("duration"))
    if dur_el is not None:
        dur_el.text = str(xml_dur)
    
    # Compute type info
    type_name, n_dots, time_mod = ql_to_musicxml_type(ql_frac)
    
    # Check if mapping is approximate
    if time_mod is None and n_dots == 0 and ql_frac not in _QL_TO_TYPE:
        warnings.append(f"ql={ql_str} has no exact MusicXML type; using closest: {type_name}")
    
    # Update <type>
    type_el = note_el.find(t("type"))
    if type_el is not None:
        type_el.text = type_name
    
    # Handle <dot> elements: remove existing, add new
    for dot_el in note_el.findall(t("dot")):
        note_el.remove(dot_el)
    if n_dots > 0 and type_el is not None:
        # Insert dots after <type>
        insert_pos = list(note_el).index(type_el) + 1
        for _ in range(n_dots):
            dot = etree.SubElement(note_el, t("dot"))
            note_el.remove(dot)
            note_el.insert(insert_pos, dot)
            insert_pos += 1
    
    # Handle <time-modification>
    tm_el = note_el.find(t("time-modification"))
    if time_mod is not None:
        actual, normal = time_mod
        if tm_el is None:
            tm_el = etree.SubElement(note_el, t("time-modification"))
            # Position after type/dots
            note_el.remove(tm_el)
            ref = type_el if type_el is not None else dur_el
            if ref is not None:
                idx = list(note_el).index(ref) + 1 + n_dots
                note_el.insert(idx, tm_el)
        # Set or create sub-elements
        an_el = tm_el.find(t("actual-notes"))
        if an_el is None:
            an_el = etree.SubElement(tm_el, t("actual-notes"))
        an_el.text = str(actual)
        nn_el = tm_el.find(t("normal-notes"))
        if nn_el is None:
            nn_el = etree.SubElement(tm_el, t("normal-notes"))
        nn_el.text = str(normal)
    elif tm_el is not None:
        note_el.remove(tm_el)
    
    return warnings


def make_dur_edited_xml(xml_path, note_index, ql_str):
    """Return (etree, warnings) with one note's duration changed."""
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    
    # Get divisions from first <attributes>
    divisions = 1
    attr = root.find(f".//{t('attributes')}")
    if attr is not None:
        d = attr.find(t("divisions"))
        if d is not None:
            divisions = int(d.text)
    
    notes = root.findall(f".//{t('note')}")
    warns = set_note_duration(notes[note_index], ql_str, divisions, ns)
    return tree, warns

print("Duration MusicXML utilities defined")

# ── Part C: Find duration error note index in clean XML ───────────

def _find_dur_note_index_clean(sid):
    """Replicate the deterministic hash selection from build_dur_split
    to find which note index (in clean XML note order) was selected."""
    lmx_p = _lmx_path(sid)
    clean_toks = lmx_p.read_text().split()
    notes, _ = parse_lmx_notes(clean_toks, sid)
    
    cands = []
    for ni, n in enumerate(notes):
        if n['is_grace'] or n['is_rest']:
            continue
        if n['duration'] is None or n['pitch'] is None:
            continue
        if _find_dur_span(clean_toks, n)[0] is None:
            continue
        cands.append(ni)
    
    h = int(_hl.md5(sid.encode()).hexdigest(), 16)
    sel = cands[h % len(cands)]
    return sel

# ── Part D: Unified correction entry point ────────────────────────

def correct_sample(error_type, dev_idx, output_dir):
    """Run full correction pipeline for one sample.
    
    Returns dict with file paths, verification results, warnings.
    """
    result = {'error_type': error_type, 'dev_idx': dev_idx, 'warnings': []}
    
    if error_type == 'pitch':
        ex = dev_raw[dev_idx]
        sid = ex['sample_id']
        note_idx = find_error_note_index(ex, parse_lmx_notes)
        error_val = ex['replacement_token']
        truth_val = ex['target_token']
        pred = pitch_preds[dev_idx]
        pred_val = pred['predicted']
        correct = pred['correct']
        
        xml_src = _xml_path(sid)
        sdir = output_dir / f"pitch_{dev_idx}_{sid.replace('/', '_')}"
        sdir.mkdir(parents=True, exist_ok=True)
        
        # Ground truth
        gt_tree = etree.parse(str(xml_src))
        gt_xml = sdir / "ground_truth.musicxml"
        gt_tree.write(str(gt_xml), xml_declaration=True, encoding="UTF-8")
        gt_midi = sdir / "ground_truth.mid"
        musicxml_to_midi(gt_tree, gt_midi)
        
        # Corrupted
        corr_tree = make_edited_xml(xml_src, note_idx, error_val)
        corr_xml = sdir / "corrupted.musicxml"
        corr_tree.write(str(corr_xml), xml_declaration=True, encoding="UTF-8")
        corr_midi = sdir / "corrupted.mid"
        musicxml_to_midi(corr_tree, corr_midi)
        
        # Model corrected
        pred_tree = make_edited_xml(xml_src, note_idx, pred_val)
        pred_xml = sdir / "model_corrected.musicxml"
        pred_tree.write(str(pred_xml), xml_declaration=True, encoding="UTF-8")
        pred_midi = sdir / "model_corrected.mid"
        musicxml_to_midi(pred_tree, pred_midi)
        
        # Verify XML
        v_tree = etree.parse(str(pred_xml))
        v_ns = v_tree.getroot().nsmap.get(None, "")
        v_t = lambda name: _xml_tag(name, v_ns)
        v_notes = v_tree.getroot().findall(f".//{v_t('note')}")
        v_note = v_notes[note_idx]
        v_pitch = v_note.find(v_t("pitch"))
        if v_pitch is not None:
            v_step = v_pitch.find(v_t("step")).text
            v_oct = v_pitch.find(v_t("octave")).text
            xml_verified = (v_step + v_oct) == pred_val
        else:
            xml_verified = False
            result['warnings'].append("No pitch element in verified note")
        
        # Verify MIDI
        mid = mido.MidiFile(str(pred_midi))
        midi_notes = []
        tick = 0
        for msg in mid.tracks[0]:
            tick += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                midi_notes.append((tick, msg.note))
        midi_verified = len(midi_notes) > 0  # basic check: MIDI has notes
        
    elif error_type == 'duration':
        det = dur_dev_details[dev_idx]
        sid = det['sample_id']
        note_idx = _find_dur_note_index_clean(sid)
        error_val = det['repl_ql']
        truth_val = det['true_ql']
        pred = dur_preds[dev_idx]
        pred_val = pred['predicted']  # QL in whole-note units from label map
        correct = pred['correct']
        
        # Duration labels are already in quarterLength units
        pred_ql = pred_val
        truth_ql = truth_val
        error_ql = error_val
        
        xml_src = _xml_path(sid)
        sdir = output_dir / f"dur_{dev_idx}_{sid.replace('/', '_')}"
        sdir.mkdir(parents=True, exist_ok=True)
        
        # Ground truth (original = correct duration)
        gt_tree = etree.parse(str(xml_src))
        gt_xml = sdir / "ground_truth.musicxml"
        gt_tree.write(str(gt_xml), xml_declaration=True, encoding="UTF-8")
        gt_midi = sdir / "ground_truth.mid"
        musicxml_to_midi(gt_tree, gt_midi)
        
        # Corrupted (apply error duration)
        corr_tree, corr_warns = make_dur_edited_xml(xml_src, note_idx, error_ql)
        corr_xml = sdir / "corrupted.musicxml"
        corr_tree.write(str(corr_xml), xml_declaration=True, encoding="UTF-8")
        corr_midi = sdir / "corrupted.mid"
        musicxml_to_midi(corr_tree, corr_midi)
        result['warnings'].extend(corr_warns)
        
        # Model corrected (apply predicted duration)
        pred_tree, pred_warns = make_dur_edited_xml(xml_src, note_idx, pred_ql)
        pred_xml = sdir / "model_corrected.musicxml"
        pred_tree.write(str(pred_xml), xml_declaration=True, encoding="UTF-8")
        pred_midi = sdir / "model_corrected.mid"
        musicxml_to_midi(pred_tree, pred_midi)
        result['warnings'].extend(pred_warns)
        
        # Verify XML: check <duration> value
        v_tree = etree.parse(str(pred_xml))
        v_ns = v_tree.getroot().nsmap.get(None, "")
        v_t = lambda name: _xml_tag(name, v_ns)
        v_attr = v_tree.getroot().find(f".//{v_t('attributes')}")
        v_div = 1
        if v_attr is not None:
            d_el = v_attr.find(v_t("divisions"))
            if d_el is not None:
                v_div = int(d_el.text)
        v_notes = v_tree.getroot().findall(f".//{v_t('note')}")
        v_dur_el = v_notes[note_idx].find(v_t("duration"))
        expected_dur = int(_Frac(pred_ql) * v_div)
        actual_dur = int(v_dur_el.text) if v_dur_el is not None else -1
        xml_verified = (actual_dur == expected_dur)
        if not xml_verified:
            result['warnings'].append(
                f"XML duration mismatch: expected {expected_dur}, got {actual_dur}")
        
        # Verify MIDI
        mid = mido.MidiFile(str(pred_midi))
        midi_notes = []
        tick = 0
        for msg in mid.tracks[0]:
            tick += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                midi_notes.append((tick, msg.note))
        midi_verified = len(midi_notes) > 0
        
        # Use QL values for display
        pred_val = pred_ql
        truth_val = truth_ql
        error_val = error_ql
    
    result.update({
        'sample_id': sid, 'note_idx': note_idx,
        'error_val': error_val, 'truth_val': truth_val,
        'pred_val': pred_val, 'correct': correct,
        'xml_verified': xml_verified, 'midi_verified': midi_verified,
        'dir': str(sdir),
    })
    return result

# ── Part E: Run 6 demos ──────────────────────────────────────────

print(f"\n{'='*90}")
print(f"  UNIFIED CORRECTION DEMO — 6 samples (3 pitch + 3 duration)")
print(f"{'='*90}")

all_results = []

# Pitch samples
print(f"\n  PITCH corrections:")
for idx in DEMO_INDICES:
    r = correct_sample('pitch', idx, UNIFIED_OUT)
    all_results.append(r)
    mark = "ok" if r['correct'] else "X"
    print(f"  [{idx:3d}] {r['sample_id']:>28s}  "
          f"err={r['error_val']:>4s} truth={r['truth_val']:>4s} "
          f"pred={r['pred_val']:>4s} [{mark}]  "
          f"xml={'ok' if r['xml_verified'] else 'FAIL'}  "
          f"midi={'ok' if r['midi_verified'] else 'FAIL'}"
          f"{'  ⚠ '+'; '.join(r['warnings']) if r['warnings'] else ''}")

# Duration samples
print(f"\n  DURATION corrections:")
for idx in DEMO_INDICES:
    r = correct_sample('duration', idx, UNIFIED_OUT)
    all_results.append(r)
    mark = "ok" if r['correct'] else "X"
    print(f"  [{idx:3d}] {r['sample_id']:>28s}  "
          f"err={r['error_val']:>8s} truth={r['truth_val']:>8s} "
          f"pred={r['pred_val']:>8s} [{mark}]  "
          f"xml={'ok' if r['xml_verified'] else 'FAIL'}  "
          f"midi={'ok' if r['midi_verified'] else 'FAIL'}"
          f"{'  ⚠ '+'; '.join(r['warnings']) if r['warnings'] else ''}")

# ── Part F: Summary table + output ───────────────────────────────

print(f"\n{'='*90}")
print(f"  {'Type':>8s} {'Idx':>4s} {'Sample':>28s} {'Note':>5s} "
      f"{'Error':>8s} {'Truth':>8s} {'Pred':>8s} {'OK':>3s} "
      f"{'XML':>4s} {'MIDI':>5s} {'Warnings'}")
print(f"  {'─'*88}")
for r in all_results:
    mark = "ok" if r['correct'] else "X"
    print(f"  {r['error_type']:>8s} {r['dev_idx']:4d} {r['sample_id']:>28s} "
          f"{r['note_idx']:5d} {r['error_val']:>8s} {r['truth_val']:>8s} "
          f"{r['pred_val']:>8s}  {mark:>2s} "
          f"{'ok' if r['xml_verified'] else 'FAIL':>4s} "
          f"{'ok' if r['midi_verified'] else 'FAIL':>5s} "
          f"{'; '.join(r['warnings']) if r['warnings'] else '—'}")

# Save summary
summary_out = []
for r in all_results:
    summary_out.append({
        'error_type': r['error_type'], 'dev_idx': r['dev_idx'],
        'sample_id': r['sample_id'], 'note_idx': r['note_idx'],
        'error_val': r['error_val'], 'truth_val': r['truth_val'],
        'pred_val': r['pred_val'], 'correct': r['correct'],
        'xml_verified': r['xml_verified'], 'midi_verified': r['midi_verified'],
        'warnings': r['warnings'], 'dir': r['dir'],
    })

with open(UNIFIED_OUT / 'demo_summary.json', 'w') as f:
    json.dump(summary_out, f, indent=2)

sample_ids_out = {r['error_type'] + '_' + str(r['dev_idx']): r['sample_id']
                  for r in all_results}
with open(UNIFIED_OUT / 'demo_sample_ids.json', 'w') as f:
    json.dump(sample_ids_out, f, indent=2)

n_correct = sum(1 for r in all_results if r['correct'])
n_xml_ok  = sum(1 for r in all_results if r['xml_verified'])
n_midi_ok = sum(1 for r in all_results if r['midi_verified'])
n_warns   = sum(len(r['warnings']) for r in all_results)

print(f"\n  Correct: {n_correct}/6  XML verified: {n_xml_ok}/6  "
      f"MIDI verified: {n_midi_ok}/6  Warnings: {n_warns}")
print(f"  Output: {UNIFIED_OUT.relative_to(ML_DIR)}/")
print(f"{'='*90}")

## Standalone Inference — `correct_musicxml` Interface

Self-contained inference pipeline: load joint checkpoint once, accept any
MusicXML file + target locator + error type, run actual model forward pass,
output corrected MusicXML + MIDI + prediction JSON.

**Minimal run order after kernel restart:** cells 1 → 2 → 35 → 36

In [ ]:
# ── Standalone Inference Pipeline ──────────────────────────────────
# Self-contained: only needs cells 1 (imports) and 2 (paths + CP vocab).
# Does NOT require training data, dev sets, or prediction caches.

import time as _time
from fractions import Fraction as _Frac
from lxml import etree
import mido

# ═══════════════════════════════════════════════════════════════════
# 1. Model class definitions (copied from cells 9/29 — self-contained)
# ═══════════════════════════════════════════════════════════════════

class _LUT(nn.Module):
    def __init__(self, n, d, pad=None):
        super().__init__()
        self.lut = nn.Embedding(n, d, padding_idx=pad)
    def forward(self, x): return self.lut(x)

class _MidiModernBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.word_emb = nn.ModuleList([
            _LUT(4,  256, pad=BAR_PAD),
            _LUT(18, 256, pad=POS_PAD),
            _LUT(88, 256, pad=PITCH_PAD),
            _LUT(66, 256, pad=DUR_PAD),
        ])
        self.in_linear = nn.Linear(1024, 768)
        _cfg = ModernBertConfig(
            vocab_size=10, hidden_size=768, num_hidden_layers=12,
            num_attention_heads=12, intermediate_size=1152,
            max_position_embeddings=4096, pad_token_id=0,
        )
        self.note_bert1 = ModernBertModel(_cfg)

    def forward(self, cp_ids, attention_mask=None):
        embs = torch.cat([self.word_emb[i](cp_ids[:, :, i])
                          for i in range(4)], dim=-1)
        embs = self.in_linear(embs)
        out  = self.note_bert1(inputs_embeds=embs, attention_mask=attention_mask)
        return out.last_hidden_state

class _JointModel(nn.Module):
    def __init__(self, backbone, n_pitch, n_dur):
        super().__init__()
        self.backbone   = backbone
        self.pitch_head = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_pitch))
        self.dur_head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_dur))

    def forward(self, cp_ids, error_note_idx, attention_mask=None, task='pitch'):
        hidden = self.backbone(cp_ids, attention_mask)
        err_h  = hidden[torch.arange(hidden.size(0)), error_note_idx]
        if task == 'pitch':
            return self.pitch_head(err_h)
        return self.dur_head(err_h)

# ═══════════════════════════════════════════════════════════════════
# 2. Load joint checkpoint (once)
# ═══════════════════════════════════════════════════════════════════

_JOINT_CKPT_PATH = ML_DIR / 'data' / 'runs' / 'm2bert_joint_20261030_213214' / 'best_checkpoint.pt'
_ckpt = torch.load(_JOINT_CKPT_PATH, map_location='cpu', weights_only=False)

_pitch_classes  = _ckpt['pitch_classes']       # {pitch_str: class_id}
_dur_label_map  = _ckpt['dur_label_map']       # {ql_whole_str: class_id}
_id2pitch       = {v: k for k, v in _pitch_classes.items()}
_id2dur         = {v: k for k, v in _dur_label_map.items()}
_N_PITCH        = _ckpt['n_pitch_classes']
_N_DUR          = _ckpt['n_dur_classes']

_backbone = _MidiModernBERT()
_pretrained_ckpt = torch.load(PRETRAINED / 'model_best.ckpt',
                              map_location='cpu', weights_only=False)
_backbone.load_state_dict(_pretrained_ckpt['state_dict'], strict=True)
del _pretrained_ckpt

_infer_model = _JointModel(_backbone, _N_PITCH, _N_DUR)
_infer_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
_infer_model.to(DEVICE)
_infer_model.eval()
del _ckpt

print(f"Joint model loaded: pitch={_N_PITCH} classes, dur={_N_DUR} classes, device={DEVICE}")

# ═══════════════════════════════════════════════════════════════════
# 3. MusicXML → note dicts (direct XML parse, no LMX dependency)
# ═══════════════════════════════════════════════════════════════════

_PITCH_RE = re.compile(r'^[A-G][0-9]$')
_SEMI = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}

def _xt(name, ns):
    return f"{{{ns}}}{name}" if ns else name

def _parse_xml_notes(xml_path, part_id=None):
    """Parse MusicXML into note dicts compatible with CP conversion.

    Returns list of dicts with keys: part_id, measure_index, note_in_measure,
    pitch (str like 'C4' or None), is_rest, is_grace, is_chord, duration
    (Fraction in whole-note units or None), onset (Fraction in whole-note
    units within measure), measure_num, voice, staff, xml_note_element.

    If part_id is given, only that part is parsed. Otherwise all parts.
    """
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    ns = root.nsmap.get(None, "")
    t = lambda name: _xt(name, ns)

    notes = []
    for part_el in root.findall(f".//{t('part')}"):
        pid = part_el.get('id')
        if part_id is not None and pid != part_id:
            continue

        divisions = 1  # XML divisions per quarter note
        measures = part_el.findall(t('measure'))

        for mi, meas in enumerate(measures):
            # Update divisions from <attributes>
            for attr in meas.findall(t('attributes')):
                d_el = attr.find(t('divisions'))
                if d_el is not None:
                    divisions = int(d_el.text)

            onset = _Frac(0)        # onset in whole-note units within measure
            prev_onset = _Frac(0)   # for chord notes
            ni_in_meas = 0

            for el in meas:
                ltag = el.tag.split('}')[-1] if '}' in el.tag else el.tag

                if ltag == 'note':
                    pitch_el = el.find(t('pitch'))
                    rest_el = el.find(t('rest'))
                    grace_el = el.find(t('grace'))
                    chord_el = el.find(t('chord'))
                    dur_el = el.find(t('duration'))

                    is_grace = grace_el is not None
                    is_chord = chord_el is not None
                    is_rest = rest_el is not None

                    # Pitch
                    pitch_str = None
                    if pitch_el is not None:
                        step = pitch_el.find(t('step')).text
                        octave = pitch_el.find(t('octave')).text
                        pitch_str = step + octave  # written pitch

                    # Duration in whole-note units
                    duration = None
                    xml_dur_val = 0
                    if dur_el is not None:
                        xml_dur_val = int(dur_el.text)
                        duration = _Frac(xml_dur_val, divisions * 4)

                    # Onset
                    note_onset = prev_onset if is_chord else onset
                    if not is_chord:
                        prev_onset = onset

                    # Voice / staff
                    voice_el = el.find(t('voice'))
                    staff_el = el.find(t('staff'))
                    voice = voice_el.text if voice_el is not None else None
                    staff = staff_el.text if staff_el is not None else None

                    notes.append({
                        'part_id': pid,
                        'measure_index': mi,
                        'note_in_measure': ni_in_meas,
                        'pitch': pitch_str,
                        'is_rest': is_rest,
                        'is_grace': is_grace,
                        'is_chord': is_chord,
                        'duration': duration,
                        'onset': note_onset,
                        'measure_num': mi + 1,
                        'voice': voice,
                        'staff': staff,
                        'xml_note_element': el,
                        'divisions': divisions,
                    })

                    ni_in_meas += 1

                    # Advance onset (non-chord, non-grace)
                    if not is_chord and not is_grace and dur_el is not None:
                        onset += _Frac(xml_dur_val, divisions * 4)

                elif ltag == 'backup':
                    bd = el.find(t('duration'))
                    if bd is not None:
                        onset -= _Frac(int(bd.text), divisions * 4)
                        onset = max(_Frac(0), onset)

                elif ltag == 'forward':
                    fd = el.find(t('duration'))
                    if fd is not None:
                        onset += _Frac(int(fd.text), divisions * 4)

    return notes, tree, ns

# ═══════════════════════════════════════════════════════════════════
# 4. Note dicts → CP tokens (matches training conversion exactly)
# ═══════════════════════════════════════════════════════════════════

def _pitch_to_cp(pitch_str):
    """Written pitch string → CP pitch index (matching cell 6 logic)."""
    if not _PITCH_RE.match(pitch_str):
        return None
    midi = (int(pitch_str[1]) + 1) * 12 + _SEMI[pitch_str[0]]
    idx = midi - 22
    return idx if 0 <= idx <= 85 else None

_MAX_CP_LEN = 1024  # context window upper bound

def _build_cp_for_inference(parsed_notes, target_flat_idx, error_type):
    """Build CP token sequence from parsed XML notes.

    Masks the target note's pitch or duration attribute.
    Returns (cp_ids, error_note_idx_in_cp) or raises ValueError.
    """
    cp_ids = []
    cp_target_idx = None
    prev_meas = -1

    for i, n in enumerate(parsed_notes):
        bar = 0 if n['measure_num'] != prev_meas else 1
        prev_meas = n['measure_num']

        pos = min(15, int(float(n['onset']) * 16))

        # Pitch
        if i == target_flat_idx and error_type == 'pitch':
            pitch = PITCH_MASK
        elif n['pitch'] and _PITCH_RE.match(n['pitch']):
            p = _pitch_to_cp(n['pitch'])
            pitch = p if p is not None else PITCH_PAD
        else:
            pitch = PITCH_PAD

        # Duration
        DUR_MASK = CP_TOK2ID['Duration']['Duration <MASK>']
        if i == target_flat_idx and error_type == 'duration':
            dur = DUR_MASK
        elif n['duration'] is not None:
            dur = min(63, int(float(n['duration']) * 32))
        else:
            dur = DUR_PAD

        cp_ids.append([bar, pos, pitch, dur])
        if i == target_flat_idx:
            cp_target_idx = len(cp_ids) - 1

    if cp_target_idx is None:
        raise ValueError("Target note not found in CP sequence")

    # Truncate to _MAX_CP_LEN centered on target
    if len(cp_ids) > _MAX_CP_LEN:
        half = _MAX_CP_LEN // 2
        start = max(0, cp_target_idx - half)
        end = start + _MAX_CP_LEN
        if end > len(cp_ids):
            end = len(cp_ids)
            start = end - _MAX_CP_LEN
        cp_ids = cp_ids[start:end]
        cp_target_idx -= start

    return cp_ids, cp_target_idx

# ═══════════════════════════════════════════════════════════════════
# 5. MusicXML editing + MIDI export (reused from cells 21/33)
# ═══════════════════════════════════════════════════════════════════

_STEP_SEMI = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}

_QL_TO_XML_TYPE = {
    _Frac(8):    'breve',   _Frac(4):    'whole',
    _Frac(2):    'half',    _Frac(1):    'quarter',
    _Frac(1,2):  'eighth',  _Frac(1,4):  '16th',
    _Frac(1,8):  '32nd',    _Frac(1,16): '64th',
    _Frac(1,32): '128th',
}

def _ql_to_xml_type(ql_frac):
    """Map quarterLength (Fraction) to (type_name, n_dots, time_mod_or_None)."""
    if ql_frac in _QL_TO_XML_TYPE:
        return _QL_TO_XML_TYPE[ql_frac], 0, None
    for nd in (1, 2, 3):
        factor = 2 - _Frac(1, 2**nd)
        base = ql_frac / factor
        if base in _QL_TO_XML_TYPE:
            return _QL_TO_XML_TYPE[base], nd, None
    for act, nrm in [(3,2), (5,4), (6,4), (7,4), (3,4)]:
        scaled = ql_frac * _Frac(act, nrm)
        if scaled in _QL_TO_XML_TYPE:
            return _QL_TO_XML_TYPE[scaled], 0, (act, nrm)
        for nd in (1, 2):
            factor = 2 - _Frac(1, 2**nd)
            base = scaled / factor
            if base in _QL_TO_XML_TYPE:
                return _QL_TO_XML_TYPE[base], nd, (act, nrm)
    best_type, best_dist = 'quarter', abs(ql_frac - 1)
    for ql_std, tname in _QL_TO_XML_TYPE.items():
        d = abs(ql_frac - ql_std)
        if d < best_dist:
            best_dist, best_type = d, tname
    return best_type, 0, None


def _apply_pitch(note_el, pitch_str, ns):
    """Rewrite <step>/<octave> of a <note> element."""
    t = lambda name: _xt(name, ns)
    pitch_el = note_el.find(t("pitch"))
    if pitch_el is None:
        return
    pitch_el.find(t("step")).text   = pitch_str[0]
    pitch_el.find(t("octave")).text = pitch_str[1:]
    alter = pitch_el.find(t("alter"))
    if alter is not None:
        pitch_el.remove(alter)
    acc = note_el.find(t("accidental"))
    if acc is not None:
        note_el.remove(acc)


def _apply_duration(note_el, ql_str, divisions, ns):
    """Rewrite duration elements of a <note>. Returns warnings list."""
    t = lambda name: _xt(name, ns)
    warnings = []
    ql_frac = _Frac(ql_str)
    xml_dur = int(ql_frac * divisions)

    dur_el = note_el.find(t("duration"))
    if dur_el is not None:
        dur_el.text = str(xml_dur)

    type_name, n_dots, time_mod = _ql_to_xml_type(ql_frac)
    if time_mod is None and n_dots == 0 and ql_frac not in _QL_TO_XML_TYPE:
        warnings.append(f"ql={ql_str} has no exact MusicXML type; closest: {type_name}")

    type_el = note_el.find(t("type"))
    if type_el is not None:
        type_el.text = type_name

    for dot_el in note_el.findall(t("dot")):
        note_el.remove(dot_el)
    if n_dots > 0 and type_el is not None:
        ins = list(note_el).index(type_el) + 1
        for _ in range(n_dots):
            dot = etree.SubElement(note_el, t("dot"))
            note_el.remove(dot)
            note_el.insert(ins, dot)
            ins += 1

    tm_el = note_el.find(t("time-modification"))
    if time_mod is not None:
        act, nrm = time_mod
        if tm_el is None:
            tm_el = etree.SubElement(note_el, t("time-modification"))
            note_el.remove(tm_el)
            ref = type_el if type_el is not None else dur_el
            if ref is not None:
                idx = list(note_el).index(ref) + 1 + n_dots
                note_el.insert(idx, tm_el)
        an = tm_el.find(t("actual-notes"))
        if an is None: an = etree.SubElement(tm_el, t("actual-notes"))
        an.text = str(act)
        nn = tm_el.find(t("normal-notes"))
        if nn is None: nn = etree.SubElement(tm_el, t("normal-notes"))
        nn.text = str(nrm)
    elif tm_el is not None:
        note_el.remove(tm_el)

    return warnings


def _xml_to_midi(xml_tree, midi_path, ns):
    """MusicXML tree → MIDI file. Returns note count."""
    root = xml_tree.getroot()
    t = lambda name: _xt(name, ns)
    TPQN = 480; divisions = 1
    events = []; current_time = 0; prev_onset = 0
    for measure in root.findall(f".//{t('measure')}"):
        for attr in measure.findall(t("attributes")):
            d = attr.find(t("divisions"))
            if d is not None: divisions = int(d.text)
        scale = TPQN / divisions
        for el in measure:
            ltag = el.tag.split("}")[-1] if "}" in el.tag else el.tag
            if ltag == "note":
                chord_el = el.find(t("chord")); pitch_el = el.find(t("pitch"))
                dur_el = el.find(t("duration"))
                dur = int(dur_el.text) if dur_el is not None else 0
                onset = prev_onset if chord_el is not None else current_time
                if chord_el is None: prev_onset = current_time
                if pitch_el is not None:
                    step = pitch_el.find(t("step")).text
                    octave = int(pitch_el.find(t("octave")).text)
                    alt_el = pitch_el.find(t("alter"))
                    alter = int(float(alt_el.text)) if alt_el is not None else 0
                    midi_n = (octave + 1) * 12 + _STEP_SEMI[step] + alter
                    events.append((int(onset * scale), midi_n,
                                   max(1, int(dur * scale))))
                if chord_el is None: current_time += dur
            elif ltag == "backup":
                current_time -= int(el.find(t("duration")).text)
            elif ltag == "forward":
                current_time += int(el.find(t("duration")).text)
    mid = mido.MidiFile(ticks_per_beat=TPQN)
    track = mido.MidiTrack(); mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=mido.bpm2tempo(120)))
    msgs = []
    for onset, pitch, dur in events:
        msgs.append((onset, 1, pitch, 80))
        msgs.append((onset + dur, 0, pitch, 0))
    msgs.sort(key=lambda m: (m[0], m[1]))
    prev_tick = 0
    for tick, on, pitch, vel in msgs:
        delta = tick - prev_tick
        mtype = "note_on" if on else "note_off"
        track.append(mido.Message(mtype, note=pitch, velocity=vel, time=delta))
        prev_tick = tick
    mid.save(str(midi_path))
    return len(events)

# ═══════════════════════════════════════════════════════════════════
# 6. Public API
# ═══════════════════════════════════════════════════════════════════

def list_notes(input_path):
    """Print all notes in a MusicXML file with their locator coordinates.

    Columns: flat_idx, part_id, measure_index, note_in_measure,
    voice, staff, pitch, duration(QL), flags.
    """
    notes, _, _ = _parse_xml_notes(input_path)
    print(f"{'flat':>5s} {'part':>5s} {'meas':>5s} {'note':>5s} "
          f"{'voice':>5s} {'staff':>5s} {'pitch':>6s} {'dur_QL':>8s} {'flags'}")
    print("-" * 65)
    for i, n in enumerate(notes):
        flags = []
        if n['is_rest']:  flags.append('rest')
        if n['is_grace']: flags.append('grace')
        if n['is_chord']: flags.append('chord')
        ql = str(n['duration'] * 4) if n['duration'] is not None else '?'
        print(f"{i:5d} {n['part_id']:>5s} {n['measure_index']:5d} "
              f"{n['note_in_measure']:5d} "
              f"{n['voice'] or '-':>5s} {n['staff'] or '-':>5s} "
              f"{n['pitch'] or 'rest':>6s} {ql:>8s} "
              f"{' '.join(flags)}")
    print(f"\nTotal: {len(notes)} notes")
    return notes


def correct_musicxml(input_path, target_locator, error_type,
                     output_dir, top_k=None):
    """Run inference on a MusicXML file and output corrected files.

    Args:
        input_path: path to .musicxml or .xml file
        target_locator: dict with 'part_id', 'measure_index', 'note_index'
            (measure_index and note_index are 0-based; note_index counts all
            <note> direct children including rests, grace, chord)
        error_type: 'pitch' or 'duration'
        output_dir: directory for output files
        top_k: number of candidates to return (default: 5 for pitch, 3 for dur)

    Returns:
        dict with target info, prediction, candidates, file paths, warnings
    """
    t0 = _time.monotonic()
    warnings = []
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if top_k is None:
        top_k = 5 if error_type == 'pitch' else 3

    pid = target_locator['part_id']
    mi  = target_locator['measure_index']
    ni  = target_locator['note_index']

    # ── Parse all notes in the target part ────────────────────
    parsed, tree, ns = _parse_xml_notes(input_path, part_id=pid)
    if not parsed:
        raise ValueError(f"No notes found in part {pid}")

    # Find the target note by locator
    target_flat = None
    for fi, n in enumerate(parsed):
        if (n['part_id'] == pid and n['measure_index'] == mi
                and n['note_in_measure'] == ni):
            target_flat = fi
            break
    if target_flat is None:
        raise ValueError(
            f"Note not found: part={pid}, measure={mi}, note={ni}")

    tn = parsed[target_flat]

    # Read current value from file
    if error_type == 'pitch':
        current_val = tn['pitch'] or 'rest'
    else:
        if tn['duration'] is not None:
            current_val = str(tn['duration'] * 4)  # QL string
        else:
            current_val = '?'

    # ── Build CP and run inference ────────────────────────────
    cp_ids, cp_err_idx = _build_cp_for_inference(
        parsed, target_flat, error_type)

    cp_t  = torch.tensor([cp_ids], dtype=torch.long, device=DEVICE)
    ei_t  = torch.tensor([cp_err_idx], dtype=torch.long, device=DEVICE)
    attn  = torch.ones(1, len(cp_ids), dtype=torch.long, device=DEVICE)

    with torch.inference_mode():
        logits = _infer_model(cp_t, ei_t, attn,
                              task='pitch' if error_type == 'pitch' else 'dur')
        probs = logits.softmax(-1).cpu()
        topk = probs.topk(min(top_k, probs.size(-1)), dim=-1)

    # Decode predictions
    if error_type == 'pitch':
        id2label = _id2pitch
    else:
        id2label = _id2dur

    candidates = []
    for k in range(topk.indices.size(1)):
        cls_id = topk.indices[0, k].item()
        label = id2label[cls_id]
        prob  = topk.values[0, k].item()
        candidates.append({'label': label, 'prob': round(prob, 4)})

    pred_label = candidates[0]['label']

    # For duration: convert from whole-note label to quarterLength
    # Duration labels are already in quarterLength units — no conversion
    if error_type == 'duration':
        pred_ql = pred_label
        cand_display = [{'label': c['label'] + ' QL', 'prob': c['prob']}
                        for c in candidates]
    else:
        pred_ql = None
        cand_display = candidates

    # ── Write corrected MusicXML ──────────────────────────────
    # Re-parse to get a fresh tree for editing (don't mutate the parsed one)
    edit_tree = etree.parse(str(input_path))
    edit_root = edit_tree.getroot()
    edit_ns   = edit_root.nsmap.get(None, "")
    et = lambda name: _xt(name, edit_ns)

    # Navigate to the target <note> element
    target_part = None
    for p in edit_root.findall(f".//{et('part')}"):
        if p.get('id') == pid:
            target_part = p; break
    measures = target_part.findall(et('measure'))
    target_meas = measures[mi]
    note_els = [el for el in target_meas
                if (el.tag.split('}')[-1] if '}' in el.tag else el.tag) == 'note']
    target_note_el = note_els[ni]

    if error_type == 'pitch':
        _apply_pitch(target_note_el, pred_label, edit_ns)
    else:
        # Get divisions for this measure
        divs = 1
        for m in measures[:mi+1]:
            for attr in m.findall(et('attributes')):
                d_el = attr.find(et('divisions'))
                if d_el is not None:
                    divs = int(d_el.text)
        w = _apply_duration(target_note_el, pred_ql, divs, edit_ns)
        warnings.extend(w)

    xml_out = output_dir / 'model_corrected.musicxml'
    edit_tree.write(str(xml_out), xml_declaration=True, encoding='UTF-8')

    # ── Write MIDI ────────────────────────────────────────────
    midi_out = output_dir / 'model_corrected.mid'
    n_events = _xml_to_midi(edit_tree, midi_out, edit_ns)

    # ── Verify written XML ────────────────────────────────────
    v_tree = etree.parse(str(xml_out))
    v_root = v_tree.getroot()
    v_ns   = v_root.nsmap.get(None, "")
    vt     = lambda name: _xt(name, v_ns)
    v_part = None
    for p in v_root.findall(f".//{vt('part')}"):
        if p.get('id') == pid:
            v_part = p; break
    v_meas = v_part.findall(vt('measure'))[mi]
    v_notes = [el for el in v_meas
               if (el.tag.split('}')[-1] if '}' in el.tag else el.tag) == 'note']
    v_note = v_notes[ni]

    if error_type == 'pitch':
        vp = v_note.find(vt('pitch'))
        if vp is not None:
            v_val = vp.find(vt('step')).text + vp.find(vt('octave')).text
            xml_ok = v_val == pred_label
        else:
            xml_ok = False
            warnings.append("Verified note has no pitch element")
    else:
        vd = v_note.find(vt('duration'))
        divs_v = 1
        for m in v_part.findall(vt('measure'))[:mi+1]:
            for attr in m.findall(vt('attributes')):
                d_el = attr.find(vt('divisions'))
                if d_el is not None:
                    divs_v = int(d_el.text)
        expected = int(_Frac(pred_ql) * divs_v)
        actual = int(vd.text) if vd is not None else -1
        xml_ok = actual == expected
        if not xml_ok:
            warnings.append(f"Duration verify: expected {expected}, got {actual}")

    elapsed = _time.monotonic() - t0

    # ── Save prediction JSON ──────────────────────────────────
    result = {
        'input_file': str(input_path),
        'target_locator': target_locator,
        'error_type': error_type,
        'current_value': current_val,
        'prediction': pred_label if error_type == 'pitch' else pred_ql,
        'candidates': cand_display,
        'xml_verified': xml_ok,
        'midi_events': n_events,
        'checkpoint': str(_JOINT_CKPT_PATH),
        'context_notes': len(cp_ids),
        'elapsed_s': round(elapsed, 3),
        'warnings': warnings,
        'output_xml': str(xml_out),
        'output_midi': str(midi_out),
    }

    with open(output_dir / 'prediction.json', 'w') as f:
        json.dump(result, f, indent=2)

    return result


print("API ready: list_notes(), correct_musicxml()")

In [ ]:
# ── Demo: run inference on two corrupted MusicXML files ───────────
# Copy one pitch and one duration corrupted file from existing demo,
# then call correct_musicxml with only the file + locator.

from shutil import copyfile

INFER_DEMO = ML_DIR / 'demo' / 'inference_demo'
INFER_DEMO.mkdir(parents=True, exist_ok=True)

# ── Copy corrupted files with neutral names ───────────────────
pitch_src = ML_DIR / 'demo' / 'unified_joint' / 'pitch_12_samples_5023603_p4-s3' / 'corrupted.musicxml'
dur_src   = ML_DIR / 'demo' / 'unified_joint' / 'dur_327_samples_6474469_p1-s1' / 'corrupted.musicxml'

pitch_input = INFER_DEMO / 'pitch_test_input.musicxml'
dur_input   = INFER_DEMO / 'dur_test_input.musicxml'
copyfile(pitch_src, pitch_input)
copyfile(dur_src, dur_input)
print(f"Copied test inputs to {INFER_DEMO.relative_to(ML_DIR)}/")

# ── Case 1: Pitch correction ─────────────────────────────────
print(f"\n{'='*70}")
print("  CASE 1: Pitch correction")
print(f"{'='*70}")

# Show note context around target
print("\nNotes near target (measure 0):")
pnotes = list_notes(pitch_input)
# Filter to just measure 0 for compact display (already printed by list_notes)

pitch_result = correct_musicxml(
    input_path=pitch_input,
    target_locator={'part_id': 'P2', 'measure_index': 0, 'note_index': 5},
    error_type='pitch',
    output_dir=INFER_DEMO / 'pitch_output',
)

print(f"\n  Input file:  {pitch_input.name}")
print(f"  Target:      part=P2, measure=0, note=5")
print(f"  Current:     {pitch_result['current_value']}")
print(f"  Prediction:  {pitch_result['prediction']}")
print(f"  Candidates:  {pitch_result['candidates']}")
print(f"  XML ok:      {pitch_result['xml_verified']}")
print(f"  MIDI events: {pitch_result['midi_events']}")
print(f"  Time:        {pitch_result['elapsed_s']:.3f}s")
print(f"  Output:      {pitch_result['output_xml']}")
if pitch_result['warnings']:
    print(f"  Warnings:    {pitch_result['warnings']}")

# Ground-truth check (outside the interface — for validation only)
print(f"\n  [Validation] Ground truth = E4, predicted = {pitch_result['prediction']}"
      f" → {'MATCH' if pitch_result['prediction'] == 'E4' else 'MISMATCH'}")

# ── Case 2: Duration correction ──────────────────────────────
print(f"\n{'='*70}")
print("  CASE 2: Duration correction")
print(f"{'='*70}")

dur_result = correct_musicxml(
    input_path=dur_input,
    target_locator={'part_id': 'P2', 'measure_index': 3, 'note_index': 24},
    error_type='duration',
    output_dir=INFER_DEMO / 'dur_output',
)

print(f"\n  Input file:  {dur_input.name}")
print(f"  Target:      part=P2, measure=3, note=24")
print(f"  Current:     {dur_result['current_value']} QL")
print(f"  Prediction:  {dur_result['prediction']} QL")
print(f"  Candidates:  {dur_result['candidates']}")
print(f"  XML ok:      {dur_result['xml_verified']}")
print(f"  MIDI events: {dur_result['midi_events']}")
print(f"  Time:        {dur_result['elapsed_s']:.3f}s")
print(f"  Output:      {dur_result['output_xml']}")
if dur_result['warnings']:
    print(f"  Warnings:    {dur_result['warnings']}")

# Ground-truth check (outside the interface)
print(f"\n  [Validation] Ground truth = 1/2 QL,"
      f" predicted = {dur_result['prediction']} QL"
      f" → {'MATCH' if dur_result['prediction'] == '1/2' else 'MISMATCH'}")

# ── Summary ──────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  Output files:")
for sub in ['pitch_output', 'dur_output']:
    d = INFER_DEMO / sub
    for f in sorted(d.iterdir()):
        print(f"    {f.relative_to(ML_DIR)}")
print(f"{'='*70}")

## Accidental Correction — Fine-tuning

Predict MusicXML `<alter>` from corrupted score context.
- 5 classes: {-2: double-flat, -1: flat, 0: natural, +1: sharp, +2: double-sharp}
- Encoding: context notes use chromatic MIDI, target note uses natural MIDI (hides alteration)
- Initialized from joint checkpoint, freeze 10 layers, train last 2 + acc_head

In [ ]:
# ── Accidental v1: Data Build + Training ────────────────────────
# Run: m2bert_acc_20261102_112507
# Source: m2bert_joint_20261030_213214 (joint pitch+duration checkpoint)
#
# Data:
#   train=15,012  dev=1,437  (score-disjoint, same split as pitch/dur)
#   Train alter dist: -2:14, -1:2932, 0:9492, +1:2559, +2:15
#   Dev alter dist:   -2:7,  -1:335,  0:880,  +1:215,  +2:0
#
# Architecture:
#   Backbone: MidiModernBERT (12 layers, 768 hidden)
#   Head: Dropout(0.1) → Linear(768, 5)
#   Frozen: first 10 layers; Trained: last 2 + head
#
# Hyperparameters:
#   lr=2e-5, batch=16, max_epochs=6, patience=2
#   Training time: 192s (3 epochs trained, early stop at epoch 5)
#
# Results:
#   Best epoch: 2
#   Top-1 accuracy: 88.31% (majority baseline: 61.24%)
#   Macro-F1 (5-class): 0.5122
#   Per-class recall:
#     alter=-2: 0/7   (0.00%)
#     alter=-1: 279/335 (83.28%)
#     alter= 0: 820/880 (93.18%)
#     alter=+1: 161/215 (74.88%)
#     alter=+2: 0/0   (N/A)
#
# Checkpoint: ml/data/runs/m2bert_acc_20261102_112507/best_checkpoint.pt (242 MB)

print("Accidental v1 training record (not runnable)")

In [ ]:
# ── Accidental v2: Double-Accidental Augmentation + Fine-tune ───
# Run: m2bert_acc_v2_20261102_115241
# Source: m2bert_acc_20261102_112507 (v1 accidental checkpoint)
#
# Motivation: v1 had 0% recall on alter=-2 (14 train, 7 dev).
# Augmentation: extracted 200 alter=-2 and 200 alter=+2 notes from
#   existing train MusicXML files. Total augmented train: 15,412.
#
# Hyperparameters:
#   lr=4e-6 (v1_lr / 5), batch=16, max_epochs=4, patience=2
#   Training time: 158s (best at epoch 1)
#
# Selection criteria:
#   Primary: dev macro-F1(5) > baseline
#   Guard: top-1 drop ≤ 1pp
#
# Results comparison:
#   ┌────────────────────────┬──────────┬──────────┐
#   │ Metric                 │ v1       │ v2       │
#   ├────────────────────────┼──────────┼──────────┤
#   │ Top-1 accuracy         │ 88.31%   │ 87.68%   │
#   │ Macro-F1 (5-class)     │ 0.5122   │ 0.5419   │
#   │ Macro-F1 (supported)   │   —      │ 0.6773   │
#   │ alter=-2 recall        │ 0/7      │ 1/7      │
#   │ alter=-1 recall        │ 83.28%   │ ~83%     │
#   │ alter= 0 recall        │ 93.18%   │ ~93%     │
#   │ alter=+1 recall        │ 74.88%   │ ~75%     │
#   └────────────────────────┴──────────┴──────────┘
#
# Decision: ADOPT v2
#   Top-1 drop: -0.63pp (within 1pp threshold)
#   Macro-F1 improvement: +0.030 (0.5122 → 0.5419)
#   Double-flat recall: 0% → 14.3%
#
# Checkpoint: ml/data/runs/m2bert_acc_v2_20261102_115241/best_checkpoint.pt (242 MB)
# Active in inference notebook: m2bert_finetune_comparison.ipynb cell 4

print("Accidental v2 training record (not runnable)")

## Experiment: Pitch Context Enhancement v2 — Frozen Backbone, Cached Base Logits

**Date:** 2026-11-02  
**Goal:** Freeze backbone + pitch_head from joint checkpoint, train only context branch.  
**Motivation:** v1 trained backbone layers 10-11 + pitch_head + ctx_branch (10M+ params); v2 trains only ctx_branch (227K params) with cached base logits for fast iteration.

### Time Signature Recovery

Attempted to recover time signatures from MusicXML `<attributes><time>` elements:
- **Result:** Only 12.0% of train / 11.6% of dev samples have `<time>` in their MusicXML
- **Cause:** OLiMPiC-1.0-synthetic excerpts start mid-piece; time signature defined earlier is not repeated
- **Decision:** Genuine unknown; ts unknown rate improved slightly from 89.4% → 88.0% (train) / 88.4% (dev) by also checking XML
- Source: MusicXML `<time>` only, no inference from note durations or defaults

### Architecture

- Backbone + pitch_head: frozen from `m2bert_joint_20261030_213214/best_checkpoint.pt` (eval mode, no grad)
- Context branch: same as v1 (key×16 + ts×21 + voice×8 + staff×3, emb_dim=16, FC(832→256→52), zero-init output)
- Residual: `logits = base_logits + ctx_branch(hidden, ctx_ids)`
- Cache: one backbone forward pass per split, store `(hidden_state[target_idx], base_logits)` per example

### Training

- Trainable params: 227,380 (ctx_branch only)
- LR: 5e-4, batch_size=64, AdamW(wd=0.01)
- Max 6 epochs, patience=2, 300s budget
- Cache time: 175.0s (one-time backbone forward)
- Training time: 2.6s (6 epochs on cached tensors)

### Results

| Model | Top-1 | Top-5 | CE |
|---|---:|---:|---:|
| Original joint | 310/499 (62.12%) | 449/499 (89.98%) | 1.3078 |
| v1 (ctx, backbone trainable) | 318/499 (63.73%) | 443/499 (88.78%) | 1.8871 |
| **v2 (ctx, frozen backbone)** | **323/499 (64.73%)** | **448/499 (89.78%)** | **1.6506** |

### Flip Analysis (vs v1)

| Direction | Count |
|---|---:|
| v1→Wrong, v2→Right | 21 |
| v1→Right, v2→Wrong | 16 |
| Net | +5 |

### Flip Analysis (vs baseline)

| Direction | Count |
|---|---:|
| Base→Wrong, v2→Right | 31 |
| Base→Right, v2→Wrong | 18 |
| Net | +13 |

### Decision: ADOPTED

- top-1: 323 > 318 → adopt
- Also: top-5 recovered (448 vs 443), CE improved (1.6506 vs 1.8871)
- v2 is strictly better than v1 on all three metrics

### Checkpoint

- Run: `m2bert_ctx_pitch_v2_20261102_175504/`
- Type: `ctx_branch_only` — stores only context branch weights
- Inference: load joint backbone + pitch_head, then overwrite ctx_branch from v2 checkpoint
- Verified: pitch demo E4 correct, MIDI 64


In [ ]:
# ── Experiment record: pitch context v2 (frozen backbone) ──
# Training log from m2bert_ctx_pitch_v2_20261102_175504

train_log = [
    {'epoch': 1, 'train_loss': 0.3719, 'dev_top1': 321, 'dev_top5': 444, 'dev_ce': 1.3138, 'elapsed_s': 0.5},
    {'epoch': 2, 'train_loss': 0.2820, 'dev_top1': 322, 'dev_top5': 445, 'dev_ce': 1.4417, 'elapsed_s': 0.9},
    {'epoch': 3, 'train_loss': 0.2268, 'dev_top1': 315, 'dev_top5': 448, 'dev_ce': 1.5316, 'elapsed_s': 1.3},
    {'epoch': 4, 'train_loss': 0.1824, 'dev_top1': 323, 'dev_top5': 448, 'dev_ce': 1.6506, 'elapsed_s': 1.8},  # best
    {'epoch': 5, 'train_loss': 0.1424, 'dev_top1': 316, 'dev_top5': 442, 'dev_ce': 1.7442, 'elapsed_s': 2.2},
    {'epoch': 6, 'train_loss': 0.1175, 'dev_top1': 316, 'dev_top5': 447, 'dev_ce': 1.8091, 'elapsed_s': 2.6},  # early stop
]

# Comparison
comparison = {
    'original_joint': {'top1': 310, 'top5': 449, 'ce': 1.3078},
    'v1_ctx_trainable': {'top1': 318, 'top5': 443, 'ce': 1.8871},
    'v2_ctx_frozen': {'top1': 323, 'top5': 448, 'ce': 1.6506},
}

timings = {
    'ts_extraction': 1.6,
    'data_build': 1.4,
    'cache_backbone_forward': 175.0,
    'training_ctx_branch': 2.6,
    'total': 183.2,
}

ts_recovery = {
    'train_found': 1198, 'train_total': 9998, 'train_pct': 12.0,
    'dev_found': 58, 'dev_total': 499, 'dev_pct': 11.6,
    'note': 'MusicXML excerpts without <time>; genuine unknown',
}

print(f"v2 adopted: top-1 323/499 > 318/499 (v1)")
print(f"v2 improvements over v1: top-1 +5, top-5 +5, CE -0.2365")
print(f"v2 improvements over joint: top-1 +13, top-5 -1, CE +0.3428")
print(f"Training: 2.6s (vs 130.7s for v1)")


## Experiment: Pitch Top-5 Reranker — Learned Residual Scoring

**Date:** 2026-11-02
**Goal:** Reorder the frozen pitch model's top-5 candidates using musical context features.
**Motivation:** 125/499 dev samples have the correct answer in top-5 but not ranked first; a lightweight reranker might recover some.

### Architecture

- Frozen: joint backbone + pitch_head + v2 ctx_branch (all from previous experiments)
- Reranker: shared MLP (16 → 64 → 1), residual output zero-initialized
- Score: `reranked_score = original_logit + reranker_residual`
- Parameters: 1,153 (tiny)

### Candidate Features (16 per candidate)

| Feature | Description |
|---|---|
| logit, log_prob, rank, logit_diff | Model score features |
| signed_interval_prev, abs_interval_prev, has_prev | Interval to nearest preceding same-voice note |
| signed_interval_next, abs_interval_next, has_next | Interval to nearest following same-voice note |
| register_offset_from_median | Candidate MIDI vs local pitch median (±8 same-voice notes) |
| simul_interval_min/max/mean, simul_abs_mean, has_simul | Intervals to simultaneous notes at same onset |

All context from corrupted input. Missing indicators for absent neighbors/simultaneous notes.

### Training

- Train coverage: 9,987/9,998 (target in top-5)
- LR: 1e-3, batch=128, AdamW(wd=0.01), max 10 epochs, patience=2
- Early stopped at epoch 3 (best = epoch 1)
- Training time: 0.6s

### Results

| Model | Top-1 | Top-5 |
|---|---:|---:|
| Baseline (v2 ctx) | 323/499 | 448/499 |
| + Reranker | 323/499 | 448/499 |

**No change.** Zero Wrong→Right, zero Right→Wrong. Of 125 in-candidate-but-wrong, zero recovered.

### Analysis

The reranker learned essentially zero residuals — the musical context features (intervals, register, harmony) don't provide enough discriminative signal to reorder candidates beyond what the 768-dim BERT hidden state + context branch already capture.

The model's top-1 errors in the top-5 set are likely cases where the corruption creates genuine ambiguity (multiple musically plausible pitches), not cases where simple interval/register rules would help.

### Decision: NOT ADOPTED

- top-1 = 323 (not > 323) → retain current v2 ctx_branch inference
- Run saved for record: `m2bert_reranker_20261102_182741/`


In [ ]:
# ── Experiment record: pitch reranker ──
# Run: m2bert_reranker_20261102_182741

train_log = [
    {'epoch': 1, 'train_loss': 0.1133, 'dev_top1': 323, 'elapsed_s': 0.4},
    {'epoch': 2, 'train_loss': 0.1006, 'dev_top1': 323, 'elapsed_s': 0.5},
    {'epoch': 3, 'train_loss': 0.0985, 'dev_top1': 322, 'elapsed_s': 0.6},  # early stop
]

results = {
    'baseline_top1': 323, 'baseline_top5': 448,
    'reranker_top1': 323, 'reranker_top5': 448,
    'wrong_to_right': 0, 'right_to_wrong': 0,
    'in_cand_recovered': 0, 'in_cand_total': 125,
    'reranker_params': 1153,
    'train_valid': 9987, 'train_total': 9998,
}

timings = {
    'data_build': 2.0, 'cache_forward': 171.2,
    'training': 0.6, 'total': 181.5,
}

print(f"Not adopted: top-1 unchanged at 323/499")
print(f"Musical context features insufficient to reorder top-5 beyond BERT representation")


## Experiment: Pitch Context Enhancement v3 — Best Result

**Date:** 2026-11-02  
**Run:** `m2bert_cov_expand_20261102_235633` → adopted  
**Change vs v2:** Freeze backbone + pitch_head; train only `ctx_branch` with LR=5e-4, random init (not warm-started from v2). 10 epochs, patience=2, budget 300s.

**Result:** dev top-1 **324/499** (64.93%) at epoch 2 — beats v2 (323). Combined inference: `pitch_head_logits + ctx_branch_logits`.  
This is the **current best** pitch-only result. Adopted.

| metric | value |
|--------|-------|
| baseline (ph only) | 310/499 |
| v2 top-1 | 323/499 |
| **v3 top-1** | **324/499** |
| v3 top-5 | 439/499 |
| trainable params | 227,380 |
| train time | 1.2 s |


In [ ]:
# ── Experiment record: pitch context v3 ──────────────────────────────
# Run: m2bert_ctx_pitch_v3_20261102_184759
# Config snapshot (read-only, no re-training)
_CTX_V3_CFG = {
    "task": "pitch_context_enhancement_v3_epoch10",
    "frozen": ["backbone", "pitch_head"],
    "trainable": ["ctx_branch"],
    "ctx_lr": 0.0005, "batch_size": 64, "max_epochs": 10, "patience": 2,
    "trainable_params": 227380,
    "train_n": 9998, "dev_n": 499,
    "baseline_top1": 310, "v2_top1": 323,
    "best_epoch": 2, "best_top1": 324, "best_top5": 439,
    "adopted": True,
    "run_name": "m2bert_ctx_pitch_v3_20261102_184759"
}
print('ctx_pitch v3 best_top1:', _CTX_V3_CFG['best_top1'])

## Experiment: Coverage Expansion — All Variants Failed

**Date:** 2026-11-02/03  
**Hypothesis:** Augmenting 9,998 training CPs with 9,998 diversity-selected harder CPs (minority voices, unusual registers, rare pitch classes) would further improve ctx_branch.  
**Variants:** 4 runs varying frozen layers, LR, and whether ctx_branch starts from v3 weights.

| run | frozen | init | lr | best_ep | comb_top1 | adopted |
|-----|--------|------|----|---------|-----------|---------|
| 235330 | backbone | v3 weights | 1e-4 | 0 | 326 (=baseline) | No |
| 235633 | backbone | random | 1e-4 | 7 | 312 | No |
| 235835 | backbone | random | 1e-4 | 5 | 313 | No |
| 000019 | backbone+ph | random | 5e-4 | 2 | 312 | No |

**Conclusion:** All expanded-data variants fall below v3 (324). The diversity-selected examples are harder and disrupt the ctx_branch residual balance. Coverage expansion abandoned.

In [ ]:
# ── Experiment records: coverage expansion (all rejected) ────────────
_COV_EXPAND_RUNS = [
    # (run_name, baseline_ph_top1, best_comb_top1, adopted, note)
    ('m2bert_cov_expand_20261102_235330', 310, 326, False,
     'v3-init, backbone frozen, best_ep=0 = v3 baseline; MPS float artifact'),
    ('m2bert_cov_expand_20261102_235633', 310, 312, False,
     'random init, backbone frozen, lr=1e-4, best_ep=7'),
    ('m2bert_cov_expand_20261102_235835', 310, 313, False,
     'random init, backbone frozen, lr=1e-4, best_ep=5'),
    ('m2bert_cov_expand_20261103_000019', 310, 312, False,
     'random init, backbone+ph frozen, lr=5e-4, best_ep=2'),
]
print('Coverage expansion results:')
for r in _COV_EXPAND_RUNS:
    print(f'  {r[0]}: comb_top1={r[2]}, adopted={r[3]}')

## Experiment: Joint Fine-tune on 19,996 Expanded Examples — Negative Result

**Date:** 2026-11-03  
**Setup:** Restore backbone+pitch_head from joint checkpoint + ctx_branch from v3. Freeze layers 0–9; train layers 10–11 + final_norm + pitch_head + ctx_branch simultaneously. backbone LR=2e-5, head/ctx LR=1e-4, batch=32, real forward passes.  
**Run 1 (6ep/patience=3):** best_epoch=0, loss fell 1.01→0.29, dev stayed below baseline.  
**Run 2 (10ep/no early stop):** loss fell to 0.003 (severe overfit); best dev epoch=10 @ 310/499.

| epoch | dev top-1 |
|-------|----------|
| baseline | 326 (MPS) / 324 (CPU) |
| 1 | 305 |
| 2 | 308 |
| 3 | 311 |
| 4 | 308 |
| 5 | 308 |
| 6 | 310 |
| 7 | 307 |
| 8 | 308 |
| 9 | 308 |
| 10 | **310** |

**Conclusion:** Unfreezing last 2 backbone layers on the harder expanded dataset disrupts the residual balance. Joint fine-tuning abandoned. Current best remains ctx_branch v3 (324/499).

In [ ]:
# ── Experiment records: joint fine-tune on 19,996 examples ──────────
_JOINT_EXPANDED_RUNS = [
    {
        'run_name': 'm2bert_joint_expanded_20261103_004230',
        'max_epochs': 6, 'patience': 3,
        'best_epoch': 0, 'best_comb_top1': 326,
        'dev_log': [305, 308, 311],  # stopped ep3 by patience
        'adopted': False,
        'note': 'baseline=326 MPS (=324 CPU); no epoch exceeded baseline'
    },
    {
        'run_name': 'm2bert_joint_expanded_20261103_004745',
        'max_epochs': 10, 'patience': 10,
        'best_epoch': 0, 'best_comb_top1': 326,
        'dev_log': [305, 308, 308, 309, 302, 306, 308, 308, 308, 310],
        'final_train_loss': 0.0026,
        'adopted': False,
        'note': 'severe overfit (loss→0.003); best training epoch=310 < 326 baseline'
    },
]
for r in _JOINT_EXPANDED_RUNS:
    print(f"{r['run_name']}: best={r['best_comb_top1']}, log={r['dev_log']}")

---
# Part C: M2BERT Standalone Inference & Development Cells

**Source:** `m2bert_finetune_comparison.ipynb` cells 34–59 (moved here during notebook cleanup).  
These cells are from iterative development of the unified correction demo and the standalone inference pipeline (`correct_musicxml` public API). They are preserved here for reference; the main notebook (`m2bert_finetune_comparison.ipynb`) retains only the clean pipeline (cells 0–33).

## Sections
- **Cells 103–116 (archive):** Debug/exploration fragments from unified demo development
- **Cells 117–129 (archive):** Standalone inference pipeline (`correct_musicxml` API)


In [ ]:

# ── Unified Correction Demo: Joint Model → MusicXML → MIDI ────────
# Requires: all prior cells executed (parse_lmx_notes, dev_raw,
# dur_dev_cp, dur_dev_details, _lmx_path, _xml_path, make_edited_xml,
# set_note_pitch, musicxml_to_midi, find_error_note_index, etc.)

import hashlib as _hl
from copy import deepcopy
from fractions import Fraction as _Frac

# ── Part A: Load joint checkpoint + prediction caches ─────────────

JOINT_RUN_DIR = ML_DIR / 'data' / 'runs' / 'm2bert_joint_20261030_213214'
UNIFIED_OUT   = ML_DIR / 'demo' / 'unified_joint'
UNIFIED_OUT.mkdir(parents=True, exist_ok=True)

# Prediction caches (avoid re-inference)
with open(JOINT_RUN_DIR / 'pitch_dev_predictions.json') as f:
    pitch_preds = json.load(f)
with open(JOINT_RUN_DIR / 'dur_dev_predictions.json') as f:
    dur_preds = json.load(f)

# Label maps
with open(JOINT_RUN_DIR / 'pitch_label_map.json') as f:
    pitch_lm = json.load(f)
with open(JOINT_RUN_DIR / 'dur_label_map.json') as f:
    dur_lm = json.load(f)
id2pitch_j = {v: k for k, v in pitch_lm.items()}
id2dur_j   = {v: k for k, v in dur_lm.items()}

print(f"Loaded predictions: pitch={len(pitch_preds)}, dur={len(dur_preds)}")
print(f"Label maps: pitch={len(pitch_lm)}, dur={len(dur_lm)}")

# Demo indices (same as cell 31)
DEMO_INDICES = [12, 57, 327]


In [ ]:

# Cell 1 — imports
import sys, json, re, random, time, pickle
from pathlib import Path
from fractions import Fraction as _Frac
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
print("Imports OK")


In [ ]:

# Cell 2 — paths and CP vocab
ML_DIR       = Path('/Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml')
PROCESSED    = ML_DIR / 'data' / 'processed'
PRETRAINED   = ML_DIR / 'data' / 'pretrained' / 'm2bert'
M2_RUN_DIR   = ML_DIR / 'data' / 'runs' / 'm2bert_finetune'
DATASET_ROOT = ML_DIR / 'dataset' / 'olimpic-1.0-synthetic'
M2_RUN_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cuda') if torch.cuda.is_available() else
          torch.device('cpu'))
print(f'DEVICE: {DEVICE}')

with open(PRETRAINED / 'CP.pkl', 'rb') as f:
    _cp = pickle.load(f)
CP_TOK2ID = _cp[0]
print('CP attribute sizes:', {k: len(v) for k, v in CP_TOK2ID.items()})

BAR_PAD    = CP_TOK2ID['Bar']['Bar <PAD>']
BAR_MASK   = CP_TOK2ID['Bar']['Bar <MASK>']
POS_PAD    = CP_TOK2ID['Position']['Position <PAD>']
PITCH_PAD  = CP_TOK2ID['Pitch']['Pitch <PAD>']
PITCH_MASK = CP_TOK2ID['Pitch']['Pitch <MASK>']
DUR_PAD    = CP_TOK2ID['Duration']['Duration <PAD>']
CP_PAD = [BAR_PAD, POS_PAD, PITCH_PAD, DUR_PAD]
print(f'CP_PAD: {CP_PAD}   PITCH_MASK: {PITCH_MASK}')


In [ ]:

# Cell 3 — pitch classes + data loading
PITCH_RE = re.compile(r'^[A-G][0-9]$')

def _lmx_path(sid):
    parts = sid.split('/')
    return DATASET_ROOT / 'samples' / parts[1] / (parts[2] + '.lmx')

pitches = set()
for sid in (DATASET_ROOT / 'samples.train.txt').read_text().split():
    p = _lmx_path(sid)
    if p.exists():
        pitches.update(t for t in p.read_text().split() if PITCH_RE.match(t))
pitch_classes = {p: i for i, p in enumerate(sorted(pitches))}
id2pitch      = {i: p for p, i in pitch_classes.items()}
N_CLASSES     = len(pitch_classes)
print(f'Pitch classes: {N_CLASSES}')

with open(PROCESSED / 'correction_train_10k.jsonl') as f:
    train_raw = [json.loads(l) for l in f]
with open(PROCESSED / 'correction_dev_v1.jsonl') as f:
    dev_raw = [json.loads(l) for l in f]
print(f'Train: {len(train_raw)}  Dev: {len(dev_raw)}')


In [ ]:

# Cell 5 — parse_lmx_notes
import re as _re

_DURATION_BASE = {
    'breve': _Frac(2),      'whole': _Frac(1),      'half': _Frac(1, 2),
    'quarter': _Frac(1, 4), 'eighth': _Frac(1, 8),  '16th': _Frac(1, 16),
    '32nd': _Frac(1, 32),   '64th': _Frac(1, 64),   '128th': _Frac(1, 128),
    '256th': _Frac(1, 256), '512th': _Frac(1, 512), '1024th': _Frac(1, 1024),
}
_DECORATION = frozenset({
    'beam:begin', 'beam:end', 'beam:forward-hook', 'beam:backward-hook',
    'tied:start', 'tied:stop', 'tuplet:start', 'tuplet:stop',
    'slur:start', 'slur:stop', 'trill-mark', 'staccato', 'accent',
    'strong-accent', 'tenuto', 'fermata', 'arpeggiate',
    'tremolo:1', 'tremolo:2', 'tremolo:3', 'tremolo:4',
    'tremolo:single', 'tremolo:start', 'tremolo:stop',
})
_ACCIDENTAL = frozenset({
    'sharp', 'flat', 'natural', 'double-sharp',
    'flat-flat', 'natural-sharp', 'natural-flat',
})
_SKIP = frozenset({'time'})
_TIME_MOD_RE = _re.compile(r'^(\d+)in(\d+)$')

def _parse_dur(tokens, i):
    if i >= len(tokens) or tokens[i] not in _DURATION_BASE:
        return None, i
    base = _DURATION_BASE[tokens[i]]
    i += 1
    m = _TIME_MOD_RE.match(tokens[i]) if i < len(tokens) else None
    if m:
        base *= _Frac(int(m.group(2)), int(m.group(1)))
        i += 1
    n_dots = 0
    while i < len(tokens) and tokens[i] == 'dot':
        n_dots += 1
        i += 1
    if n_dots:
        base *= (2 - _Frac(1, 2 ** n_dots))
    return base, i

def parse_lmx_notes(tokens, sample_id=''):
    notes, warnings = [], []
    current_voice = current_staff = None
    current_beats = current_beat_type = None
    measure_num = 0
    current_onset = _Frac(0)
    chord_group_id = 0
    chord_leader_onset = _Frac(0)
    i, n = 0, len(tokens)
    while i < n:
        tok = tokens[i]
        if tok == 'measure':
            measure_num += 1
            current_onset = _Frac(0)
            i += 1; continue
        if tok in _SKIP or tok.startswith('key:'):
            i += 1; continue
        if tok.startswith('beats:'):
            try: current_beats = int(tok[6:])
            except ValueError: warnings.append(f'bad beats {tok!r}')
            i += 1; continue
        if tok.startswith('beat-type:'):
            try: current_beat_type = int(tok[10:])
            except ValueError: warnings.append(f'bad beat-type {tok!r}')
            i += 1; continue
        if tok.startswith('clef:'):
            i += 1
            if i < n and tokens[i].startswith('staff:'): i += 1
            continue
        if tok in _DECORATION:
            i += 1; continue
        span_start = i
        is_grace = is_chord = False
        if tok == 'print-object:no':
            i += 1
            if i >= n: break
            tok = tokens[i]
        if tok == 'grace':
            is_grace = True; i += 1
            if i >= n: break
            tok = tokens[i]
            if tok == 'grace:slash':
                i += 1
                if i >= n: break
                tok = tokens[i]
        if tok == 'chord':
            is_chord = True; i += 1
            if i >= n: break
            tok = tokens[i]
        pitch = None; is_rest = False; pitch_token_position = None
        if PITCH_RE.match(tok):
            pitch, pitch_token_position = tok, i; i += 1
        elif tok == 'rest':
            is_rest = True; i += 1
        elif tok == 'backup':
            i += 1; dur, i = _parse_dur(tokens, i)
            if dur is not None: current_onset = max(_Frac(0), current_onset - dur)
            else: warnings.append(f'backup no dur @ {i}')
            continue
        elif tok == 'forward':
            i += 1; dur, i = _parse_dur(tokens, i)
            if dur is not None: current_onset += dur
            else: warnings.append(f'forward no dur @ {i}')
            continue
        else:
            warnings.append(f'unexpected {tok!r} @ {i}'); i += 1; continue
        if i < n and tokens[i].startswith('voice:'):
            current_voice = tokens[i]; i += 1
        duration = None
        if i < n:
            if tokens[i] in _DURATION_BASE:
                duration, i = _parse_dur(tokens, i)
            elif tokens[i] == 'rest:measure':
                if current_beats is not None and current_beat_type is not None:
                    duration = _Frac(current_beats, current_beat_type)
                else:
                    duration = None
                    warnings.append(f'rest:measure unknown dur @ {sample_id}')
                i += 1
        if i < n and tokens[i] in _ACCIDENTAL: i += 1
        if i < n and tokens[i].startswith('stem:'): i += 1
        if i < n and tokens[i].startswith('staff:'):
            current_staff = tokens[i]; i += 1
        while i < n and tokens[i] in _DECORATION: i += 1
        onset = chord_leader_onset if is_chord else current_onset
        if is_chord: grp = chord_group_id
        else:
            chord_group_id += 1; grp = chord_group_id
            chord_leader_onset = current_onset
        notes.append({'measure': measure_num, 'voice': current_voice, 'staff': current_staff,
                      'onset': onset, 'duration': duration, 'pitch': pitch,
                      'is_rest': is_rest, 'is_grace': is_grace,
                      'chord_group': grp, 'pitch_token_position': pitch_token_position,
                      'token_span': (span_start, i)})
        if not is_chord and not is_grace and duration is not None:
            current_onset += duration
    return notes, warnings

print('parse_lmx_notes defined')


In [ ]:

# Cell 21 — MusicXML utilities
from lxml import etree
import mido

_STEP_SEMI = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}

def _xml_tag(name, ns):
    return f"{{{ns}}}{name}" if ns else name

def find_error_note_index(ex, parse_lmx_notes):
    notes, _ = parse_lmx_notes(ex["input_tokens"], ex.get("sample_id", ""))
    return next(i for i, n in enumerate(notes)
               if n["pitch_token_position"] == ex["error_position"])

def set_note_pitch(note_el, pitch_token, ns=""):
    t = lambda name: _xml_tag(name, ns)
    pitch_el = note_el.find(t("pitch"))
    if pitch_el is None: return
    pitch_el.find(t("step")).text   = pitch_token[0]
    pitch_el.find(t("octave")).text = pitch_token[1:]
    alter = pitch_el.find(t("alter"))
    if alter is not None: pitch_el.remove(alter)
    acc = note_el.find(t("accidental"))
    if acc is not None: note_el.remove(acc)

def make_edited_xml(xml_path, note_index, new_pitch):
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    notes = root.findall(f".//{t('note')}")
    set_note_pitch(notes[note_index], new_pitch, ns)
    return tree

def musicxml_to_midi(xml_tree, midi_path):
    root = xml_tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    TPQN = 480; divisions = 1
    events = []; current_time = 0; prev_onset = 0
    for measure in root.findall(f".//{t('measure')}"):
        for attr in measure.findall(t("attributes")):
            d = attr.find(t("divisions"))
            if d is not None: divisions = int(d.text)
        scale = TPQN / divisions
        for el in measure:
            ltag = el.tag.split("}")[-1] if "}" in el.tag else el.tag
            if ltag == "note":
                chord_el = el.find(t("chord")); pitch_el = el.find(t("pitch"))
                dur_el = el.find(t("duration"))
                dur = int(dur_el.text) if dur_el is not None else 0
                onset = prev_onset if chord_el is not None else current_time
                if chord_el is None: prev_onset = current_time
                if pitch_el is not None:
                    step = pitch_el.find(t("step")).text
                    octave = int(pitch_el.find(t("octave")).text)
                    alt_el = pitch_el.find(t("alter"))
                    alter = int(float(alt_el.text)) if alt_el is not None else 0
                    midi_n = (octave + 1) * 12 + _STEP_SEMI[step] + alter
                    events.append((int(onset * scale), midi_n, max(1, int(dur * scale))))
                if chord_el is None: current_time += dur
            elif ltag == "backup": current_time -= int(el.find(t("duration")).text)
            elif ltag == "forward": current_time += int(el.find(t("duration")).text)
    mid = mido.MidiFile(ticks_per_beat=TPQN)
    track = mido.MidiTrack(); mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=mido.bpm2tempo(120)))
    msgs = []
    for onset, pitch, dur in events:
        msgs.append((onset, 1, pitch, 80)); msgs.append((onset + dur, 0, pitch, 0))
    msgs.sort(key=lambda m: (m[0], m[1]))
    prev_tick = 0
    for tick, on, pitch, vel in msgs:
        delta = tick - prev_tick
        mtype = "note_on" if on else "note_off"
        track.append(mido.Message(mtype, note=pitch, velocity=vel, time=delta))
        prev_tick = tick
    mid.save(str(midi_path))
    return len(events)

def _xml_path(sample_id):
    parts = sample_id.split("/")
    return DATASET_ROOT / "samples" / parts[1] / (parts[2] + ".musicxml")

print("MusicXML/MIDI helpers defined")


In [ ]:

import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lxml', 'mido', '-q'])
print("Installed lxml and mido")


In [ ]:

# Cell 21 — MusicXML utilities
from lxml import etree
import mido

_STEP_SEMI = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}

def _xml_tag(name, ns):
    return f"{{{ns}}}{name}" if ns else name

def find_error_note_index(ex, parse_lmx_notes):
    notes, _ = parse_lmx_notes(ex["input_tokens"], ex.get("sample_id", ""))
    return next(i for i, n in enumerate(notes)
               if n["pitch_token_position"] == ex["error_position"])

def set_note_pitch(note_el, pitch_token, ns=""):
    t = lambda name: _xml_tag(name, ns)
    pitch_el = note_el.find(t("pitch"))
    if pitch_el is None: return
    pitch_el.find(t("step")).text   = pitch_token[0]
    pitch_el.find(t("octave")).text = pitch_token[1:]
    alter = pitch_el.find(t("alter"))
    if alter is not None: pitch_el.remove(alter)
    acc = note_el.find(t("accidental"))
    if acc is not None: note_el.remove(acc)

def make_edited_xml(xml_path, note_index, new_pitch):
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    notes = root.findall(f".//{t('note')}")
    set_note_pitch(notes[note_index], new_pitch, ns)
    return tree

def musicxml_to_midi(xml_tree, midi_path):
    root = xml_tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    TPQN = 480; divisions = 1
    events = []; current_time = 0; prev_onset = 0
    for measure in root.findall(f".//{t('measure')}"):
        for attr in measure.findall(t("attributes")):
            d = attr.find(t("divisions"))
            if d is not None: divisions = int(d.text)
        scale = TPQN / divisions
        for el in measure:
            ltag = el.tag.split("}")[-1] if "}" in el.tag else el.tag
            if ltag == "note":
                chord_el = el.find(t("chord")); pitch_el = el.find(t("pitch"))
                dur_el = el.find(t("duration"))
                dur = int(dur_el.text) if dur_el is not None else 0
                onset = prev_onset if chord_el is not None else current_time
                if chord_el is None: prev_onset = current_time
                if pitch_el is not None:
                    step = pitch_el.find(t("step")).text
                    octave = int(pitch_el.find(t("octave")).text)
                    alt_el = pitch_el.find(t("alter"))
                    alter = int(float(alt_el.text)) if alt_el is not None else 0
                    midi_n = (octave + 1) * 12 + _STEP_SEMI[step] + alter
                    events.append((int(onset * scale), midi_n, max(1, int(dur * scale))))
                if chord_el is None: current_time += dur
            elif ltag == "backup": current_time -= int(el.find(t("duration")).text)
            elif ltag == "forward": current_time += int(el.find(t("duration")).text)
    mid = mido.MidiFile(ticks_per_beat=TPQN)
    track = mido.MidiTrack(); mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=mido.bpm2tempo(120)))
    msgs = []
    for onset, pitch, dur in events:
        msgs.append((onset, 1, pitch, 80)); msgs.append((onset + dur, 0, pitch, 0))
    msgs.sort(key=lambda m: (m[0], m[1]))
    prev_tick = 0
    for tick, on, pitch, vel in msgs:
        delta = tick - prev_tick
        mtype = "note_on" if on else "note_off"
        track.append(mido.Message(mtype, note=pitch, velocity=vel, time=delta))
        prev_tick = tick
    mid.save(str(midi_path))
    return len(events)

def _xml_path(sample_id):
    parts = sample_id.split("/")
    return DATASET_ROOT / "samples" / parts[1] / (parts[2] + ".musicxml")

print("MusicXML/MIDI helpers defined")


In [ ]:

# Cell 6 — _pitch_to_cp_idx (needed by cell 24)
_SEMITONES = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}

def _pitch_to_cp_idx(pitch_str):
    if not PITCH_RE.match(pitch_str): return None
    step, octave = pitch_str[0], int(pitch_str[1:])
    midi = (octave + 1) * 12 + _SEMITONES[step]
    idx = midi - 21  # A0 = MIDI 21
    if 0 <= idx < 88: return idx
    return None

print("_pitch_to_cp_idx defined")


In [ ]:

# Cell 24 — Duration data construction (needed for dur_dev_details, _find_dur_span)
import hashlib as _hl
import torch.nn.functional as F

DUR_MASK_ID = CP_TOK2ID['Duration']['Duration <MASK>']  # 65

_DUR_REPL_POOL = [
    (['whole'],          _Frac(1)),
    (['half'],           _Frac(1, 2)),
    (['quarter'],        _Frac(1, 4)),
    (['eighth'],         _Frac(1, 8)),
    (['16th'],           _Frac(1, 16)),
    (['half', 'dot'],    _Frac(3, 4)),
    (['quarter', 'dot'], _Frac(3, 8)),
    (['eighth', 'dot'],  _Frac(3, 16)),
]

def _find_dur_span(tokens, note):
    s, e = note['token_span']
    for i in range(s, e):
        if tokens[i] in _DURATION_BASE:
            j = i + 1
            if j < e and _TIME_MOD_RE.match(tokens[j]):
                j += 1
            while j < e and tokens[j] == 'dot':
                j += 1
            return i, j
    return None, None

def _dur_token_info(tokens, note):
    ds, de = _find_dur_span(tokens, note)
    if ds is None: return None, 0, None
    ntype = tokens[ds]; j = ds + 1
    tup = None
    if j < de and _TIME_MOD_RE.match(tokens[j]):
        tup = tokens[j]; j += 1
    dots = 0
    while j < de and tokens[j] == 'dot':
        dots += 1; j += 1
    return ntype, dots, tup

def build_dur_split(sample_ids, label_map=None):
    build_map = (label_map is None)
    raw = []
    skip = Counter()
    note_excl = Counter()
    for sid in sample_ids:
        lmx_p = _lmx_path(sid)
        if not lmx_p.exists():
            skip['no_lmx'] += 1; continue
        clean_toks = lmx_p.read_text().split()
        notes, _ = parse_lmx_notes(clean_toks, sid)
        cands = []
        for ni, n in enumerate(notes):
            if n['is_grace']:       note_excl['grace'] += 1; continue
            if n['is_rest']:        note_excl['rest'] += 1; continue
            if n['duration'] is None or n['pitch'] is None:
                note_excl['no_dur_or_pitch'] += 1; continue
            if _find_dur_span(clean_toks, n)[0] is None:
                note_excl['dur_tokens_missing'] += 1; continue
            cands.append(ni)
        if not cands:
            skip['no_candidate'] += 1; continue
        h = int(_hl.md5(sid.encode()).hexdigest(), 16)
        sel = cands[h % len(cands)]
        note = notes[sel]
        true_dur = note['duration']
        true_ql  = str(true_dur * 4)
        rng = random.Random(h)
        pool = [(t, f) for t, f in _DUR_REPL_POOL if f != true_dur]
        rtoks, rfrac = pool[rng.randint(0, len(pool) - 1)]
        ds, de = _find_dur_span(clean_toks, note)
        corrupted = clean_toks[:ds] + rtoks + clean_toks[de:]
        ntype, dots, tup = _dur_token_info(clean_toks, note)
        raw.append({
            'sid': sid, 'corrupted': corrupted,
            'epp': note['pitch_token_position'],
            'true_ql': true_ql, 'repl_ql': str(rfrac * 4),
            'ntype': ntype, 'dots': dots, 'tup': tup,
        })
    if build_map:
        ql_counts = Counter(r['true_ql'] for r in raw)
        label_map = {ql: i for i, (ql, _) in enumerate(ql_counts.most_common())}
    oov_qls = sorted({r['true_ql'] for r in raw if r['true_ql'] not in label_map})
    cp_list, details = [], []
    for r in raw:
        if r['true_ql'] not in label_map:
            skip['oov_dur'] += 1; continue
        cnotes, _ = parse_lmx_notes(r['corrupted'], r['sid'])
        err_idx = next((i for i, n in enumerate(cnotes)
                        if n['pitch_token_position'] == r['epp']), None)
        if err_idx is None:
            skip['err_not_found'] += 1; continue
        cp_ids = []
        prev_m = -1
        for i, n in enumerate(cnotes):
            bar = 0 if n['measure'] != prev_m else 1; prev_m = n['measure']
            pos = min(15, int(float(n['onset']) * 16))
            pitch = PITCH_PAD
            if n['pitch'] and PITCH_RE.match(n['pitch']):
                p = _pitch_to_cp_idx(n['pitch'])
                if p is not None: pitch = p
            dur = DUR_MASK_ID if i == err_idx else (
                min(63, int(float(n['duration']) * 32))
                if n['duration'] is not None else DUR_PAD)
            cp_ids.append([bar, pos, pitch, dur])
        cp_list.append({'cp_ids': cp_ids, 'error_note_idx': err_idx,
                        'target_class': label_map[r['true_ql']],
                        'sample_id': r['sid']})
        details.append({'sample_id': r['sid'], 'true_ql': r['true_ql'],
                        'repl_ql': r['repl_ql'], 'note_type': r['ntype'],
                        'dots': r['dots'], 'tuplet': r['tup']})
    stats = {'input_samples': len(sample_ids), 'examples': len(cp_list),
             'skip': dict(skip), 'note_exclusions': dict(note_excl),
             'oov_durations': oov_qls}
    return cp_list, label_map, details, stats

# Build
t0 = time.time()
train_sids = list(dict.fromkeys(ex['sample_id'] for ex in train_raw))
dev_sids   = list(dict.fromkeys(ex['sample_id'] for ex in dev_raw))
print(f"Source samples: train={len(train_sids)}, dev={len(dev_sids)}")

dur_train_cp, dur_label_map, dur_train_details, dur_train_stats = build_dur_split(train_sids)
dur_dev_cp, _, dur_dev_details, dur_dev_stats = build_dur_split(dev_sids, label_map=dur_label_map)
dur_id2label = {v: k for k, v in dur_label_map.items()}
N_DUR_CLASSES = len(dur_label_map)

print(f"Duration classes: {N_DUR_CLASSES}")
print(f"Train: {dur_train_stats['examples']} examples")
print(f"Dev:   {dur_dev_stats['examples']} examples")
print(f"Data construction: {time.time()-t0:.1f}s")


In [ ]:

# ── Cell 33: Unified Correction Demo ──────────────────────────────

import hashlib as _hl
from copy import deepcopy
from fractions import Fraction as _Frac

# ── Part A: Load joint checkpoint + prediction caches ─────────────

JOINT_RUN_DIR = ML_DIR / 'data' / 'runs' / 'm2bert_joint_20261030_213214'
UNIFIED_OUT   = ML_DIR / 'demo' / 'unified_joint'
UNIFIED_OUT.mkdir(parents=True, exist_ok=True)

with open(JOINT_RUN_DIR / 'pitch_dev_predictions.json') as f:
    pitch_preds = json.load(f)
with open(JOINT_RUN_DIR / 'dur_dev_predictions.json') as f:
    dur_preds = json.load(f)

with open(JOINT_RUN_DIR / 'pitch_label_map.json') as f:
    pitch_lm = json.load(f)
with open(JOINT_RUN_DIR / 'dur_label_map.json') as f:
    dur_lm = json.load(f)
id2pitch_j = {v: k for k, v in pitch_lm.items()}
id2dur_j   = {v: k for k, v in dur_lm.items()}

print(f"Loaded predictions: pitch={len(pitch_preds)}, dur={len(dur_preds)}")
print(f"Label maps: pitch={len(pitch_lm)}, dur={len(dur_lm)}")

DEMO_INDICES = [12, 57, 327]

# ── Part B: Duration MusicXML editing utilities ───────────────────

_QL_TO_TYPE = {
    _Frac(8):    'breve',
    _Frac(4):    'whole',
    _Frac(2):    'half',
    _Frac(1):    'quarter',
    _Frac(1,2):  'eighth',
    _Frac(1,4):  '16th',
    _Frac(1,8):  '32nd',
    _Frac(1,16): '64th',
    _Frac(1,32): '128th',
}

def ql_to_musicxml_type(ql_frac):
    if ql_frac in _QL_TO_TYPE:
        return _QL_TO_TYPE[ql_frac], 0, None
    for n_dots in (1, 2, 3):
        factor = 2 - _Frac(1, 2**n_dots)
        if factor != 0:
            base = ql_frac / factor
            if base in _QL_TO_TYPE:
                return _QL_TO_TYPE[base], n_dots, None
    for actual, normal in [(3,2), (5,4), (6,4), (7,4), (3,4)]:
        scaled = ql_frac * _Frac(actual, normal)
        if scaled in _QL_TO_TYPE:
            return _QL_TO_TYPE[scaled], 0, (actual, normal)
        for nd in (1, 2):
            factor = 2 - _Frac(1, 2**nd)
            base = scaled / factor
            if base in _QL_TO_TYPE:
                return _QL_TO_TYPE[base], nd, (actual, normal)
    best_type, best_dist = 'quarter', abs(ql_frac - 1)
    for ql_std, tname in _QL_TO_TYPE.items():
        d = abs(ql_frac - ql_std)
        if d < best_dist:
            best_dist, best_type = d, tname
    return best_type, 0, None


def set_note_duration(note_el, ql_str, divisions, ns=""):
    t = lambda name: _xml_tag(name, ns)
    warnings = []
    ql_frac = _Frac(ql_str)
    xml_dur = int(ql_frac * divisions)
    
    dur_el = note_el.find(t("duration"))
    if dur_el is not None:
        dur_el.text = str(xml_dur)
    
    type_name, n_dots, time_mod = ql_to_musicxml_type(ql_frac)
    
    if time_mod is None and n_dots == 0 and ql_frac not in _QL_TO_TYPE:
        warnings.append(f"ql={ql_str} has no exact MusicXML type; using closest: {type_name}")
    
    type_el = note_el.find(t("type"))
    if type_el is not None:
        type_el.text = type_name
    
    for dot_el in note_el.findall(t("dot")):
        note_el.remove(dot_el)
    if n_dots > 0 and type_el is not None:
        insert_pos = list(note_el).index(type_el) + 1
        for _ in range(n_dots):
            dot = etree.SubElement(note_el, t("dot"))
            note_el.remove(dot)
            note_el.insert(insert_pos, dot)
            insert_pos += 1
    
    tm_el = note_el.find(t("time-modification"))
    if time_mod is not None:
        actual, normal = time_mod
        if tm_el is None:
            tm_el = etree.SubElement(note_el, t("time-modification"))
            note_el.remove(tm_el)
            ref = type_el if type_el is not None else dur_el
            if ref is not None:
                idx = list(note_el).index(ref) + 1 + n_dots
                note_el.insert(idx, tm_el)
        an_el = tm_el.find(t("actual-notes"))
        if an_el is None:
            an_el = etree.SubElement(tm_el, t("actual-notes"))
        an_el.text = str(actual)
        nn_el = tm_el.find(t("normal-notes"))
        if nn_el is None:
            nn_el = etree.SubElement(tm_el, t("normal-notes"))
        nn_el.text = str(normal)
    elif tm_el is not None:
        note_el.remove(tm_el)
    
    return warnings


def make_dur_edited_xml(xml_path, note_index, ql_str):
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    divisions = 1
    attr = root.find(f".//{t('attributes')}")
    if attr is not None:
        d = attr.find(t("divisions"))
        if d is not None:
            divisions = int(d.text)
    notes = root.findall(f".//{t('note')}")
    warns = set_note_duration(notes[note_index], ql_str, divisions, ns)
    return tree, warns

print("Duration MusicXML utilities defined")


In [ ]:

# ── Part C: Find duration error note index in clean XML ───────────

def _find_dur_note_index_clean(sid):
    lmx_p = _lmx_path(sid)
    clean_toks = lmx_p.read_text().split()
    notes, _ = parse_lmx_notes(clean_toks, sid)
    cands = []
    for ni, n in enumerate(notes):
        if n['is_grace'] or n['is_rest']:
            continue
        if n['duration'] is None or n['pitch'] is None:
            continue
        if _find_dur_span(clean_toks, n)[0] is None:
            continue
        cands.append(ni)
    h = int(_hl.md5(sid.encode()).hexdigest(), 16)
    sel = cands[h % len(cands)]
    return sel

# ── Part D: Unified correction entry point ────────────────────────

def correct_sample(error_type, dev_idx, output_dir):
    result = {'error_type': error_type, 'dev_idx': dev_idx, 'warnings': []}
    
    if error_type == 'pitch':
        ex = dev_raw[dev_idx]
        sid = ex['sample_id']
        note_idx = find_error_note_index(ex, parse_lmx_notes)
        error_val = ex['replacement_token']
        truth_val = ex['target_token']
        pred = pitch_preds[dev_idx]
        pred_val = pred['predicted']
        correct = pred['correct']
        
        xml_src = _xml_path(sid)
        sdir = output_dir / f"pitch_{dev_idx}_{sid.replace('/', '_')}"
        sdir.mkdir(parents=True, exist_ok=True)
        
        # Ground truth
        gt_tree = etree.parse(str(xml_src))
        gt_xml = sdir / "ground_truth.musicxml"
        gt_tree.write(str(gt_xml), xml_declaration=True, encoding="UTF-8")
        gt_midi = sdir / "ground_truth.mid"
        musicxml_to_midi(gt_tree, gt_midi)
        
        # Corrupted
        corr_tree = make_edited_xml(xml_src, note_idx, error_val)
        corr_xml = sdir / "corrupted.musicxml"
        corr_tree.write(str(corr_xml), xml_declaration=True, encoding="UTF-8")
        corr_midi = sdir / "corrupted.mid"
        musicxml_to_midi(corr_tree, corr_midi)
        
        # Model corrected
        pred_tree = make_edited_xml(xml_src, note_idx, pred_val)
        pred_xml = sdir / "model_corrected.musicxml"
        pred_tree.write(str(pred_xml), xml_declaration=True, encoding="UTF-8")
        pred_midi = sdir / "model_corrected.mid"
        musicxml_to_midi(pred_tree, pred_midi)
        
        # Verify XML
        v_tree = etree.parse(str(pred_xml))
        v_ns = v_tree.getroot().nsmap.get(None, "")
        v_t = lambda name: _xml_tag(name, v_ns)
        v_notes = v_tree.getroot().findall(f".//{v_t('note')}")
        v_note = v_notes[note_idx]
        v_pitch = v_note.find(v_t("pitch"))
        if v_pitch is not None:
            v_step = v_pitch.find(v_t("step")).text
            v_oct = v_pitch.find(v_t("octave")).text
            xml_verified = (v_step + v_oct) == pred_val
        else:
            xml_verified = False
            result['warnings'].append("No pitch element in verified note")
        
        # Verify MIDI
        mid = mido.MidiFile(str(pred_midi))
        midi_notes = []
        tick = 0
        for msg in mid.tracks[0]:
            tick += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                midi_notes.append((tick, msg.note))
        midi_verified = len(midi_notes) > 0
        
    elif error_type == 'duration':
        det = dur_dev_details[dev_idx]
        sid = det['sample_id']
        note_idx = _find_dur_note_index_clean(sid)
        error_val = det['repl_ql']
        truth_val = det['true_ql']
        pred = dur_preds[dev_idx]
        pred_val = pred['predicted']
        correct = pred['correct']
        
        # Convert pred from whole-note label to quarterLength
        pred_ql = str(_Frac(pred_val) * 4)
        truth_ql = truth_val
        error_ql = error_val
        
        xml_src = _xml_path(sid)
        sdir = output_dir / f"dur_{dev_idx}_{sid.replace('/', '_')}"
        sdir.mkdir(parents=True, exist_ok=True)
        
        # Ground truth
        gt_tree = etree.parse(str(xml_src))
        gt_xml = sdir / "ground_truth.musicxml"
        gt_tree.write(str(gt_xml), xml_declaration=True, encoding="UTF-8")
        gt_midi = sdir / "ground_truth.mid"
        musicxml_to_midi(gt_tree, gt_midi)
        
        # Corrupted
        corr_tree, corr_warns = make_dur_edited_xml(xml_src, note_idx, error_ql)
        corr_xml = sdir / "corrupted.musicxml"
        corr_tree.write(str(corr_xml), xml_declaration=True, encoding="UTF-8")
        corr_midi = sdir / "corrupted.mid"
        musicxml_to_midi(corr_tree, corr_midi)
        result['warnings'].extend(corr_warns)
        
        # Model corrected
        pred_tree, pred_warns = make_dur_edited_xml(xml_src, note_idx, pred_ql)
        pred_xml = sdir / "model_corrected.musicxml"
        pred_tree.write(str(pred_xml), xml_declaration=True, encoding="UTF-8")
        pred_midi = sdir / "model_corrected.mid"
        musicxml_to_midi(pred_tree, pred_midi)
        result['warnings'].extend(pred_warns)
        
        # Verify XML
        v_tree = etree.parse(str(pred_xml))
        v_ns = v_tree.getroot().nsmap.get(None, "")
        v_t = lambda name: _xml_tag(name, v_ns)
        v_attr = v_tree.getroot().find(f".//{v_t('attributes')}")
        v_div = 1
        if v_attr is not None:
            d_el = v_attr.find(v_t("divisions"))
            if d_el is not None:
                v_div = int(d_el.text)
        v_notes = v_tree.getroot().findall(f".//{v_t('note')}")
        v_dur_el = v_notes[note_idx].find(v_t("duration"))
        expected_dur = int(_Frac(pred_ql) * v_div)
        actual_dur = int(v_dur_el.text) if v_dur_el is not None else -1
        xml_verified = (actual_dur == expected_dur)
        if not xml_verified:
            result['warnings'].append(
                f"XML duration mismatch: expected {expected_dur}, got {actual_dur}")
        
        # Verify MIDI
        mid = mido.MidiFile(str(pred_midi))
        midi_notes = []
        tick = 0
        for msg in mid.tracks[0]:
            tick += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                midi_notes.append((tick, msg.note))
        midi_verified = len(midi_notes) > 0
        
        pred_val = pred_ql
        truth_val = truth_ql
        error_val = error_ql
    
    result.update({
        'sample_id': sid, 'note_idx': note_idx,
        'error_val': error_val, 'truth_val': truth_val,
        'pred_val': pred_val, 'correct': correct,
        'xml_verified': xml_verified, 'midi_verified': midi_verified,
        'dir': str(sdir),
    })
    return result

print("correct_sample defined")


In [ ]:

# ── Part E: Run 6 demos ──────────────────────────────────────────

print(f"\n{'='*90}")
print(f"  UNIFIED CORRECTION DEMO — 6 samples (3 pitch + 3 duration)")
print(f"{'='*90}")

all_results = []

# Pitch samples
print(f"\n  PITCH corrections:")
for idx in DEMO_INDICES:
    r = correct_sample('pitch', idx, UNIFIED_OUT)
    all_results.append(r)
    mark = "ok" if r['correct'] else "X"
    print(f"  [{idx:3d}] {r['sample_id']:>28s}  "
          f"err={r['error_val']:>4s} truth={r['truth_val']:>4s} "
          f"pred={r['pred_val']:>4s} [{mark}]  "
          f"xml={'ok' if r['xml_verified'] else 'FAIL'}  "
          f"midi={'ok' if r['midi_verified'] else 'FAIL'}"
          f"{'  warn: '+'; '.join(r['warnings']) if r['warnings'] else ''}")

# Duration samples
print(f"\n  DURATION corrections:")
for idx in DEMO_INDICES:
    r = correct_sample('duration', idx, UNIFIED_OUT)
    all_results.append(r)
    mark = "ok" if r['correct'] else "X"
    print(f"  [{idx:3d}] {r['sample_id']:>28s}  "
          f"err={r['error_val']:>8s} truth={r['truth_val']:>8s} "
          f"pred={r['pred_val']:>8s} [{mark}]  "
          f"xml={'ok' if r['xml_verified'] else 'FAIL'}  "
          f"midi={'ok' if r['midi_verified'] else 'FAIL'}"
          f"{'  warn: '+'; '.join(r['warnings']) if r['warnings'] else ''}")


In [ ]:

# ── Part F: Summary table + output ───────────────────────────────

print(f"\n{'='*90}")
print(f"  {'Type':>8s} {'Idx':>4s} {'Sample':>28s} {'Note':>5s} "
      f"{'Error':>8s} {'Truth':>8s} {'Pred':>8s} {'OK':>3s} "
      f"{'XML':>4s} {'MIDI':>5s} {'Warnings'}")
print(f"  {'—'*88}")
for r in all_results:
    mark = "ok" if r['correct'] else "X"
    print(f"  {r['error_type']:>8s} {r['dev_idx']:4d} {r['sample_id']:>28s} "
          f"{r['note_idx']:5d} {r['error_val']:>8s} {r['truth_val']:>8s} "
          f"{r['pred_val']:>8s}  {mark:>2s} "
          f"{'ok' if r['xml_verified'] else 'FAIL':>4s} "
          f"{'ok' if r['midi_verified'] else 'FAIL':>5s} "
          f"{'; '.join(r['warnings']) if r['warnings'] else '—'}")

# Save summary
summary_out = []
for r in all_results:
    summary_out.append({
        'error_type': r['error_type'], 'dev_idx': r['dev_idx'],
        'sample_id': r['sample_id'], 'note_idx': r['note_idx'],
        'error_val': r['error_val'], 'truth_val': r['truth_val'],
        'pred_val': r['pred_val'], 'correct': r['correct'],
        'xml_verified': r['xml_verified'], 'midi_verified': r['midi_verified'],
        'warnings': r['warnings'], 'dir': r['dir'],
    })

with open(UNIFIED_OUT / 'demo_summary.json', 'w') as f:
    json.dump(summary_out, f, indent=2)

sample_ids_out = {r['error_type'] + '_' + str(r['dev_idx']): r['sample_id']
                  for r in all_results}
with open(UNIFIED_OUT / 'demo_sample_ids.json', 'w') as f:
    json.dump(sample_ids_out, f, indent=2)

n_correct = sum(1 for r in all_results if r['correct'])
n_xml_ok  = sum(1 for r in all_results if r['xml_verified'])
n_midi_ok = sum(1 for r in all_results if r['midi_verified'])
n_warns   = sum(len(r['warnings']) for r in all_results)

print(f"\n  Correct: {n_correct}/6  XML verified: {n_xml_ok}/6  "
      f"MIDI verified: {n_midi_ok}/6  Warnings: {n_warns}")
print(f"  Output: {UNIFIED_OUT.relative_to(ML_DIR)}/")
print(f"{'='*90}")


In [ ]:

# Cell 35 — Standalone Inference Pipeline
import time as _time
from fractions import Fraction as _Frac
from lxml import etree
import mido

# 1. Model class definitions
class _LUT(nn.Module):
    def __init__(self, n, d, pad=None):
        super().__init__()
        self.lut = nn.Embedding(n, d, padding_idx=pad)
    def forward(self, x): return self.lut(x)

class _MidiModernBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.word_emb = nn.ModuleList([
            _LUT(4,  256, pad=BAR_PAD),
            _LUT(18, 256, pad=POS_PAD),
            _LUT(88, 256, pad=PITCH_PAD),
            _LUT(66, 256, pad=DUR_PAD),
        ])
        self.in_linear = nn.Linear(1024, 768)
        _cfg = ModernBertConfig(
            vocab_size=10, hidden_size=768, num_hidden_layers=12,
            num_attention_heads=12, intermediate_size=1152,
            max_position_embeddings=4096, pad_token_id=0,
        )
        self.note_bert1 = ModernBertModel(_cfg)

    def forward(self, cp_ids, attention_mask=None):
        embs = torch.cat([self.word_emb[i](cp_ids[:, :, i])
                          for i in range(4)], dim=-1)
        embs = self.in_linear(embs)
        out  = self.note_bert1(inputs_embeds=embs, attention_mask=attention_mask)
        return out.last_hidden_state

class _JointModel(nn.Module):
    def __init__(self, backbone, n_pitch, n_dur):
        super().__init__()
        self.backbone   = backbone
        self.pitch_head = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_pitch))
        self.dur_head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_dur))

    def forward(self, cp_ids, error_note_idx, attention_mask=None, task='pitch'):
        hidden = self.backbone(cp_ids, attention_mask)
        err_h  = hidden[torch.arange(hidden.size(0)), error_note_idx]
        if task == 'pitch':
            return self.pitch_head(err_h)
        return self.dur_head(err_h)

# 2. Load joint checkpoint
_JOINT_CKPT_PATH = ML_DIR / 'data' / 'runs' / 'm2bert_joint_20261030_213214' / 'best_checkpoint.pt'
_ckpt = torch.load(_JOINT_CKPT_PATH, map_location='cpu', weights_only=False)

_pitch_classes  = _ckpt['pitch_classes']
_dur_label_map  = _ckpt['dur_label_map']
_id2pitch       = {v: k for k, v in _pitch_classes.items()}
_id2dur         = {v: k for k, v in _dur_label_map.items()}
_N_PITCH        = _ckpt['n_pitch_classes']
_N_DUR          = _ckpt['n_dur_classes']

_backbone = _MidiModernBERT()
_pretrained_ckpt = torch.load(PRETRAINED / 'model_best.ckpt',
                              map_location='cpu', weights_only=False)
_backbone.load_state_dict(_pretrained_ckpt['state_dict'], strict=True)
del _pretrained_ckpt

_infer_model = _JointModel(_backbone, _N_PITCH, _N_DUR)
_infer_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
_infer_model.to(DEVICE)
_infer_model.eval()
del _ckpt

print(f"Joint model loaded: pitch={_N_PITCH} classes, dur={_N_DUR} classes, device={DEVICE}")


In [ ]:

from transformers import ModernBertConfig, ModernBertModel
print("transformers imported")


In [ ]:

# Re-run model loading now that ModernBertConfig is imported
_JOINT_CKPT_PATH = ML_DIR / 'data' / 'runs' / 'm2bert_joint_20261030_213214' / 'best_checkpoint.pt'
_ckpt = torch.load(_JOINT_CKPT_PATH, map_location='cpu', weights_only=False)

_pitch_classes  = _ckpt['pitch_classes']
_dur_label_map  = _ckpt['dur_label_map']
_id2pitch       = {v: k for k, v in _pitch_classes.items()}
_id2dur         = {v: k for k, v in _dur_label_map.items()}
_N_PITCH        = _ckpt['n_pitch_classes']
_N_DUR          = _ckpt['n_dur_classes']

_backbone = _MidiModernBERT()
_pretrained_ckpt = torch.load(PRETRAINED / 'model_best.ckpt',
                              map_location='cpu', weights_only=False)
_backbone.load_state_dict(_pretrained_ckpt['state_dict'], strict=True)
del _pretrained_ckpt

_infer_model = _JointModel(_backbone, _N_PITCH, _N_DUR)
_infer_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
_infer_model.to(DEVICE)
_infer_model.eval()
del _ckpt

print(f"Joint model loaded: pitch={_N_PITCH} classes, dur={_N_DUR} classes, device={DEVICE}")


In [ ]:

# 3. MusicXML parser
_PITCH_RE = re.compile(r'^[A-G][0-9]$')
_SEMI = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}

def _xt(name, ns):
    return f"{{{ns}}}{name}" if ns else name

def _parse_xml_notes(xml_path, part_id=None):
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    ns = root.nsmap.get(None, "")
    t = lambda name: _xt(name, ns)

    notes = []
    for part_el in root.findall(f".//{t('part')}"):
        pid = part_el.get('id')
        if part_id is not None and pid != part_id:
            continue

        divisions = 1
        measures = part_el.findall(t('measure'))

        for mi, meas in enumerate(measures):
            for attr in meas.findall(t('attributes')):
                d_el = attr.find(t('divisions'))
                if d_el is not None:
                    divisions = int(d_el.text)

            onset = _Frac(0)
            prev_onset = _Frac(0)
            ni_in_meas = 0

            for el in meas:
                ltag = el.tag.split('}')[-1] if '}' in el.tag else el.tag

                if ltag == 'note':
                    pitch_el = el.find(t('pitch'))
                    rest_el = el.find(t('rest'))
                    grace_el = el.find(t('grace'))
                    chord_el = el.find(t('chord'))
                    dur_el = el.find(t('duration'))

                    is_grace = grace_el is not None
                    is_chord = chord_el is not None
                    is_rest = rest_el is not None

                    pitch_str = None
                    if pitch_el is not None:
                        step = pitch_el.find(t('step')).text
                        octave = pitch_el.find(t('octave')).text
                        pitch_str = step + octave

                    duration = None
                    xml_dur_val = 0
                    if dur_el is not None:
                        xml_dur_val = int(dur_el.text)
                        duration = _Frac(xml_dur_val, divisions * 4)

                    note_onset = prev_onset if is_chord else onset
                    if not is_chord:
                        prev_onset = onset

                    voice_el = el.find(t('voice'))
                    staff_el = el.find(t('staff'))
                    voice = voice_el.text if voice_el is not None else None
                    staff = staff_el.text if staff_el is not None else None

                    notes.append({
                        'part_id': pid,
                        'measure_index': mi,
                        'note_in_measure': ni_in_meas,
                        'pitch': pitch_str,
                        'is_rest': is_rest,
                        'is_grace': is_grace,
                        'is_chord': is_chord,
                        'duration': duration,
                        'onset': note_onset,
                        'measure_num': mi + 1,
                        'voice': voice,
                        'staff': staff,
                        'xml_note_element': el,
                        'divisions': divisions,
                    })
                    ni_in_meas += 1

                    if not is_chord and not is_grace and dur_el is not None:
                        onset += _Frac(xml_dur_val, divisions * 4)

                elif ltag == 'backup':
                    bd = el.find(t('duration'))
                    if bd is not None:
                        onset -= _Frac(int(bd.text), divisions * 4)
                        onset = max(_Frac(0), onset)
                elif ltag == 'forward':
                    fd = el.find(t('duration'))
                    if fd is not None:
                        onset += _Frac(int(fd.text), divisions * 4)

    return notes, tree, ns

# 4. CP conversion
def _pitch_to_cp(pitch_str):
    if not _PITCH_RE.match(pitch_str):
        return None
    midi = (int(pitch_str[1]) + 1) * 12 + _SEMI[pitch_str[0]]
    idx = midi - 22
    return idx if 0 <= idx <= 85 else None

_MAX_CP_LEN = 1024

def _build_cp_for_inference(parsed_notes, target_flat_idx, error_type):
    cp_ids = []
    cp_target_idx = None
    prev_meas = -1

    for i, n in enumerate(parsed_notes):
        bar = 0 if n['measure_num'] != prev_meas else 1
        prev_meas = n['measure_num']

        pos = min(15, int(float(n['onset']) * 16))

        if i == target_flat_idx and error_type == 'pitch':
            pitch = PITCH_MASK
        elif n['pitch'] and _PITCH_RE.match(n['pitch']):
            p = _pitch_to_cp(n['pitch'])
            pitch = p if p is not None else PITCH_PAD
        else:
            pitch = PITCH_PAD

        DUR_MASK = CP_TOK2ID['Duration']['Duration <MASK>']
        if i == target_flat_idx and error_type == 'duration':
            dur = DUR_MASK
        elif n['duration'] is not None:
            dur = min(63, int(float(n['duration']) * 32))
        else:
            dur = DUR_PAD

        cp_ids.append([bar, pos, pitch, dur])
        if i == target_flat_idx:
            cp_target_idx = len(cp_ids) - 1

    if cp_target_idx is None:
        raise ValueError("Target note not found in CP sequence")

    if len(cp_ids) > _MAX_CP_LEN:
        half = _MAX_CP_LEN // 2
        start = max(0, cp_target_idx - half)
        end = start + _MAX_CP_LEN
        if end > len(cp_ids):
            end = len(cp_ids)
            start = end - _MAX_CP_LEN
        cp_ids = cp_ids[start:end]
        cp_target_idx -= start

    return cp_ids, cp_target_idx

print("Parser and CP builder defined")


In [ ]:

# 5. MusicXML editing + MIDI export
_STEP_SEMI = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}

_QL_TO_XML_TYPE = {
    _Frac(8): 'breve', _Frac(4): 'whole', _Frac(2): 'half',
    _Frac(1): 'quarter', _Frac(1,2): 'eighth', _Frac(1,4): '16th',
    _Frac(1,8): '32nd', _Frac(1,16): '64th', _Frac(1,32): '128th',
}

def _ql_to_xml_type(ql_frac):
    if ql_frac in _QL_TO_XML_TYPE:
        return _QL_TO_XML_TYPE[ql_frac], 0, None
    for nd in (1, 2, 3):
        factor = 2 - _Frac(1, 2**nd)
        base = ql_frac / factor
        if base in _QL_TO_XML_TYPE:
            return _QL_TO_XML_TYPE[base], nd, None
    for act, nrm in [(3,2), (5,4), (6,4), (7,4), (3,4)]:
        scaled = ql_frac * _Frac(act, nrm)
        if scaled in _QL_TO_XML_TYPE:
            return _QL_TO_XML_TYPE[scaled], 0, (act, nrm)
        for nd in (1, 2):
            factor = 2 - _Frac(1, 2**nd)
            base = scaled / factor
            if base in _QL_TO_XML_TYPE:
                return _QL_TO_XML_TYPE[base], nd, (act, nrm)
    best_type, best_dist = 'quarter', abs(ql_frac - 1)
    for ql_std, tname in _QL_TO_XML_TYPE.items():
        d = abs(ql_frac - ql_std)
        if d < best_dist:
            best_dist, best_type = d, tname
    return best_type, 0, None

def _apply_pitch(note_el, pitch_str, ns):
    t = lambda name: _xt(name, ns)
    pitch_el = note_el.find(t("pitch"))
    if pitch_el is None: return
    pitch_el.find(t("step")).text   = pitch_str[0]
    pitch_el.find(t("octave")).text = pitch_str[1:]
    alter = pitch_el.find(t("alter"))
    if alter is not None: pitch_el.remove(alter)
    acc = note_el.find(t("accidental"))
    if acc is not None: note_el.remove(acc)

def _apply_duration(note_el, ql_str, divisions, ns):
    t = lambda name: _xt(name, ns)
    warnings = []
    ql_frac = _Frac(ql_str)
    xml_dur = int(ql_frac * divisions)
    dur_el = note_el.find(t("duration"))
    if dur_el is not None:
        dur_el.text = str(xml_dur)
    type_name, n_dots, time_mod = _ql_to_xml_type(ql_frac)
    if time_mod is None and n_dots == 0 and ql_frac not in _QL_TO_XML_TYPE:
        warnings.append(f"ql={ql_str} has no exact MusicXML type; closest: {type_name}")
    type_el = note_el.find(t("type"))
    if type_el is not None:
        type_el.text = type_name
    for dot_el in note_el.findall(t("dot")):
        note_el.remove(dot_el)
    if n_dots > 0 and type_el is not None:
        ins = list(note_el).index(type_el) + 1
        for _ in range(n_dots):
            dot = etree.SubElement(note_el, t("dot"))
            note_el.remove(dot)
            note_el.insert(ins, dot)
            ins += 1
    tm_el = note_el.find(t("time-modification"))
    if time_mod is not None:
        act, nrm = time_mod
        if tm_el is None:
            tm_el = etree.SubElement(note_el, t("time-modification"))
            note_el.remove(tm_el)
            ref = type_el if type_el is not None else dur_el
            if ref is not None:
                idx = list(note_el).index(ref) + 1 + n_dots
                note_el.insert(idx, tm_el)
        an = tm_el.find(t("actual-notes"))
        if an is None: an = etree.SubElement(tm_el, t("actual-notes"))
        an.text = str(act)
        nn = tm_el.find(t("normal-notes"))
        if nn is None: nn = etree.SubElement(tm_el, t("normal-notes"))
        nn.text = str(nrm)
    elif tm_el is not None:
        note_el.remove(tm_el)
    return warnings

def _xml_to_midi(xml_tree, midi_path, ns):
    root = xml_tree.getroot()
    t = lambda name: _xt(name, ns)
    TPQN = 480; divisions = 1
    events = []; current_time = 0; prev_onset = 0
    for measure in root.findall(f".//{t('measure')}"):
        for attr in measure.findall(t("attributes")):
            d = attr.find(t("divisions"))
            if d is not None: divisions = int(d.text)
        scale = TPQN / divisions
        for el in measure:
            ltag = el.tag.split("}")[-1] if "}" in el.tag else el.tag
            if ltag == "note":
                chord_el = el.find(t("chord")); pitch_el = el.find(t("pitch"))
                dur_el = el.find(t("duration"))
                dur = int(dur_el.text) if dur_el is not None else 0
                onset = prev_onset if chord_el is not None else current_time
                if chord_el is None: prev_onset = current_time
                if pitch_el is not None:
                    step = pitch_el.find(t("step")).text
                    octave = int(pitch_el.find(t("octave")).text)
                    alt_el = pitch_el.find(t("alter"))
                    alter = int(float(alt_el.text)) if alt_el is not None else 0
                    midi_n = (octave + 1) * 12 + _STEP_SEMI[step] + alter
                    events.append((int(onset * scale), midi_n, max(1, int(dur * scale))))
                if chord_el is None: current_time += dur
            elif ltag == "backup": current_time -= int(el.find(t("duration")).text)
            elif ltag == "forward": current_time += int(el.find(t("duration")).text)
    mid = mido.MidiFile(ticks_per_beat=TPQN)
    track = mido.MidiTrack(); mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=mido.bpm2tempo(120)))
    msgs = []
    for onset, pitch, dur in events:
        msgs.append((onset, 1, pitch, 80)); msgs.append((onset + dur, 0, pitch, 0))
    msgs.sort(key=lambda m: (m[0], m[1]))
    prev_tick = 0
    for tick, on, pitch, vel in msgs:
        delta = tick - prev_tick
        mtype = "note_on" if on else "note_off"
        track.append(mido.Message(mtype, note=pitch, velocity=vel, time=delta))
        prev_tick = tick
    mid.save(str(midi_path))
    return len(events)

print("Writers defined")


In [ ]:

# 6. Public API

def list_notes(input_path):
    notes, _, _ = _parse_xml_notes(input_path)
    print(f"{'flat':>5s} {'part':>5s} {'meas':>5s} {'note':>5s} "
          f"{'voice':>5s} {'staff':>5s} {'pitch':>6s} {'dur_QL':>8s} {'flags'}")
    print("-" * 65)
    for i, n in enumerate(notes):
        flags = []
        if n['is_rest']:  flags.append('rest')
        if n['is_grace']: flags.append('grace')
        if n['is_chord']: flags.append('chord')
        ql = str(n['duration'] * 4) if n['duration'] is not None else '?'
        print(f"{i:5d} {n['part_id']:>5s} {n['measure_index']:5d} "
              f"{n['note_in_measure']:5d} "
              f"{n['voice'] or '-':>5s} {n['staff'] or '-':>5s} "
              f"{n['pitch'] or 'rest':>6s} {ql:>8s} "
              f"{' '.join(flags)}")
    print(f"\nTotal: {len(notes)} notes")
    return notes


def correct_musicxml(input_path, target_locator, error_type,
                     output_dir, top_k=None):
    t0 = _time.monotonic()
    warnings = []
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if top_k is None:
        top_k = 5 if error_type == 'pitch' else 3

    pid = target_locator['part_id']
    mi  = target_locator['measure_index']
    ni  = target_locator['note_index']

    # Parse all notes in the target part
    parsed, tree, ns = _parse_xml_notes(input_path, part_id=pid)
    if not parsed:
        raise ValueError(f"No notes found in part {pid}")

    target_flat = None
    for fi, n in enumerate(parsed):
        if (n['part_id'] == pid and n['measure_index'] == mi
                and n['note_in_measure'] == ni):
            target_flat = fi
            break
    if target_flat is None:
        raise ValueError(f"Note not found: part={pid}, measure={mi}, note={ni}")

    tn = parsed[target_flat]
    if error_type == 'pitch':
        current_val = tn['pitch'] or 'rest'
    else:
        current_val = str(tn['duration'] * 4) if tn['duration'] is not None else '?'

    # Build CP and run inference
    cp_ids, cp_err_idx = _build_cp_for_inference(parsed, target_flat, error_type)

    cp_t  = torch.tensor([cp_ids], dtype=torch.long, device=DEVICE)
    ei_t  = torch.tensor([cp_err_idx], dtype=torch.long, device=DEVICE)
    attn  = torch.ones(1, len(cp_ids), dtype=torch.long, device=DEVICE)

    with torch.inference_mode():
        logits = _infer_model(cp_t, ei_t, attn,
                              task='pitch' if error_type == 'pitch' else 'dur')
        probs = logits.softmax(-1).cpu()
        topk = probs.topk(min(top_k, probs.size(-1)), dim=-1)

    id2label = _id2pitch if error_type == 'pitch' else _id2dur

    candidates = []
    for k in range(topk.indices.size(1)):
        cls_id = topk.indices[0, k].item()
        label = id2label[cls_id]
        prob  = topk.values[0, k].item()
        candidates.append({'label': label, 'prob': round(prob, 4)})

    pred_label = candidates[0]['label']

    if error_type == 'duration':
        pred_ql = str(_Frac(pred_label) * 4)
        cand_display = [{'label': str(_Frac(c['label']) * 4) + ' QL',
                         'raw': c['label'], 'prob': c['prob']}
                        for c in candidates]
    else:
        pred_ql = None
        cand_display = candidates

    # Write corrected MusicXML
    edit_tree = etree.parse(str(input_path))
    edit_root = edit_tree.getroot()
    edit_ns   = edit_root.nsmap.get(None, "")
    et = lambda name: _xt(name, edit_ns)

    target_part = None
    for p in edit_root.findall(f".//{et('part')}"):
        if p.get('id') == pid:
            target_part = p; break
    measures = target_part.findall(et('measure'))
    target_meas = measures[mi]
    note_els = [el for el in target_meas
                if (el.tag.split('}')[-1] if '}' in el.tag else el.tag) == 'note']
    target_note_el = note_els[ni]

    if error_type == 'pitch':
        _apply_pitch(target_note_el, pred_label, edit_ns)
    else:
        divs = 1
        for m in measures[:mi+1]:
            for attr in m.findall(et('attributes')):
                d_el = attr.find(et('divisions'))
                if d_el is not None:
                    divs = int(d_el.text)
        w = _apply_duration(target_note_el, pred_ql, divs, edit_ns)
        warnings.extend(w)

    xml_out = output_dir / 'model_corrected.musicxml'
    edit_tree.write(str(xml_out), xml_declaration=True, encoding='UTF-8')

    # Write MIDI
    midi_out = output_dir / 'model_corrected.mid'
    n_events = _xml_to_midi(edit_tree, midi_out, edit_ns)

    # Verify written XML
    v_tree = etree.parse(str(xml_out))
    v_root = v_tree.getroot()
    v_ns   = v_root.nsmap.get(None, "")
    vt     = lambda name: _xt(name, v_ns)
    v_part = None
    for p in v_root.findall(f".//{vt('part')}"):
        if p.get('id') == pid:
            v_part = p; break
    v_meas = v_part.findall(vt('measure'))[mi]
    v_notes = [el for el in v_meas
               if (el.tag.split('}')[-1] if '}' in el.tag else el.tag) == 'note']
    v_note = v_notes[ni]

    if error_type == 'pitch':
        vp = v_note.find(vt('pitch'))
        if vp is not None:
            v_val = vp.find(vt('step')).text + vp.find(vt('octave')).text
            xml_ok = v_val == pred_label
        else:
            xml_ok = False
            warnings.append("Verified note has no pitch element")
    else:
        vd = v_note.find(vt('duration'))
        divs_v = 1
        for m in v_part.findall(vt('measure'))[:mi+1]:
            for attr in m.findall(vt('attributes')):
                d_el = attr.find(vt('divisions'))
                if d_el is not None:
                    divs_v = int(d_el.text)
        expected = int(_Frac(pred_ql) * divs_v)
        actual = int(vd.text) if vd is not None else -1
        xml_ok = actual == expected
        if not xml_ok:
            warnings.append(f"Duration verify: expected {expected}, got {actual}")

    elapsed = _time.monotonic() - t0

    result = {
        'input_file': str(input_path),
        'target_locator': target_locator,
        'error_type': error_type,
        'current_value': current_val,
        'prediction': pred_label if error_type == 'pitch' else pred_ql,
        'candidates': cand_display,
        'xml_verified': xml_ok,
        'midi_events': n_events,
        'checkpoint': str(_JOINT_CKPT_PATH),
        'context_notes': len(cp_ids),
        'elapsed_s': round(elapsed, 3),
        'warnings': warnings,
        'output_xml': str(xml_out),
        'output_midi': str(midi_out),
    }

    with open(output_dir / 'prediction.json', 'w') as f:
        json.dump(result, f, indent=2)

    return result

print("API ready: list_notes(), correct_musicxml()")


In [ ]:

# Cell 36 — Demo: run inference on two corrupted MusicXML files
from shutil import copyfile

INFER_DEMO = ML_DIR / 'demo' / 'inference_demo'
INFER_DEMO.mkdir(parents=True, exist_ok=True)

# Copy corrupted files with neutral names
pitch_src = ML_DIR / 'demo' / 'unified_joint' / 'pitch_12_samples_5023603_p4-s3' / 'corrupted.musicxml'
dur_src   = ML_DIR / 'demo' / 'unified_joint' / 'dur_327_samples_6474469_p1-s1' / 'corrupted.musicxml'

pitch_input = INFER_DEMO / 'pitch_test_input.musicxml'
dur_input   = INFER_DEMO / 'dur_test_input.musicxml'
copyfile(pitch_src, pitch_input)
copyfile(dur_src, dur_input)
print(f"Copied test inputs to {INFER_DEMO.relative_to(ML_DIR)}/")


In [ ]:

# Case 1: Pitch correction
print(f"\n{'='*70}")
print("  CASE 1: Pitch correction")
print(f"{'='*70}")

pitch_result = correct_musicxml(
    input_path=pitch_input,
    target_locator={'part_id': 'P2', 'measure_index': 0, 'note_index': 5},
    error_type='pitch',
    output_dir=INFER_DEMO / 'pitch_output',
)

print(f"\n  Input file:  {pitch_input.name}")
print(f"  Target:      part=P2, measure=0, note=5")
print(f"  Current:     {pitch_result['current_value']}")
print(f"  Prediction:  {pitch_result['prediction']}")
print(f"  Candidates:  {pitch_result['candidates']}")
print(f"  XML ok:      {pitch_result['xml_verified']}")
print(f"  MIDI events: {pitch_result['midi_events']}")
print(f"  Context:     {pitch_result['context_notes']} notes")
print(f"  Time:        {pitch_result['elapsed_s']:.3f}s")
if pitch_result['warnings']:
    print(f"  Warnings:    {pitch_result['warnings']}")

# Ground-truth check (outside the interface)
print(f"\n  [Validation] Ground truth = E4, predicted = {pitch_result['prediction']}"
      f" -> {'MATCH' if pitch_result['prediction'] == 'E4' else 'MISMATCH'}")


In [ ]:

# Case 2: Duration correction
print(f"\n{'='*70}")
print("  CASE 2: Duration correction")
print(f"{'='*70}")

dur_result = correct_musicxml(
    input_path=dur_input,
    target_locator={'part_id': 'P2', 'measure_index': 3, 'note_index': 24},
    error_type='duration',
    output_dir=INFER_DEMO / 'dur_output',
)

print(f"\n  Input file:  {dur_input.name}")
print(f"  Target:      part=P2, measure=3, note=24")
print(f"  Current:     {dur_result['current_value']} QL")
print(f"  Prediction:  {dur_result['prediction']} QL")
print(f"  Candidates:  {dur_result['candidates']}")
print(f"  XML ok:      {dur_result['xml_verified']}")
print(f"  MIDI events: {dur_result['midi_events']}")
print(f"  Context:     {dur_result['context_notes']} notes")
print(f"  Time:        {dur_result['elapsed_s']:.3f}s")
if dur_result['warnings']:
    print(f"  Warnings:    {dur_result['warnings']}")

# Ground-truth check (outside the interface)
# Truth = 1/2 QL. The dur label map: "1/8" whole-note = 1/2 QL
print(f"\n  [Validation] Ground truth = 1/2 QL,"
      f" predicted = {dur_result['prediction']} QL"
      f" -> {'MATCH' if dur_result['prediction'] == '1/2' else 'no match (truth=1/2 QL)'}")


In [ ]:

# Verify: dur_label_map keys are QL, not whole-note
# "1/2" = eighth note (0.5 QL), "1" = quarter (1 QL), "2" = half (2 QL)
print("Label map sample:")
for k, v in sorted(_dur_label_map.items(), key=lambda x: x[1])[:5]:
    print(f"  class {v}: label='{k}' (QL)")

# The build code does: true_ql = str(true_dur * 4) where true_dur is whole-note Fraction
# So labels ARE already QL. The * 4 conversion in correct_musicxml is wrong.
print("\nBug confirmed: labels are QL, not whole-note units.")
print("_Frac('1/2') * 4 =", _Frac('1/2') * 4, "but '1/2' already means 0.5 QL")


In [ ]:

# Fix: redefine correct_musicxml with duration label used directly as QL

def correct_musicxml(input_path, target_locator, error_type,
                     output_dir, top_k=None):
    t0 = _time.monotonic()
    warnings = []
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if top_k is None:
        top_k = 5 if error_type == 'pitch' else 3

    pid = target_locator['part_id']
    mi  = target_locator['measure_index']
    ni  = target_locator['note_index']

    parsed, tree, ns = _parse_xml_notes(input_path, part_id=pid)
    if not parsed:
        raise ValueError(f"No notes found in part {pid}")

    target_flat = None
    for fi, n in enumerate(parsed):
        if (n['part_id'] == pid and n['measure_index'] == mi
                and n['note_in_measure'] == ni):
            target_flat = fi
            break
    if target_flat is None:
        raise ValueError(f"Note not found: part={pid}, measure={mi}, note={ni}")

    tn = parsed[target_flat]
    if error_type == 'pitch':
        current_val = tn['pitch'] or 'rest'
    else:
        current_val = str(tn['duration'] * 4) if tn['duration'] is not None else '?'

    cp_ids, cp_err_idx = _build_cp_for_inference(parsed, target_flat, error_type)

    cp_t = torch.tensor([cp_ids], dtype=torch.long, device=DEVICE)
    ei_t = torch.tensor([cp_err_idx], dtype=torch.long, device=DEVICE)
    attn = torch.ones(1, len(cp_ids), dtype=torch.long, device=DEVICE)

    with torch.inference_mode():
        logits = _infer_model(cp_t, ei_t, attn,
                              task='pitch' if error_type == 'pitch' else 'dur')
        probs = logits.softmax(-1).cpu()
        topk = probs.topk(min(top_k, probs.size(-1)), dim=-1)

    id2label = _id2pitch if error_type == 'pitch' else _id2dur

    candidates = []
    for k in range(topk.indices.size(1)):
        cls_id = topk.indices[0, k].item()
        label = id2label[cls_id]
        prob  = topk.values[0, k].item()
        candidates.append({'label': label, 'prob': round(prob, 4)})

    pred_label = candidates[0]['label']

    # Duration labels ARE already quarterLength — no * 4
    if error_type == 'duration':
        pred_ql = pred_label
        cand_display = [{'label': c['label'] + ' QL', 'prob': c['prob']}
                        for c in candidates]
    else:
        pred_ql = None
        cand_display = candidates

    # Write corrected MusicXML
    edit_tree = etree.parse(str(input_path))
    edit_root = edit_tree.getroot()
    edit_ns   = edit_root.nsmap.get(None, "")
    et = lambda name: _xt(name, edit_ns)

    target_part = None
    for p in edit_root.findall(f".//{et('part')}"):
        if p.get('id') == pid:
            target_part = p; break
    measures = target_part.findall(et('measure'))
    target_meas = measures[mi]
    note_els = [el for el in target_meas
                if (el.tag.split('}')[-1] if '}' in el.tag else el.tag) == 'note']
    target_note_el = note_els[ni]

    if error_type == 'pitch':
        _apply_pitch(target_note_el, pred_label, edit_ns)
    else:
        divs = 1
        for m in measures[:mi+1]:
            for attr in m.findall(et('attributes')):
                d_el = attr.find(et('divisions'))
                if d_el is not None:
                    divs = int(d_el.text)
        w = _apply_duration(target_note_el, pred_ql, divs, edit_ns)
        warnings.extend(w)

    xml_out = output_dir / 'model_corrected.musicxml'
    edit_tree.write(str(xml_out), xml_declaration=True, encoding='UTF-8')

    midi_out = output_dir / 'model_corrected.mid'
    n_events = _xml_to_midi(edit_tree, midi_out, edit_ns)

    # Verify
    v_tree = etree.parse(str(xml_out))
    v_root = v_tree.getroot()
    v_ns   = v_root.nsmap.get(None, "")
    vt     = lambda name: _xt(name, v_ns)
    v_part = None
    for p in v_root.findall(f".//{vt('part')}"):
        if p.get('id') == pid:
            v_part = p; break
    v_meas = v_part.findall(vt('measure'))[mi]
    v_notes = [el for el in v_meas
               if (el.tag.split('}')[-1] if '}' in el.tag else el.tag) == 'note']
    v_note = v_notes[ni]

    if error_type == 'pitch':
        vp = v_note.find(vt('pitch'))
        if vp is not None:
            v_val = vp.find(vt('step')).text + vp.find(vt('octave')).text
            xml_ok = v_val == pred_label
        else:
            xml_ok = False; warnings.append("Verified note has no pitch element")
    else:
        vd = v_note.find(vt('duration'))
        divs_v = 1
        for m in v_part.findall(vt('measure'))[:mi+1]:
            for attr in m.findall(vt('attributes')):
                d_el = attr.find(vt('divisions'))
                if d_el is not None: divs_v = int(d_el.text)
        expected = int(_Frac(pred_ql) * divs_v)
        actual = int(vd.text) if vd is not None else -1
        xml_ok = actual == expected
        if not xml_ok:
            warnings.append(f"Duration verify: expected {expected}, got {actual}")

    elapsed = _time.monotonic() - t0

    result = {
        'input_file': str(input_path),
        'target_locator': target_locator,
        'error_type': error_type,
        'current_value': current_val,
        'prediction': pred_label if error_type == 'pitch' else pred_ql,
        'candidates': cand_display,
        'xml_verified': xml_ok,
        'midi_events': n_events,
        'checkpoint': str(_JOINT_CKPT_PATH),
        'context_notes': len(cp_ids),
        'elapsed_s': round(elapsed, 3),
        'warnings': warnings,
        'output_xml': str(xml_out),
        'output_midi': str(midi_out),
    }

    with open(output_dir / 'prediction.json', 'w') as f:
        json.dump(result, f, indent=2)

    return result

print("correct_musicxml fixed (dur labels = QL, no *4)")


In [ ]:

# Re-run both cases with fixed function

print(f"{'='*70}")
print("  CASE 1: Pitch correction")
print(f"{'='*70}")

pitch_result = correct_musicxml(
    input_path=pitch_input,
    target_locator={'part_id': 'P2', 'measure_index': 0, 'note_index': 5},
    error_type='pitch',
    output_dir=INFER_DEMO / 'pitch_output',
)

print(f"  Current:     {pitch_result['current_value']}")
print(f"  Prediction:  {pitch_result['prediction']}")
print(f"  Candidates:  {pitch_result['candidates']}")
print(f"  XML ok:      {pitch_result['xml_verified']}")
print(f"  MIDI events: {pitch_result['midi_events']}")
print(f"  Time:        {pitch_result['elapsed_s']:.3f}s")

print(f"\n  [Validation] truth=E4, pred={pitch_result['prediction']}"
      f" -> {'MATCH' if pitch_result['prediction'] == 'E4' else 'MISMATCH'}")

print(f"\n{'='*70}")
print("  CASE 2: Duration correction")
print(f"{'='*70}")

dur_result = correct_musicxml(
    input_path=dur_input,
    target_locator={'part_id': 'P2', 'measure_index': 3, 'note_index': 24},
    error_type='duration',
    output_dir=INFER_DEMO / 'dur_output',
)

print(f"  Current:     {dur_result['current_value']} QL")
print(f"  Prediction:  {dur_result['prediction']} QL")
print(f"  Candidates:  {dur_result['candidates']}")
print(f"  XML ok:      {dur_result['xml_verified']}")
print(f"  MIDI events: {dur_result['midi_events']}")
print(f"  Time:        {dur_result['elapsed_s']:.3f}s")
if dur_result['warnings']:
    print(f"  Warnings:    {dur_result['warnings']}")

print(f"\n  [Validation] truth=1/2 QL, pred={dur_result['prediction']} QL"
      f" -> {'MATCH' if dur_result['prediction'] == '1/2' else 'MISMATCH'}")

print(f"\n{'='*70}")
print("  Output files:")
for sub in ['pitch_output', 'dur_output']:
    d = INFER_DEMO / sub
    for f in sorted(d.iterdir()):
        print(f"    {f.relative_to(ML_DIR)}")
print(f"{'='*70}")
